In [1]:
import torch
print(torch.version)
print(torch.version.cuda)
print(torch.cuda.is_available())

<module 'torch.version' from 'c:\\Users\\ILIA\\anaconda3\\envs\\medssl\\lib\\site-packages\\torch\\version.py'>
11.8
True


In [2]:
# ==============================================================================
# MedSSL-3D v0.8-B
# Self-Supervised 3D Medical Representation Learning (SimCLR) + a U-Net-style
# residual segmentation decoder with deep supervision.
#
# v0.7-A SCOPE (per project roadmap): "Single-modality baseline stabilization".
# This version does NOT touch the architecture (Encoder/Decoder/Skip
# connections/Deep Supervision are exactly v0.6, byte-for-byte). It only
# makes the SEGMENTATION TRAINING PROTOCOL statistically trustworthy:
#   1. Patient-level Train/Val/Test split (70/15/15), persisted to JSON and
#      reloaded on every subsequent run so v0.7-B/v0.8/v0.9 use the SAME
#      patients. The Test set is loaded/counted only — never evaluated here.
#   2. Multi-seed (3 seeds) training: each seed re-seeds everything, reloads
#      the SSL encoder fresh, reinitializes the decoder from scratch, trains
#      independently, and reports its own final metrics. Mean +/- Std across
#      seeds is the reported v0.7-A baseline number.
#   3. NUM_EPOCHS_SEG increased 20 -> 45, CosineAnnealingLR added.
#   4. Checkpoint selection now uses a 3-epoch moving average of Val Dice
#      instead of a single noisy epoch value.
#   5. BraTS-style region metrics (WT / TC / ET Dice) added on top of the
#      existing 4-class Dice/IoU, computed on the already-remapped {0,1,2,3}
#      labels.
#   6. Logging extended (val loss, WT/TC/ET, current LR) but keeps v0.6's
#      emoji/logging style.
#   7. All new knobs centralized in Config.
#
# EVERYTHING IN MODE A (SimCLR pretraining) IS REPRODUCED UNCHANGED FROM v0.6.
#
# Single Google Colab / Jupyter cell — single continuous run.
#
# ONLY LINE YOU SHOULD NEED TO EDIT: Config.DATA_ROOT, below.
# ==============================================================================

# ==============================================================================
# NEW in v0.8 — Shared Encoder Feature Attention + Linear Probe Sanity Check
#
# v0.8 SCOPE (per project roadmap): single independent variable = Channel-wise
# SE Attention applied to the encoder bottleneck (l4) only. Everything else
# added below is evaluation-only and never touches training/optimization:
#   1. NEW: SEChannelAttention3D — applied ONLY to l4 inside SegmentationModel3D,
#      gated by Config.USE_CHANNEL_SE_ATTENTION (default True). When False,
#      forward() behaves byte-for-byte like v0.7-B.
#   2. NEW: post-training Channel Attention logging (mean weight per channel).
#   3. NEW: post-hoc Modality Ablation (zero-mask each of FLAIR/T1/T1CE/T2,
#      measure Dice drop) — evaluation only, no trainable parameters.
#   4. NEW: Linear Probe sanity check (Linear(512->2) on the frozen, already
#      fine-tuned v0.8 encoder) using HGG/LGG grades from name_mapping.csv.
#   5. NEW: dedicated v0.8 output root (outputs/v0_8/...) — nothing in
#      previous version folders is ever overwritten.
#
# The ResNet3D encoder, ResidualDecoder3D, losses, dataset, split logic, and
# checkpoint-selection strategy are all UNCHANGED from v0.7-B.
# ==============================================================================

# ==============================================================================
# NEW in v0.9 — MULTI-TASK LEARNING (Segmentation + HGG/LGG Classification)
#
# v0.9 SCOPE: add a second task head on the SHARED encoder bottleneck, without
# rewriting anything v0.8 already validated. Implemented ADDITIVELY — every
# addition is marked `# NEW in v0.9`:
#   Task A: SegmentationModel3D.forward(..., return_bottleneck=False). Default
#           False -> byte-for-byte identical to v0.8. True -> also returns l4.
#   Task B: ClassificationHead3D — AdaptiveAvgPool3d(1) -> Linear(512,128) ->
#           ReLU -> Dropout(Config.CLS_HEAD_DROPOUT) -> Linear(128,2).
#   Task C: MultiTaskModel3D — wraps an unmodified SegmentationModel3D plus the
#           new head; returns (seg_output, cls_logits). Gated by
#           Config.USE_CLASSIFICATION_HEAD (False -> plain SegmentationModel3D).
#   Task D: build_fresh_multitask_model() — calls the UNCHANGED
#           build_fresh_segmentation_model(), so the from-scratch-per-seed
#           protocol is literally the same code (no warm start from v0.8).
#   Task E: MultiTaskSegmentationDataset — SUBCLASS of BraTSSegmentationDataset
#           returning (volume, mask, grade_label); grade -1 = unknown, masked
#           out of the classification loss via ignore_index=-1 while still
#           training segmentation.
#   Task F: train_one_epoch_multitask() / run_single_seed_multitask() —
#           SIBLINGS of the v0.8 functions, which are untouched.
#           L_total = LAMBDA_SEG * L_seg + LAMBDA_CLS * L_cls.
#   Task G: WeightedRandomSampler (own re-seeded generator), elastic /
#           gamma / bias-field augmentation behind individual flags, LR warmup
#           inside the existing 25-epoch budget, gradient clipping, and early
#           stopping on the moving-average Val Dice.
#
# PRESERVED FROM v0.8, ON PURPOSE:
#   - The DataLoader `generator=torch.Generator()` + per-seed manual_seed(seed)
#     RNG fix (now applied to the new sampler's generator too).
#   - DiceCELoss for segmentation (NOT Focal) — so any Dice movement is
#     attributable to the Cls head alone.
#   - Checkpoint selection = 3-epoch moving average of Val Dice ONLY;
#     classification accuracy is NOT folded into it.
#   - SE Attention, encoder/decoder, split file, CSV format, output isolation.
#
# v0.9 writes ONLY under outputs/v0_9/ — v0.7-A/v0.7-B/v0.8 outputs are never
# touched. Run with Config.MODE = 'multitask' (the new default).
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. Install dependencies
#    (This '!' line is Jupyter/Colab notebook-cell syntax. If you ever run this
#    file as a plain .py script instead of a notebook cell, remove this line
#    and install the packages beforehand with a normal `pip install` command.)
# ------------------------------------------------------------------------------
!pip install -q torch torchvision monai nibabel numpy tqdm matplotlib

# ------------------------------------------------------------------------------
# 2. Imports
# ------------------------------------------------------------------------------
import os
import glob
import json
import csv
import time
import random
import logging
import platform
import re                      # NEW in v0.8 — patient-ID extraction for Linear Probe grade lookup
from typing import Tuple, List, Optional, Union

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import monai
from monai.transforms import (
    Compose, LoadImage, EnsureChannelFirst, Resize, NormalizeIntensity,
    RandFlip, RandRotate90, RandGaussianNoise, RandScaleIntensity, RandShiftIntensity,
)

import matplotlib.pyplot as plt
from tqdm import tqdm

# ------------------------------------------------------------------------------
# 3. Logging
# ------------------------------------------------------------------------------
logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger('MedSSL-3D')


def log_section(title: str) -> None:
    logger.info("\n" + "=" * 80)
    logger.info(title)
    logger.info("=" * 80)


# ------------------------------------------------------------------------------
# 4. Config
# ------------------------------------------------------------------------------
class Config:
    SEED = 42
    DEVICE = 'cuda'

    # ---- Shared by both modes ----
    IMG_SIZE = (128, 128, 64)     # (H, W, D)
    NUM_WORKERS = 0                # keep 0 — safest choice on Colab

    # >>> THE ONLY LINE YOU SHOULD NEED TO EDIT <<<
    # Moved off the OneDrive-synced C:\Users\...\Desktop path onto a local D:
    # drive — OneDrive "Files On-Demand" placeholders were the root cause of
    # the ~6x per-epoch slowdown diagnosed earlier (each Train file had to be
    # fetched from the cloud on first read, serially, since NUM_WORKERS=0).
    DATA_ROOT = 'D:/MedSSL/MICCAI_BraTS2020_TrainingData'
    MODALITY = 'flair'            # 't1', 't1ce', 't2', or 'flair'

    # ---- Mode switch ----
    # NEW in v0.9: 'multitask' (Mode C) added as a THIRD option and made the
    # default so this cell runs v0.9 out of the box. Set MODE back to
    # 'segmentation' to re-run the frozen v0.8 path unchanged, or 'simclr' for
    # Mode A — both code paths are untouched.
    MODE = 'metrics'               # NEW in v0.9-metrics: 'simclr' | 'segmentation' | 'multitask' | 'diagnostic' | 'metrics' (read-only, no training)

    # ---- Mode A: SimCLR pretraining (identical defaults to v0.4/v0.5/v0.6) ----
    BATCH_SIZE = 8
    PROJECTION_DIM = 128
    TEMPERATURE = 0.5
    LR = 1e-4
    NUM_EPOCHS = 20

    # ---- Output directories ----
    # Also moved to the same local D: drive as DATA_ROOT, so checkpoints/logs
    # are never re-exposed to OneDrive sync either.
    DESKTOP_BASE = 'D:/MedSSL/MedSSL_Results'
    CHECKPOINT_DIR = os.path.join(DESKTOP_BASE, 'checkpoints')
    LOG_DIR = os.path.join(DESKTOP_BASE, 'logs')
    SEG_CHECKPOINT_DIR = os.path.join(DESKTOP_BASE, 'checkpoints', 'segmentation')
    SEG_LOG_DIR = os.path.join(DESKTOP_BASE, 'logs', 'segmentation')

    # ---- Mode B: Segmentation (dataset/loss/metrics core unchanged from v0.5/v0.6) ----
    NUM_SEG_CLASSES = 4            # background, NCR/NET, edema, enhancing (BraTS label 4 -> 3)
    BATCH_SIZE_SEG = 2              # 3D volumes are memory-heavy; lower this further if you hit OOM
    LR_SEG = 1e-4
    USE_PRETRAINED_ENCODER = True
    PRETRAINED_ENCODER_PATH = os.path.join(CHECKPOINT_DIR, 'encoder.pth')   # produced by Mode A

    # ---- Mode B, v0.6: deep-supervision auxiliary loss weight ----
    # Applied to each of the two auxiliary heads in ResidualDecoder3D.
    # Main loss keeps weight 1.0, exactly as v0.5's single-output loss did.
    AUX_LOSS_WEIGHT = 0.3

    # ==========================================================================
    # ---- Mode B, NEW IN v0.7-A: statistical baseline stabilization ----
    # ==========================================================================
    # Epochs increased from v0.6's 20 into the 40-50 range requested for v0.7-A.
    NUM_EPOCHS_SEG = 25

    # Independent training seeds. Each seed retrains the segmentation model
    # from scratch (fresh SSL encoder reload + fresh decoder init) so the
    # final reported metric is a Mean +/- Std across genuinely separate runs,
    # not 3 samples from the same noisy trajectory.
    SEEDS = [42]

    # Patient-level 3-way split fractions (must sum to 1.0).
    TRAIN_FRACTION = 0.70
    VAL_FRACTION = 0.15
    TEST_FRACTION = 0.15

    # Split is computed once and persisted here. If this file exists on a
    # later run, it is LOADED (never recomputed), guaranteeing v0.7-B, v0.8,
    # v0.9, ... all train/val/test on the exact same patients as v0.7-A.
    SPLIT_FILE_PATH = os.path.join(DESKTOP_BASE, 'data_split_v1.json')

    # CosineAnnealingLR — simplest scheduler compatible with the existing
    # per-epoch Adam loop. T_max is set to NUM_EPOCHS_SEG at scheduler
    # construction time (one full cosine cycle per training run).
    SCHEDULER_TYPE = 'cosine'
    SCHEDULER_ETA_MIN = 1e-6

    # Best-checkpoint selection: moving average window size (epochs).
    # Per spec: starting once >= this many epochs have completed, compute
    # mean(val_dice[e-2], val_dice[e-1], val_dice[e]) and save whenever it's
    # a new best. NOT a weight average — only used as the selection metric.
    MOVING_AVG_WINDOW = 3

    # Where the final multi-seed Mean +/- Std summary table is written.
    SEG_SUMMARY_CSV = os.path.join(SEG_LOG_DIR, 'v0_7a_multiseed_summary.csv')

    # ==========================================================================
    # ---- Mode B, NEW IN v0.8: Channel-wise SE Attention (single independent
    # variable) + versioned outputs + Modality Ablation + Linear Probe.
    # ==========================================================================
    # The ONLY architectural change in v0.8. When False, SegmentationModel3D's
    # forward() is byte-for-byte identical to v0.7-B (see Task 1 of the spec).
    USE_CHANNEL_SE_ATTENTION = True                                    # NEW in v0.8
    SE_REDUCTION_RATIO = 16                                            # NEW in v0.8

    # Dedicated v0.8 output root — v0.8 must NEVER overwrite v0.7-A/v0.7-B
    # outputs, which all live directly under DESKTOP_BASE/checkpoints and
    # DESKTOP_BASE/logs above.
    VERSION_ROOT_V0_8 = os.path.join(DESKTOP_BASE, 'outputs', 'v0_8')          # NEW in v0.8
    V0_8_CHECKPOINT_DIR = os.path.join(VERSION_ROOT_V0_8, 'checkpoints')      # NEW in v0.8
    V0_8_LOG_DIR = os.path.join(VERSION_ROOT_V0_8, 'logs')                    # NEW in v0.8
    V0_8_CSV_DIR = os.path.join(VERSION_ROOT_V0_8, 'csv')                     # NEW in v0.8
    V0_8_ATTENTION_DIR = os.path.join(VERSION_ROOT_V0_8, 'attention')         # NEW in v0.8
    V0_8_LINEAR_PROBE_DIR = os.path.join(VERSION_ROOT_V0_8, 'linear_probe')   # NEW in v0.8

    # NEW in v0.8, Task 3 — Linear Probe sanity check.
    NAME_MAPPING_CSV_PATH = os.path.join(DATA_ROOT, 'name_mapping.csv')       # NEW in v0.8
    LINEAR_PROBE_EPOCHS = 15                                                  # NEW in v0.8
    LINEAR_PROBE_LR = 1e-3                                                    # NEW in v0.8
    LINEAR_PROBE_BATCH_SIZE = 4                                               # NEW in v0.8

    # NEW in v0.8 — where the v0.8 multi-seed Mean +/- Std summary table is
    # written. Kept separate from v0.7-A's SEG_SUMMARY_CSV above.
    SEG_SUMMARY_CSV_V0_8 = os.path.join(V0_8_CSV_DIR, 'v0_8_multiseed_summary.csv')  # NEW in v0.8

    # ==========================================================================
    # ---- Mode C, NEW in v0.9: Multi-Task Learning (Segmentation + HGG/LGG
    #      Classification). Every key below is ADDITIVE — nothing above was
    #      removed or edited except MODE's default value (see section 4b note).
    # ==========================================================================
    # ---- Task B/C: the classification head itself ----
    USE_CLASSIFICATION_HEAD = True                                     # NEW in v0.9
    NUM_CLS_CLASSES = 2                                                # NEW in v0.9 (HGG / LGG)
    CLS_HEAD_HIDDEN_DIM = 128                                          # NEW in v0.9
    CLS_HEAD_DROPOUT = 0.3                                             # NEW in v0.9

    # ---- Task F: multi-task loss weighting ----
    # L_total = LAMBDA_SEG * L_seg + LAMBDA_CLS * L_cls
    # Segmentation stays the dominant objective; 0.3 keeps the classification
    # gradient a nudge on the shared encoder rather than a competing task.
    LAMBDA_SEG = 1.0                                                   # NEW in v0.9
    LAMBDA_CLS = 0.3                                                   # NEW in v0.9

    # Segmentation loss is STILL DiceCELoss (deliberately NOT Focal) so that
    # any Dice change vs. v0.8 is attributable to the Cls head, not to a new
    # loss function. Revisit only if ET Dice specifically stays weak.

    # ---- Task G: sampler + augmentation + optimization additions ----
    USE_WEIGHTED_SAMPLER = False                                        # NEW in v0.9

    USE_ELASTIC_DEFORM = True                                          # NEW in v0.9 (spatial: image AND mask)
    USE_GAMMA_ADJUST = True                                            # NEW in v0.9 (intensity only)
    USE_BIAS_FIELD = True                                              # NEW in v0.9 (intensity only)

    ELASTIC_PROB = 0.30                                                # NEW in v0.9
    ELASTIC_CONTROL_GRID = (4, 4, 4)                                   # NEW in v0.9 (coarse -> smooth warp)
    ELASTIC_MAX_DISPLACEMENT = 0.06                                    # NEW in v0.9 (in normalized [-1,1] grid units)

    GAMMA_PROB = 0.30                                                  # NEW in v0.9
    GAMMA_RANGE = (0.7, 1.5)                                           # NEW in v0.9

    BIAS_FIELD_PROB = 0.30                                             # NEW in v0.9
    BIAS_FIELD_CONTROL_GRID = (4, 4, 4)                                # NEW in v0.9
    BIAS_FIELD_MAX_LOG = 0.30                                          # NEW in v0.9 (exp(±0.3) ≈ ×0.74..×1.35)

    # Runtime check: NUM_WORKERS = 0 means augmentation runs synchronously on
    # the CPU before every batch. The first AUG_TIMING_EPOCHS epochs report the
    # measured augmentation share of the train epoch; if it exceeds
    # AUG_OVERHEAD_WARN_FRACTION, the log tells you to switch USE_ELASTIC_DEFORM
    # off FIRST (by far the most expensive) and keep gamma/bias-field.
    AUG_TIMING_EPOCHS = 2                                              # NEW in v0.9
    AUG_OVERHEAD_WARN_FRACTION = 0.15                                  # NEW in v0.9

    # Warmup epochs are taken FROM the existing NUM_EPOCHS_SEG budget (the
    # cosine cycle covers the remaining epochs) — total epochs stay at 25.
    WARMUP_EPOCHS = 3                                                  # NEW in v0.9
    GRAD_CLIP_NORM = 1.0                                               # NEW in v0.9

    # Early stopping on the SAME moving-average Val Dice already used for
    # checkpoint selection. Patience is generous on purpose: v0.7-B and v0.8
    # found their best checkpoint at epochs 24 and 22.
    USE_EARLY_STOPPING = True                                          # NEW in v0.9
    EARLY_STOP_PATIENCE = 8                                            # NEW in v0.9

    # ---- Versioned outputs (v0.7-A/B and v0.8 dirs are never touched) ----
    VERSION_ROOT_V0_9 = os.path.join(DESKTOP_BASE, 'outputs', 'v0_9')         # NEW in v0.9
    V0_9_CHECKPOINT_DIR = os.path.join(VERSION_ROOT_V0_9, 'checkpoints')      # NEW in v0.9
    V0_9_LOG_DIR = os.path.join(VERSION_ROOT_V0_9, 'logs')                    # NEW in v0.9
    V0_9_CSV_DIR = os.path.join(VERSION_ROOT_V0_9, 'csv')                     # NEW in v0.9
    SEG_SUMMARY_CSV_V0_9 = os.path.join(V0_9_CSV_DIR, 'v0_9_multiseed_summary.csv')  # NEW in v0.9

    # Reported v0.8 Linear Probe accuracy — logged only, as the bar the
    # jointly-trained v0.9 head is expected to clear.
    V0_8_LINEAR_PROBE_REFERENCE_ACC = 0.80                             # NEW in v0.9

    # ==========================================================================
    # ---- NEW in v0.9-diag: read-only grade-information diagnostic ----
    #      Nothing here trains anything; these keys are read only by the
    #      Section 24-DIAG functions. Added as new keys — no key above was
    #      edited or removed.
    # ==========================================================================
    RUN_GRADE_DIAGNOSTIC   = True                                      # NEW in v0.9-diag
    DIAG_CHECKPOINT_PATH   = None    # None -> outputs/v0_9/checkpoints/seed_{DIAG_SEED}/best_moving_avg_model.pth
    DIAG_SEED              = 42                                        # NEW in v0.9-diag
    # DIAG_PROBE_* MUST MIRROR THE LINEAR_PROBE_* VALUES ABOVE EXACTLY
    # (15 / 1e-3 / 4). They are duplicated rather than aliased so this block is
    # purely additive — but if a future reader changes one of them without
    # changing its LINEAR_PROBE_* twin, the comparison against the v0.8 anchor
    # 0.80 stops being valid and the decision table becomes meaningless.
    DIAG_PROBE_EPOCHS      = 15      # == LINEAR_PROBE_EPOCHS            # NEW in v0.9-diag
    DIAG_PROBE_LR          = 1e-3    # == LINEAR_PROBE_LR                # NEW in v0.9-diag
    DIAG_PROBE_BATCH_SIZE  = 4       # == LINEAR_PROBE_BATCH_SIZE        # NEW in v0.9-diag
    DIAG_FEATURE_BATCH     = 2       # forward batch during feature extraction (6GB VRAM)

    # Anchors from the two completed runs — LOGGED for reference, never re-derived.
    DIAG_REF_V0_9_JOINT_ACC     = 0.7636   # v0.9 joint cls head          # NEW in v0.9-diag
    DIAG_REF_MAJORITY_BASELINE  = 0.7636   # majority class on the val split
    DIAG_NEAR_TOL               = 0.02     # what "≈" means in the decision table
    DIAG_GAP_TOL                = 0.03     # what "<<" means in the decision table

    DIAG_CSV_PATH = os.path.join(V0_9_CSV_DIR, 'v0_9_diag_grade_information.csv')  # NEW in v0.9-diag

    # ---- Task F: global-clip instrumentation. DEFAULT False on purpose — when
    #      False nothing is installed and the multitask run path is byte-for-byte
    #      unchanged. See Section 24f.
    DIAG_LOG_GRAD_NORMS    = False                                     # NEW in v0.9-diag
    DIAG_GRAD_LOG_EVERY    = 129     # optimizer steps between log lines (~1/epoch at 258 samples / batch 2)

    # ==========================================================================
    # ---- NEW in v0.9-metrics: honest classification metrics + re-scoring ----
    #      Read-only. New keys only; nothing above was edited or removed.
    # ==========================================================================
    RUN_METRICS_RESCORE    = True                                      # NEW in v0.9-metrics
    METRICS_CSV_PATH = os.path.join(V0_9_CSV_DIR, 'v0_9_metrics_rescore.csv')      # NEW in v0.9-metrics
    # Cached feature tensors are NOT CSVs, so they go under checkpoints/, not csv/.
    METRICS_FEATURE_CACHE_DIR = os.path.join(V0_9_CHECKPOINT_DIR, 'diag_features') # NEW in v0.9-metrics
    METRICS_PERM_ITERS     = 10000   # label shuffles for the permutation p-value  # NEW in v0.9-metrics
    # Two rows differing by fewer than this many correctly-classified patients
    # are reported as UNDERPOWERED rather than ranked against each other.
    METRICS_UNDERPOWERED_PATIENTS = 3                                  # NEW in v0.9-metrics

    # The original v0.9 run's classification numbers. Task C must REPRODUCE
    # these exactly from the checkpoint; a mismatch means the re-scoring path is
    # not measuring what the training loop measured.
    METRICS_EXPECTED_V0_9_ACC   = 0.7636                               # NEW in v0.9-metrics
    METRICS_EXPECTED_V0_9_F1    = 0.8434                               # NEW in v0.9-metrics
    # (a, b, c, d) = (true HGG pred HGG, true LGG pred HGG, true HGG pred LGG, true LGG pred LGG)
    METRICS_EXPECTED_CONFUSION  = (35, 6, 7, 7)                        # NEW in v0.9-metrics

    @classmethod
    def setup_dirs(cls) -> None:
        os.makedirs(cls.CHECKPOINT_DIR, exist_ok=True)
        os.makedirs(cls.LOG_DIR, exist_ok=True)
        os.makedirs(cls.SEG_CHECKPOINT_DIR, exist_ok=True)
        os.makedirs(cls.SEG_LOG_DIR, exist_ok=True)
        # NEW in v0.8: dedicated version root, never touches the dirs above.
        os.makedirs(cls.V0_8_CHECKPOINT_DIR, exist_ok=True)
        os.makedirs(cls.V0_8_LOG_DIR, exist_ok=True)
        os.makedirs(cls.V0_8_CSV_DIR, exist_ok=True)
        os.makedirs(cls.V0_8_ATTENTION_DIR, exist_ok=True)
        os.makedirs(cls.V0_8_LINEAR_PROBE_DIR, exist_ok=True)
        # NEW in v0.9: dedicated v0.9 version root — the v0.7/v0.8 lines above
        # are kept exactly as they were, so no earlier output can be clobbered.
        os.makedirs(cls.V0_9_CHECKPOINT_DIR, exist_ok=True)                # NEW in v0.9
        os.makedirs(cls.V0_9_LOG_DIR, exist_ok=True)                       # NEW in v0.9
        os.makedirs(cls.V0_9_CSV_DIR, exist_ok=True)                       # NEW in v0.9

    @classmethod
    def to_dict(cls) -> dict:
        return {
            k: (list(v) if isinstance(v, tuple) else v)
            for k, v in vars(cls).items()
            if not k.startswith('_') and not callable(v) and not isinstance(v, classmethod)
        }


# ------------------------------------------------------------------------------
# 5. Environment report — helps diagnose Colab / library-version drift
# ------------------------------------------------------------------------------
def print_environment_report() -> None:
    log_section("🧪 ENVIRONMENT REPORT")
    logger.info(f"Python      : {platform.python_version()}")
    logger.info(f"PyTorch     : {torch.__version__}")
    logger.info(f"MONAI       : {monai.__version__}")
    logger.info(f"NumPy       : {np.__version__}")
    logger.info(f"CUDA avail. : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        logger.info(f"GPU         : {torch.cuda.get_device_name(0)}")
        total_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        logger.info(f"GPU memory  : {total_mem:.1f} GB")
    logger.info(f"Device used : {Config.DEVICE}")
    logger.info(f"Run mode    : {Config.MODE}")


# ------------------------------------------------------------------------------
# 6. Seed
# ------------------------------------------------------------------------------
def set_seed(seed: int = Config.SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    logger.info(f"🔒 Seed set to {seed}")


# ------------------------------------------------------------------------------
# 7. Base loading transforms (deterministic — no randomness) — UNCHANGED
# ------------------------------------------------------------------------------
def build_base_transform(img_size: Tuple[int, int, int]) -> Compose:
    """For MRI volumes: trilinear resize preserves intensity gradients."""
    return Compose([
        LoadImage(image_only=True),
        EnsureChannelFirst(),
        Resize(spatial_size=img_size, mode='trilinear'),
    ])


def build_mask_transform(img_size: Tuple[int, int, int]) -> Compose:
    """For segmentation masks: nearest-neighbor resize keeps integer class labels intact."""
    return Compose([
        LoadImage(image_only=True),
        EnsureChannelFirst(),
        Resize(spatial_size=img_size, mode='nearest'),
    ])


# ------------------------------------------------------------------------------
# 8. Mode A — Data Sanity Check (unchanged from v0.4/v0.5/v0.6; stops on failure)
# ------------------------------------------------------------------------------
def run_data_sanity_check(data_root: str, modality: str,
                           img_size: Tuple[int, int, int]) -> List[str]:
    """
    Pre-flight check for the SimCLR (Mode A) data pipeline:
      - directory exists
      - files for the chosen modality are found
      - first file loads, with sane shape / dtype / min / max
      - no NaN / Inf
      - a middle slice is displayed for visual confirmation
    Raises SystemExit with a clear message on any failure.
    """
    log_section("🔍 DATA SANITY CHECK (Mode A — SimCLR)")

    if not os.path.isdir(data_root):
        raise SystemExit(
            f"❌ DATA_ROOT does not exist: '{data_root}'\n"
            f"   -> Update Config.DATA_ROOT to the folder where you downloaded BraTS."
        )
    logger.info(f"✅ Directory exists: {data_root}")

    pattern = os.path.join(data_root, '**', f'*{modality}*.nii*')
    image_paths = sorted(glob.glob(pattern, recursive=True))

    if len(image_paths) == 0:
        raise SystemExit(
            f"❌ No files matching modality '{modality}' were found under '{data_root}'.\n"
            f"   Checked pattern: {pattern}\n"
            f"   -> Confirm the dataset was extracted correctly and the modality name "
            f"matches your filenames (e.g. 't1', 't1ce', 't2', 'flair')."
        )
    logger.info(f"✅ Found {len(image_paths)} file(s) for modality '{modality}'")
    logger.info(f"📄 First file: {os.path.basename(image_paths[0])}")

    base_transform = build_base_transform(img_size)
    try:
        volume = base_transform(image_paths[0])
    except Exception as e:
        raise SystemExit(f"❌ Failed to load/transform the first file: {e}")

    if not torch.is_tensor(volume):
        volume = torch.as_tensor(np.asarray(volume))

    logger.info(f"📦 Shape : {tuple(volume.shape)}")
    logger.info(f"🔢 Dtype : {volume.dtype}")

    expected_shape = (1, *img_size)
    if tuple(volume.shape) != expected_shape:
        raise SystemExit(
            f"❌ Unexpected volume shape: got {tuple(volume.shape)}, expected {expected_shape}."
        )

    vol_f = volume.float()
    logger.info(f"📉 Min value : {vol_f.min().item():.4f}")
    logger.info(f"📈 Max value : {vol_f.max().item():.4f}")

    if torch.isnan(vol_f).any():
        raise SystemExit(f"❌ NaN values detected in '{image_paths[0]}'.")
    if torch.isinf(vol_f).any():
        raise SystemExit(f"❌ Inf values detected in '{image_paths[0]}'.")
    logger.info("✅ No NaN / Inf detected")

    try:
        mid_slice = vol_f[0, :, :, vol_f.shape[-1] // 2]
        plt.figure(figsize=(4, 4))
        plt.imshow(mid_slice.numpy(), cmap='gray')
        plt.title(f"Middle slice — {os.path.basename(image_paths[0])}")
        plt.axis('off')
        plt.show()
        logger.info("✅ Middle slice displayed above.")
    except Exception as e:
        logger.warning(f"⚠️ Could not display middle slice (non-fatal): {e}")

    logger.info("\n✅ SANITY CHECK PASSED — proceeding.\n")
    return image_paths


# ------------------------------------------------------------------------------
# 9. Mode A — Contrastive dataset (unchanged from v0.4/v0.5/v0.6)
# ------------------------------------------------------------------------------
def build_augmentation_pipeline() -> Compose:
    """
    Builds a FRESH list of transform instances every time it's called.
    Two separate calls never share the same transform objects, so view1
    and view2 augmentations are genuinely independent.
    """
    return Compose([
        RandFlip(spatial_axis=0, prob=0.3),
        RandFlip(spatial_axis=1, prob=0.3),
        RandFlip(spatial_axis=2, prob=0.3),
        RandRotate90(prob=0.3, max_k=1),
        RandGaussianNoise(prob=0.2, mean=0.0, std=0.1),
        RandScaleIntensity(factors=0.1, prob=0.3),
        RandShiftIntensity(offsets=0.1, prob=0.3),
        NormalizeIntensity(nonzero=True, channel_wise=True),
    ])


class BraTSDataset(Dataset):
    """Two independently-augmented views of each volume, for SimCLR (Mode A)."""
    def __init__(self, image_paths: List[str],
                 img_size: Tuple[int, int, int] = Config.IMG_SIZE,
                 is_train: bool = True):
        self.image_paths = image_paths
        self.img_size = img_size
        self.is_train = is_train
        self.base_transform = build_base_transform(img_size)

        if is_train:
            self.augment1 = build_augmentation_pipeline()
            self.augment2 = build_augmentation_pipeline()
        else:
            self.augment1 = Compose([NormalizeIntensity(nonzero=True, channel_wise=True)])
            self.augment2 = Compose([NormalizeIntensity(nonzero=True, channel_wise=True)])

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int):
        volume = self.base_transform(self.image_paths[idx])
        if not torch.is_tensor(volume):
            volume = torch.as_tensor(np.asarray(volume))
        volume = volume.float()

        if torch.isnan(volume).any() or torch.isinf(volume).any():
            logger.warning(
                f"⚠️ NaN/Inf found in {self.image_paths[idx]} — sanitizing with "
                f"nan_to_num so one bad sample doesn't crash the whole epoch."
            )
            volume = torch.nan_to_num(volume, nan=0.0, posinf=0.0, neginf=0.0)

        view1 = self.augment1(volume.clone())
        view2 = self.augment2(volume.clone())

        if not torch.is_tensor(view1):
            view1 = torch.as_tensor(np.asarray(view1))
        if not torch.is_tensor(view2):
            view2 = torch.as_tensor(np.asarray(view2))

        return view1.float(), view2.float()


# ------------------------------------------------------------------------------
# 10. Model core — encoder UNCHANGED architecture from v0.3/v0.4/v0.5/v0.6.
# ------------------------------------------------------------------------------
class BasicBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv3d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm3d(out_channels)
        self.conv2 = nn.Conv3d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm3d(out_channels)
        self.shortcut = nn.Identity()
        if in_channels != out_channels or stride != 1:
            self.shortcut = nn.Sequential(
                nn.Conv3d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm3d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)


class ResNet3D(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv3d(in_channels, 64, 7, 2, 3, bias=False)
        self.bn1 = nn.BatchNorm3d(64)
        self.pool = nn.MaxPool3d(3, 2, 1)
        self.layer1 = self._make_layer(64, 2, 1)
        self.layer2 = self._make_layer(128, 2, 2)
        self.layer3 = self._make_layer(256, 2, 2)
        self.layer4 = self._make_layer(512, 2, 2)
        self.avgpool = nn.AdaptiveAvgPool3d((1, 1, 1))

        # Channel counts of layer1..layer4 outputs — used by the decoder to
        # size its skip-connection concatenations. Purely descriptive; does
        # not affect the encoder's own computation in any way.
        self.stage_channels = (64, 128, 256, 512)

    def _make_layer(self, out_channels, blocks, stride):
        layers = [BasicBlock3D(self.in_channels, out_channels, stride)]
        self.in_channels = out_channels
        for _ in range(1, blocks):
            layers.append(BasicBlock3D(out_channels, out_channels))
        return nn.Sequential(*layers)

    def forward(self, x, return_feature_map: bool = False, return_feature_maps: bool = False):
        """
        Default call `encoder(x)` (used by SimCLR3D, Mode A): returns just the
        (B, 512) global vector — byte-for-byte identical to v0.4/v0.5/v0.6.

        `encoder(x, return_feature_map=True)`: returns (layer4_feature_map,
        global_vector).

        `encoder(x, return_feature_maps=True)` (used by the segmentation
        path): additionally returns the layer1/2/3 feature maps needed for
        U-Net-style skip connections. The forward computation itself — every
        conv/bn/relu op — is exactly the same regardless of which of these
        flags is set; only which intermediate tensors get returned changes.
        """
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        l1 = self.layer1(x)
        l2 = self.layer2(l1)
        l3 = self.layer3(l2)
        l4 = self.layer4(l3)                               # (B, 512, d, h, w) — spatial
        pooled = self.avgpool(l4)
        global_vector = torch.flatten(pooled, 1)            # (B, 512) — exactly v0.4/v0.5/v0.6's output

        if return_feature_maps:
            return l1, l2, l3, l4, global_vector
        if return_feature_map:
            return l4, global_vector
        return global_vector                                # default path: identical to v0.4/v0.5/v0.6


def resnet18_3d(in_channels=1):
    return ResNet3D(in_channels)


# ------------------------------------------------------------------------------
# 10b. NEW in v0.8 — Channel-wise SE Attention (the ONLY architectural change
#      in v0.8). Defined OUTSIDE ResNet3D so the encoder itself remains
#      architecturally unchanged; SegmentationModel3D applies this only to
#      the l4 bottleneck feature map (see Task 1 of the v0.8 spec).
# ------------------------------------------------------------------------------
class SEChannelAttention3D(nn.Module):
    """Squeeze-and-Excitation channel attention for a 3D feature map.

    AdaptiveAvgPool3d(1) -> FC(C -> C/r) -> ReLU -> FC(C/r -> C) -> Sigmoid
    -> per-channel weights (B, C, 1, 1, 1), broadcast-multiplied against the
    input. Output shape is identical to the input shape; no spatial or
    channel dimensions are altered.
    """
    def __init__(self, channels: int, reduction_ratio: int = Config.SE_REDUCTION_RATIO):
        super().__init__()
        reduced = max(1, channels // reduction_ratio)
        self.avgpool = nn.AdaptiveAvgPool3d(1)
        self.fc1 = nn.Linear(channels, reduced)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(reduced, channels)
        self.sigmoid = nn.Sigmoid()

        # Cached (B, C) channel weights from the most recent forward() call,
        # used only for post-hoc attention logging (Task 1) — never read
        # during training or loss computation.
        self.last_weights: Optional[torch.Tensor] = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c = x.shape[0], x.shape[1]
        squeezed = self.avgpool(x).view(b, c)            # (B, C)
        weights = self.relu(self.fc1(squeezed))
        weights = self.sigmoid(self.fc2(weights))         # (B, C)
        self.last_weights = weights.detach()
        weights = weights.view(b, c, 1, 1, 1)
        return x * weights                                # broadcast over D, H, W


class ProjectionHead(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=512, output_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.net(x)


class SimCLR3D(nn.Module):
    """Unchanged — calls self.encoder(x) with no extra argument, so it only
    ever sees the (B, 512) global vector, exactly as before."""
    def __init__(self, encoder, projection_dim=128):
        super().__init__()
        self.encoder = encoder
        self.projection = ProjectionHead(input_dim=512, output_dim=projection_dim)

    def forward(self, x1, x2, return_features=False):
        h1 = self.encoder(x1)
        h2 = self.encoder(x2)
        z1 = self.projection(h1)
        z2 = self.projection(h2)
        if return_features:
            return z1, z2, h1, h2
        return z1, z2

    def get_encoder(self):
        return self.encoder


def nt_xent_loss(z1, z2, temperature=Config.TEMPERATURE):
    """Standard NT-Xent loss (Chen et al., 2020), unchanged. Inputs are cast
    to float32 so the loss math always runs at full precision, even if the
    encoder/projection ran under autocast (fp16)."""
    z1 = z1.float()
    z2 = z2.float()
    batch_size = z1.shape[0]
    device = z1.device

    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    z = torch.cat([z1, z2], dim=0)

    sim_matrix = torch.mm(z, z.T) / temperature
    mask = torch.eye(2 * batch_size, device=device).bool()
    sim_matrix = sim_matrix.masked_fill(mask, -float('inf'))

    positive_pairs = torch.cat([
        torch.diag(sim_matrix, diagonal=batch_size),
        torch.diag(sim_matrix, diagonal=-batch_size)
    ], dim=0)

    log_sum_exp = torch.logsumexp(sim_matrix, dim=1)
    loss = - (positive_pairs - log_sum_exp).mean()
    return loss


# ------------------------------------------------------------------------------
# 11. Mode A — training loop (unchanged)
# ------------------------------------------------------------------------------
def train_one_epoch_simclr(model, loader, optimizer, device, scaler=None) -> float:
    model.train()
    total_loss = 0.0
    num_batches = 0
    skipped = 0

    pbar = tqdm(loader, desc='SimCLR training', leave=False)
    for view1, view2 in pbar:
        if view1.shape[0] < 2:      # NT-Xent needs >= 2 samples per batch
            skipped += 1
            continue

        view1 = view1.to(device, non_blocking=True)
        view2 = view2.to(device, non_blocking=True)
        optimizer.zero_grad()

        if scaler is not None:
            with torch.cuda.amp.autocast():
                z1, z2 = model(view1, view2)
                loss = nt_xent_loss(z1, z2, temperature=Config.TEMPERATURE)
            if not torch.isfinite(loss):
                logger.warning("⚠️ Non-finite SimCLR loss — skipping this batch's optimizer step.")
                optimizer.zero_grad(set_to_none=True)
                continue
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            z1, z2 = model(view1, view2)
            loss = nt_xent_loss(z1, z2, temperature=Config.TEMPERATURE)
            if not torch.isfinite(loss):
                logger.warning("⚠️ Non-finite SimCLR loss — skipping this batch's optimizer step.")
                optimizer.zero_grad(set_to_none=True)
                continue
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        num_batches += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    if skipped:
        logger.info(f"ℹ️ Skipped {skipped} batch(es) with fewer than 2 samples.")

    return total_loss / num_batches if num_batches else float('nan')


# ------------------------------------------------------------------------------
# 12. Saving helpers (shared, unchanged)
# ------------------------------------------------------------------------------
def save_config(path: str) -> None:
    with open(path, 'w') as f:
        json.dump(Config.to_dict(), f, indent=2)
    logger.info(f"💾 Config saved to {path}")


def save_loss_history(history: List[Tuple[int, float]], path: str) -> None:
    with open(path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['epoch', 'loss'])
        writer.writerows(history)
    logger.info(f"💾 Loss history saved to {path}")


# ------------------------------------------------------------------------------
# 13. Mode A — main runner (unchanged behavior)
# ------------------------------------------------------------------------------
def run_mode_a_simclr() -> None:
    log_section("🚀 MedSSL-3D v0.7-A — Mode A: SimCLR pretraining (identical to v0.4/v0.5/v0.6)")

    print_environment_report()
    Config.setup_dirs()
    set_seed(Config.SEED)
    device = torch.device(Config.DEVICE)

    logger.info(f"\n📂 DATA_ROOT : {Config.DATA_ROOT}")
    logger.info(f"📂 MODALITY  : {Config.MODALITY}")
    logger.info(f"📂 Checkpoints -> {Config.CHECKPOINT_DIR}")
    logger.info(f"📂 Logs        -> {Config.LOG_DIR}")

    image_paths = run_data_sanity_check(Config.DATA_ROOT, Config.MODALITY, Config.IMG_SIZE)

    batch_size = Config.BATCH_SIZE
    if len(image_paths) < batch_size:
        logger.warning(
            f"⚠️ Dataset has only {len(image_paths)} sample(s), fewer than "
            f"BATCH_SIZE={batch_size}. Reducing BATCH_SIZE to {len(image_paths)}."
        )
        batch_size = len(image_paths)
    if batch_size < 2:
        raise SystemExit("❌ At least 2 samples are required for NT-Xent loss.")

    dataset = BraTSDataset(image_paths, img_size=Config.IMG_SIZE, is_train=True)
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=True,
        num_workers=Config.NUM_WORKERS, pin_memory=(device.type == 'cuda'), drop_last=False,
    )

    encoder = resnet18_3d(in_channels=1).to(device)
    model = SimCLR3D(encoder=encoder, projection_dim=Config.PROJECTION_DIM).to(device)
    optimizer = optim.Adam(model.parameters(), lr=Config.LR)

    scaler = None
    if device.type == 'cuda':
        scaler = torch.cuda.amp.GradScaler()
        logger.info("⚡ Mixed precision (torch.cuda.amp) enabled.")

    loss_history: List[Tuple[int, float]] = []
    best_loss = float('inf')

    for epoch in range(1, Config.NUM_EPOCHS + 1):
        logger.info(f"\n{'-' * 40}\nEpoch {epoch}/{Config.NUM_EPOCHS}")
        epoch_loss = train_one_epoch_simclr(model, loader, optimizer, device, scaler)

        if epoch_loss != epoch_loss:   # NaN check
            logger.warning("⚠️ No valid batches this epoch — skipping save.")
            continue

        loss_history.append((epoch, epoch_loss))
        logger.info(f"📉 Train loss: {epoch_loss:.4f}")

        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': epoch_loss,
        }
        torch.save(checkpoint, os.path.join(Config.CHECKPOINT_DIR, f'checkpoint_epoch_{epoch}.pth'))

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save(checkpoint, os.path.join(Config.CHECKPOINT_DIR, 'best_model.pth'))
            torch.save(model.get_encoder().state_dict(),
                       os.path.join(Config.CHECKPOINT_DIR, 'encoder.pth'))
            logger.info(f"✅ New best model + encoder saved (loss: {epoch_loss:.4f})")

    save_config(os.path.join(Config.CHECKPOINT_DIR, 'config.json'))
    if loss_history:
        save_loss_history(loss_history, os.path.join(Config.LOG_DIR, 'loss_history.csv'))

        plt.figure(figsize=(8, 5))
        epochs, losses = zip(*loss_history)
        plt.plot(epochs, losses, marker='o')
        plt.xlabel('Epoch')
        plt.ylabel('NT-Xent Loss')
        plt.title('SimCLR Training Loss over Epochs')
        plt.grid(True)
        plt.savefig(os.path.join(Config.LOG_DIR, 'loss_plot.png'), dpi=150)
        plt.show()

    if loss_history:
        log_section("🎉 Mode A RUN COMPLETE")
        logger.info(f"🏆 Best loss: {best_loss:.4f}")
        logger.info(f"💾 Encoder saved to: {Config.CHECKPOINT_DIR}/encoder.pth")
        logger.info("📌 This encoder.pth is what Mode B (segmentation) loads automatically.")
    else:
        log_section("⚠️ Mode A RUN FINISHED WITH NO VALID EPOCHS")
        logger.warning("⚠️ Training did not produce any valid epoch — nothing was saved.")


# ==============================================================================
#         ↓↓↓  MODE B — TRAINING PROTOCOL UPDATED IN v0.7-A  ↓↓↓
#   Architecture (Decoder/Skip/Deep Supervision), loss, and core Dice/IoU
#   metric are UNCHANGED from v0.6. Only the split, checkpoint criterion,
#   scheduler, multi-seed loop, and logging are new in this version.
# ==============================================================================

# ------------------------------------------------------------------------------
# 14. v0.6 Residual U-Net-style decoder — UNCHANGED, byte-for-byte from v0.6.
# ------------------------------------------------------------------------------
class ResidualDecoderBlock3D(nn.Module):
    """One U-Net decoder stage WITH a skip connection.

    Upsample(in_channels -> out_channels) -> concat with skip
    (skip_channels) -> [Conv3d-BN-ReLU -> Conv3d-BN] -> residual add
    (1x1x1 projection of the concatenated tensor, to match channel counts)
    -> ReLU.
    """
    def __init__(self, in_channels: int, skip_channels: int, out_channels: int):
        super().__init__()
        self.upsample = nn.ConvTranspose3d(in_channels, out_channels, kernel_size=2, stride=2)

        concat_channels = out_channels + skip_channels

        self.conv1 = nn.Conv3d(concat_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm3d(out_channels)
        self.conv2 = nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm3d(out_channels)

        self.shortcut = nn.Sequential(
            nn.Conv3d(concat_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm3d(out_channels),
        )

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.upsample(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)

        x = torch.cat([x, skip], dim=1)

        residual = self.shortcut(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + residual
        return F.relu(out)


class ResidualUpBlock3D(nn.Module):
    """One decoder stage with NO skip connection (used past the shallowest
    encoder skip, to recover full input resolution)."""
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.upsample = nn.ConvTranspose3d(in_channels, out_channels, kernel_size=2, stride=2)

        self.conv1 = nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm3d(out_channels)
        self.conv2 = nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm3d(out_channels)

        self.shortcut = nn.Sequential(
            nn.Conv3d(out_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm3d(out_channels),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.upsample(x)
        residual = self.shortcut(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + residual
        return F.relu(out)


class AuxHead3D(nn.Module):
    """Deep-supervision auxiliary head: 1x1x1 conv -> interpolate to full
    target resolution. Never used at inference time."""
    def __init__(self, in_channels: int, num_classes: int, target_size: Tuple[int, int, int]):
        super().__init__()
        self.conv = nn.Conv3d(in_channels, num_classes, kernel_size=1)
        self.target_size = target_size

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv(x)
        if tuple(x.shape[2:]) != tuple(self.target_size):
            x = F.interpolate(x, size=self.target_size, mode='trilinear', align_corners=False)
        return x


class ResidualDecoder3D(nn.Module):
    """
    Lightweight 3D U-Net-style decoder with classical concatenation skip
    connections, residual decoder blocks, and two deep-supervision auxiliary
    outputs. Channel progression: 512 -> 256 -> 128 -> 64 -> 32 -> num_classes.
    UNCHANGED from v0.6.
    """
    def __init__(self, num_classes: int = 4,
                 target_size: Tuple[int, int, int] = Config.IMG_SIZE,
                 encoder_channels: Tuple[int, int, int, int] = (64, 128, 256, 512)):
        super().__init__()
        c1, c2, c3, c4 = encoder_channels  # layer1..layer4 output channels
        self.target_size = target_size

        # ---- Skip-connected stages (mirror encoder layer3 -> layer1) ----
        self.dec1 = ResidualDecoderBlock3D(in_channels=c4, skip_channels=c3, out_channels=256)  # Decoder Stage 1
        self.dec2 = ResidualDecoderBlock3D(in_channels=256, skip_channels=c2, out_channels=128)  # Decoder Stage 2
        self.dec3 = ResidualDecoderBlock3D(in_channels=128, skip_channels=c1, out_channels=64)   # Decoder Stage 3

        # ---- Skip-free stages: recover the encoder stem's 4x downsampling ----
        self.dec4 = ResidualUpBlock3D(in_channels=64, out_channels=32)
        self.dec5 = ResidualUpBlock3D(in_channels=32, out_channels=32)

        self.classifier = nn.Conv3d(32, num_classes, kernel_size=1)

        # ---- Deep supervision: exactly two auxiliary heads ----
        self.aux_head1 = AuxHead3D(in_channels=256, num_classes=num_classes, target_size=target_size)  # after Decoder Stage 1
        self.aux_head2 = AuxHead3D(in_channels=64, num_classes=num_classes, target_size=target_size)   # after Decoder Stage 3

    def forward(self, l1: torch.Tensor, l2: torch.Tensor, l3: torch.Tensor, l4: torch.Tensor
                ) -> Union[torch.Tensor, Tuple[torch.Tensor, torch.Tensor, torch.Tensor]]:
        d1 = self.dec1(l4, l3)    # Decoder Stage 1 -> 256ch
        d2 = self.dec2(d1, l2)    # Decoder Stage 2 -> 128ch
        d3 = self.dec3(d2, l1)    # Decoder Stage 3 -> 64ch
        d4 = self.dec4(d3)        # -> 32ch, +1 upsample
        d5 = self.dec5(d4)        # -> 32ch, +1 upsample (back to full input resolution)

        main_out = self.classifier(d5)
        if tuple(main_out.shape[2:]) != tuple(self.target_size):
            main_out = F.interpolate(main_out, size=self.target_size, mode='trilinear', align_corners=False)

        if self.training:
            aux1 = self.aux_head1(d1)   # deep supervision after Decoder Stage 1
            aux2 = self.aux_head2(d3)   # deep supervision after Decoder Stage 3
            return main_out, aux1, aux2

        # Inference: only the main segmentation is ever returned.
        return main_out


class SegmentationModel3D(nn.Module):
    """Wraps the SAME ResNet3D class used for SimCLR — an encoder.pth can be
    loaded directly into `self.encoder`, no key renaming needed. Requests
    multi-scale features (return_feature_maps=True) and feeds them to
    ResidualDecoder3D. UNCHANGED from v0.6."""
    def __init__(self, encoder: ResNet3D, num_classes: int = 4,
                 target_size: Tuple[int, int, int] = Config.IMG_SIZE):
        super().__init__()
        self.encoder = encoder
        self.decoder = ResidualDecoder3D(num_classes=num_classes, target_size=target_size,
                                          encoder_channels=encoder.stage_channels)

        # NEW in v0.8: optional Channel-wise SE Attention on the l4 bottleneck
        # ONLY. Gated by Config.USE_CHANNEL_SE_ATTENTION for clean A/B
        # comparison — when False, self.channel_attention stays None and
        # forward() below is byte-for-byte identical to v0.7-B.
        self.channel_attention = None
        if Config.USE_CHANNEL_SE_ATTENTION:
            self.channel_attention = SEChannelAttention3D(
                channels=encoder.stage_channels[-1], reduction_ratio=Config.SE_REDUCTION_RATIO,
            )

    def forward(self, x: torch.Tensor, return_bottleneck: bool = False):
        """
        NEW in v0.9 — Task A: optional `return_bottleneck`.

        Default (return_bottleneck=False) is byte-for-byte identical to v0.8:
        the computation below is unchanged, and the same object is returned.
        When True, the (already SE-reweighted) l4 bottleneck is ALSO returned
        so MultiTaskModel3D's classification head can read it — previously l4
        was computed internally and never exposed. No extra forward work is
        done in either case.
        """
        l1, l2, l3, l4, _ = self.encoder(x, return_feature_maps=True)

        # NEW in v0.8: reweight ONLY l4 (skip features l1, l2, l3 untouched).
        if self.channel_attention is not None:
            l4 = self.channel_attention(l4)

        if torch.isnan(l4).any() or torch.isinf(l4).any():
            logger.warning(
                f"🔴 encoder feature_map is corrupted | min={l4.min().item():.4f} "
                f"max={l4.max().item():.4f} nan_count={torch.isnan(l4).sum().item()}"
            )

        output = self.decoder(l1, l2, l3, l4)

        if self.training:
            main_out, aux1, aux2 = output
            for name, t in (('main', main_out), ('aux1', aux1), ('aux2', aux2)):
                if torch.isnan(t).any() or torch.isinf(t).any():
                    logger.warning(
                        f"🔴 {name} logits are corrupted | min={t.min().item():.4f} "
                        f"max={t.max().item():.4f} nan_count={torch.isnan(t).sum().item()}"
                    )
            if return_bottleneck:                                  # NEW in v0.9
                return (main_out, aux1, aux2), l4
            return main_out, aux1, aux2

        main_out = output
        if torch.isnan(main_out).any() or torch.isinf(main_out).any():
            logger.warning(
                f"🔴 logits are corrupted | min={main_out.min().item():.4f} "
                f"max={main_out.max().item():.4f} nan_count={torch.isnan(main_out).sum().item()}"
            )
        if return_bottleneck:                                      # NEW in v0.9
            return main_out, l4
        return main_out

    def get_encoder(self) -> ResNet3D:
        return self.encoder


# ------------------------------------------------------------------------------
# 15. Dice + Cross-Entropy loss — UNCHANGED from v0.5/v0.6.
# ------------------------------------------------------------------------------
class DiceLoss(nn.Module):
    def __init__(self, num_classes: int, smooth: float = 1e-5, include_background: bool = True):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
        self.include_background = include_background

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=1)
        target_onehot = F.one_hot(target.long(), num_classes=self.num_classes)
        target_onehot = target_onehot.permute(0, 4, 1, 2, 3).float()

        start_c = 0 if self.include_background else 1
        probs = probs[:, start_c:]
        target_onehot = target_onehot[:, start_c:]

        dims = (0, 2, 3, 4)
        intersection = torch.sum(probs * target_onehot, dim=dims)
        cardinality = torch.sum(probs + target_onehot, dim=dims)
        dice_per_class = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        dice_per_class = torch.clamp(dice_per_class, min=0.0, max=1.0)
        return 1.0 - dice_per_class.mean()


class DiceCELoss(nn.Module):
    def __init__(self, num_classes: int, ce_weight: float = 1.0, dice_weight: float = 1.0):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self.dice = DiceLoss(num_classes=num_classes)
        self.ce_weight = ce_weight
        self.dice_weight = dice_weight

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        logits = logits.float()
        ce_loss = self.ce(logits, target.long())
        dice_loss = self.dice(logits, target)

        if not torch.isfinite(ce_loss):
            logger.warning(f"🔴 CE loss is non-finite: {ce_loss.item()}")
        if not torch.isfinite(dice_loss):
            logger.warning(f"🔴 Dice loss is non-finite: {dice_loss.item()}")

        return self.ce_weight * ce_loss + self.dice_weight * dice_loss


# ------------------------------------------------------------------------------
# 16. Image <-> mask pairing + BraTS label remap — UNCHANGED from v0.5/v0.6
# ------------------------------------------------------------------------------
def find_seg_path(image_path: str, modality: str) -> Optional[str]:
    """Given '..._flair.nii.gz', try to find the matching '..._seg.nii.gz'."""
    candidates = [
        image_path.replace(f'{modality}.nii.gz', 'seg.nii.gz'),
        image_path.replace(f'{modality}.nii', 'seg.nii'),
    ]
    for candidate in candidates:
        if candidate != image_path and os.path.isfile(candidate):
            return candidate
    return None


# ------------------------------------------------------------------------------
# 16b. v0.7-B, NEW: derive T1/T1CE/T2 paths from the frozen FLAIR path.
#      Uses the same suffix-anchored substitution convention as
#      find_seg_path() above (never a bare/global string replace), so
#      unrelated substrings in the path can never be accidentally corrupted.
#      The persisted data_split_v1.json still stores only the FLAIR path per
#      patient — this pairing IS the frozen split and is never touched.
# ------------------------------------------------------------------------------
def get_modality_path(flair_path: str, target_modality: str) -> str:
    for suffix in ('flair.nii.gz', 'flair.nii'):
        if suffix in flair_path:
            return flair_path.replace(suffix, suffix.replace('flair', target_modality))
    raise ValueError(f"Could not derive '{target_modality}' path from {flair_path}")


def remap_brats_labels(mask: torch.Tensor) -> torch.Tensor:
    """BraTS uses label 4 for 'enhancing tumor'; remap to 3 for contiguous 0-3 labels."""
    mask = mask.clone()
    mask[mask == 4] = 3
    return mask


# ------------------------------------------------------------------------------
# 17. Mode B — Data Sanity Check — UNCHANGED from v0.5/v0.6
# ------------------------------------------------------------------------------
def run_segmentation_sanity_check(data_root: str, modality: str,
                                   img_size: Tuple[int, int, int]) -> List[Tuple[str, str]]:
    """
    Pre-flight check for the segmentation (Mode B) data pipeline:
      - directory exists
      - image files for the modality are found
      - each image has a matching '..._seg.nii.gz' mask
      - the first pair loads, with matching shapes and no NaN/Inf
      - mask label values are inspected and the BraTS 4->3 remap is verified
      - a middle slice with the ground-truth overlay is displayed
    Raises SystemExit with a clear message on any failure.
    """
    log_section("🔍 DATA SANITY CHECK (Mode B — Segmentation)")

    if not os.path.isdir(data_root):
        raise SystemExit(f"❌ DATA_ROOT does not exist: '{data_root}'")
    logger.info(f"✅ Directory exists: {data_root}")

    pattern = os.path.join(data_root, '**', f'*{modality}*.nii*')
    image_paths = sorted(glob.glob(pattern, recursive=True))
    if len(image_paths) == 0:
        raise SystemExit(f"❌ No files matching modality '{modality}' found under '{data_root}'.")
    logger.info(f"✅ Found {len(image_paths)} '{modality}' file(s)")

    pairs: List[Tuple[str, str]] = []
    skipped = 0
    for img_path in image_paths:
        seg_path = find_seg_path(img_path, modality)
        if seg_path is not None:
            pairs.append((img_path, seg_path))
        else:
            skipped += 1

    if skipped:
        logger.warning(f"⚠️ {skipped} image file(s) had no matching segmentation mask and were skipped.")
    if len(pairs) == 0:
        raise SystemExit(
            "❌ No (image, segmentation) pairs could be matched.\n"
            "   -> Confirm each case has a '..._seg.nii.gz' file alongside the modality file."
        )
    logger.info(f"✅ Matched {len(pairs)} (image, mask) pair(s)")
    logger.info(
        f"📄 First pair:\n"
        f"   image: {os.path.basename(pairs[0][0])}\n"
        f"   mask : {os.path.basename(pairs[0][1])}"
    )

    img_transform = build_base_transform(img_size)
    mask_transform = build_mask_transform(img_size)

    try:
        volume = img_transform(pairs[0][0])
        mask = mask_transform(pairs[0][1])
    except Exception as e:
        raise SystemExit(f"❌ Failed to load the first (image, mask) pair: {e}")

    if not torch.is_tensor(volume):
        volume = torch.as_tensor(np.asarray(volume))
    if not torch.is_tensor(mask):
        mask = torch.as_tensor(np.asarray(mask))

    volume = volume.float()
    mask = mask.round().long()

    logger.info(f"📦 Volume shape: {tuple(volume.shape)} | Mask shape: {tuple(mask.shape)}")
    if volume.shape[1:] != mask.shape[1:]:
        raise SystemExit(
            f"❌ Volume and mask spatial shapes do not match: "
            f"{tuple(volume.shape)} vs {tuple(mask.shape)}."
        )

    if torch.isnan(volume).any() or torch.isinf(volume).any():
        raise SystemExit(f"❌ NaN/Inf detected in volume '{pairs[0][0]}'.")

    unique_vals = sorted(torch.unique(mask).tolist())
    logger.info(f"🏷️ Raw mask label values found: {unique_vals}")
    expected_raw = {0, 1, 2, 4}
    unexpected = set(unique_vals) - expected_raw
    if unexpected:
        logger.warning(
            f"⚠️ Unexpected label values {sorted(unexpected)} found (standard BraTS uses "
            f"{sorted(expected_raw)}). They will be clamped during training, but double-check your labels."
        )

    mask_remapped = remap_brats_labels(mask)
    logger.info(f"🏷️ Remapped label values (4->3): {sorted(torch.unique(mask_remapped).tolist())}")

    try:
        d = volume.shape[-1] // 2
        img_slice = volume[0, :, :, d].numpy()
        mask_slice = mask_remapped[0, :, :, d].numpy()
        plt.figure(figsize=(4, 4))
        plt.imshow(img_slice, cmap='gray')
        plt.imshow(np.ma.masked_where(mask_slice == 0, mask_slice), cmap='jet', alpha=0.5)
        plt.title("First pair — middle slice + GT overlay")
        plt.axis('off')
        plt.show()
        logger.info("✅ Middle slice + ground-truth overlay displayed above.")
    except Exception as e:
        logger.warning(f"⚠️ Could not display middle slice (non-fatal): {e}")

    logger.info("\n✅ SEGMENTATION SANITY CHECK PASSED.\n")
    return pairs


# ------------------------------------------------------------------------------
# 18. Mode B — Dataset — UNCHANGED from v0.5/v0.6.
#     Patient-level split logic is NEW in v0.7-A (create_or_load_split
#     replaces v0.6's split_train_val, which only produced a 2-way split
#     with no persistence and no Test set).
# ------------------------------------------------------------------------------
class BraTSSegmentationDataset(Dataset):
    """
    NEW in v0.7-B: loads all FOUR MRI modalities (FLAIR, T1, T1CE, T2) per
    patient and stacks them into a single (4, D, H, W) tensor — Early
    Fusion input. T1/T1CE/T2 paths are derived at load time from the
    frozen FLAIR path already stored in data_split_v1.json (see
    get_modality_path() above); the split file itself is never touched.

    Fixed channel order (must remain constant throughout the project):
        0 -> FLAIR, 1 -> T1, 2 -> T1CE, 3 -> T2

    NOTE: no random augmentation is applied — only the deterministic
    resize also used for Mode A's base transform, applied independently
    to each of the four modality files — specifically to avoid
    desynchronizing image/mask under independently sampled random
    transforms. Synchronized augmentation remains a well-scoped addition
    for a later version.
    """

    MODALITY_ORDER = ('flair', 't1', 't1ce', 't2')

    def __init__(self, pairs: List[Tuple[str, str]],
                 img_size: Tuple[int, int, int] = Config.IMG_SIZE,
                 num_classes: int = 4):
        self.pairs = pairs
        self.img_size = img_size
        self.num_classes = num_classes
        self.img_transform = build_base_transform(img_size)
        self.mask_transform = build_mask_transform(img_size)

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int):
        img_path, seg_path = self.pairs[idx]   # img_path = frozen FLAIR path from the split

        # Derive the other three modality paths from the frozen FLAIR path.
        modality_paths = {'flair': img_path}
        for modality in ('t1', 't1ce', 't2'):
            modality_paths[modality] = get_modality_path(img_path, modality)

        # Load + deterministically resize each modality independently, then
        # stack in the fixed channel order: FLAIR, T1, T1CE, T2.
        channel_volumes = []
        for modality in self.MODALITY_ORDER:
            vol = self.img_transform(modality_paths[modality])
            if not torch.is_tensor(vol):
                vol = torch.as_tensor(np.asarray(vol))
            channel_volumes.append(vol.float())   # each (1, D, H, W)
        volume = torch.cat(channel_volumes, dim=0)  # (4, D, H, W)

        mask = self.mask_transform(seg_path)
        if not torch.is_tensor(mask):
            mask = torch.as_tensor(np.asarray(mask))

        mask = remap_brats_labels(mask.round().long())

        # Structural bug (config/generation issue, not sensor noise) — fail
        # loudly rather than silently reshaping mismatched image/mask pairs.
        if volume.shape[1:] != mask.shape[1:]:
            raise ValueError(
                f"❌ Volume/mask spatial shape mismatch for pair "
                f"({img_path}, {seg_path}): {tuple(volume.shape)} vs {tuple(mask.shape)}."
            )

        if torch.isnan(volume).any() or torch.isinf(volume).any():
            logger.warning(
                f"⚠️ NaN/Inf found in one or more modalities for {img_path} — sanitizing "
                f"with nan_to_num so one bad sample doesn't crash the whole epoch."
            )
            volume = torch.nan_to_num(volume, nan=0.0, posinf=0.0, neginf=0.0)

        mask = mask.squeeze(0)  # (1, H, W, D) -> (H, W, D)
        mask = torch.clamp(mask, min=0, max=self.num_classes - 1)

        return volume, mask


def create_or_load_split(pairs: List[Tuple[str, str]], split_path: str,
                          train_frac: float, val_frac: float, test_frac: float,
                          seed: int) -> Tuple[List[Tuple[str, str]], List[Tuple[str, str]], List[Tuple[str, str]]]:
    """
    v0.7-A: patient-level Train/Val/Test split (70/15/15 by default).

    Each entry in `pairs` is a matched (image_path, mask_path) tuple, i.e.
    already one entry per patient (BraTS files are named per-patient per
    modality, so this list IS the patient list).

    If `split_path` already exists, the split is LOADED and NOT recomputed —
    this is what guarantees v0.7-B / v0.8 / v0.9 all train/val/test on the
    exact same patients as this v0.7-A run. The Test set is returned here
    purely for counting/logging; nothing in v0.7-A wraps it in a Dataset or
    evaluates it — see run_mode_b_segmentation() below.
    """
    log_section("📁 PATIENT-LEVEL TRAIN/VAL/TEST SPLIT (v0.7-A)")

    if os.path.isfile(split_path):
        logger.info(f"✅ Found existing split file: {split_path}")
        with open(split_path, 'r') as f:
            split = json.load(f)
        train_pairs = [tuple(p) for p in split['train']]
        val_pairs = [tuple(p) for p in split['val']]
        test_pairs = [tuple(p) for p in split['test']]
        logger.info("📌 Loaded split (NOT recomputed) — ensures identical patients "
                     "across v0.7-A / v0.7-B / v0.8 / v0.9.")
    else:
        logger.info(f"ℹ️ No split file found at {split_path} — creating a new patient-level split.")
        if abs((train_frac + val_frac + test_frac) - 1.0) > 1e-6:
            raise SystemExit(
                f"❌ TRAIN/VAL/TEST fractions must sum to 1.0, got "
                f"{train_frac} + {val_frac} + {test_frac}."
            )

        rng = random.Random(seed)
        shuffled = pairs[:]
        rng.shuffle(shuffled)

        n = len(shuffled)
        n_train = int(round(n * train_frac))
        n_val = int(round(n * val_frac))
        n_test = n - n_train - n_val   # remainder -> test, guarantees all patients accounted for

        if n >= 3:
            # Defensive: guarantee every split is non-empty when possible.
            n_train = max(n_train, 1)
            n_val = max(n_val, 1)
            n_test = max(n_test, 1)
            while n_train + n_val + n_test > n:
                n_train -= 1

        train_pairs = shuffled[:n_train]
        val_pairs = shuffled[n_train:n_train + n_val]
        test_pairs = shuffled[n_train + n_val:n_train + n_val + n_test]

        split = {
            'seed': seed,
            'train_fraction': train_frac,
            'val_fraction': val_frac,
            'test_fraction': test_frac,
            'train': [list(p) for p in train_pairs],
            'val': [list(p) for p in val_pairs],
            'test': [list(p) for p in test_pairs],
        }
        os.makedirs(os.path.dirname(split_path), exist_ok=True)
        with open(split_path, 'w') as f:
            json.dump(split, f, indent=2)
        logger.info(f"💾 New split saved to {split_path}")

    logger.info(
        f"📊 Train: {len(train_pairs)} | Val: {len(val_pairs)} | "
        f"Test: {len(test_pairs)} (reserved, untouched in v0.7-A)"
    )
    logger.info("🔒 Test set will NOT be evaluated or visualized anywhere in v0.7-A — reserved for v1.0.")
    return train_pairs, val_pairs, test_pairs


# ------------------------------------------------------------------------------
# 19. Mode B — Evaluation metrics + visualization.
#     compute_dice_iou / visualize_segmentation / evaluate_segmentation are
#     UNCHANGED from v0.5/v0.6 (kept for compatibility). NEW in v0.7-A:
#     compute_region_dice (WT/TC/ET) and evaluate_segmentation_full (adds
#     val loss + region metrics on top of the unchanged core function).
# ------------------------------------------------------------------------------
@torch.no_grad()
def compute_dice_iou(logits: torch.Tensor, target: torch.Tensor,
                      num_classes: int, smooth: float = 1e-5) -> Tuple[float, float]:
    """Hard Dice/IoU per foreground class (background is skipped — standard
    practice, since background dominates volume and is uninformative)."""
    pred = torch.argmax(logits, dim=1)
    dice_scores, iou_scores = [], []

    for c in range(1, num_classes):
        pred_c = (pred == c).float()
        target_c = (target == c).float()

        if target_c.sum() == 0:
            continue

        intersection = (pred_c * target_c).sum()
        union = pred_c.sum() + target_c.sum() - intersection
        dice = (2 * intersection + smooth) / (pred_c.sum() + target_c.sum() + smooth)
        iou = (intersection + smooth) / (union + smooth)
        dice_scores.append(dice.item())
        iou_scores.append(iou.item())

    mean_dice = float(np.mean(dice_scores)) if dice_scores else float('nan')
    mean_iou = float(np.mean(iou_scores)) if iou_scores else float('nan')
    return mean_dice, mean_iou


@torch.no_grad()
def compute_region_dice(logits: torch.Tensor, target: torch.Tensor,
                         smooth: float = 1e-5) -> Tuple[float, float, float]:
    """
    NEW in v0.7-A. BraTS-style region Dice computed on the already-remapped
    {0,1,2,3} labels used throughout this codebase:
        WT (Whole Tumor)     = union of classes {1, 2, 3}
        TC (Tumor Core)      = union of classes {1, 3}
        ET (Enhancing Tumor) = class {3}
    Standard binary Dice per region. A region is excluded (NaN) from a
    sample's contribution if it's absent from that sample's ground truth,
    matching the convention already used by compute_dice_iou.
    """
    pred = torch.argmax(logits, dim=1)

    def region_dice(pred_mask: torch.Tensor, target_mask: torch.Tensor) -> float:
        if target_mask.sum() == 0:
            return float('nan')
        intersection = (pred_mask * target_mask).sum()
        dice = (2 * intersection + smooth) / (pred_mask.sum() + target_mask.sum() + smooth)
        return dice.item()

    pred_wt = ((pred == 1) | (pred == 2) | (pred == 3)).float()
    target_wt = ((target == 1) | (target == 2) | (target == 3)).float()

    pred_tc = ((pred == 1) | (pred == 3)).float()
    target_tc = ((target == 1) | (target == 3)).float()

    pred_et = (pred == 3).float()
    target_et = (target == 3).float()

    wt = region_dice(pred_wt, target_wt)
    tc = region_dice(pred_tc, target_tc)
    et = region_dice(pred_et, target_et)
    return wt, tc, et


def visualize_segmentation(volume: torch.Tensor, gt_mask: torch.Tensor, pred_mask: torch.Tensor,
                            slice_idx: Optional[int] = None, save_path: Optional[str] = None) -> None:
    # v0.7-B: volume is now (4, D, H, W) — squeeze() no longer drops the
    # channel dim. Explicitly select channel 0 (FLAIR) for display only;
    # this does not affect training, metrics, or any scientific result.
    volume_np = volume[0].numpy()   # channel 0 = FLAIR, for display only
    gt_np = gt_mask.squeeze().numpy()
    pred_np = pred_mask.squeeze().numpy()

    if slice_idx is None:
        slice_idx = volume_np.shape[-1] // 2

    img_slice = volume_np[:, :, slice_idx]
    gt_slice = gt_np[:, :, slice_idx]
    pred_slice = pred_np[:, :, slice_idx]

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img_slice, cmap='gray')
    axes[0].set_title('MRI slice')
    axes[0].axis('off')

    axes[1].imshow(img_slice, cmap='gray')
    axes[1].imshow(np.ma.masked_where(gt_slice == 0, gt_slice), cmap='jet', alpha=0.5, vmin=0, vmax=3)
    axes[1].set_title('Ground truth overlay')
    axes[1].axis('off')

    axes[2].imshow(img_slice, cmap='gray')
    axes[2].imshow(np.ma.masked_where(pred_slice == 0, pred_slice), cmap='jet', alpha=0.5, vmin=0, vmax=3)
    axes[2].set_title('Prediction overlay')
    axes[2].axis('off')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()


@torch.no_grad()
def evaluate_segmentation(model, loader, device, num_classes, visualize=False, save_path=None):
    """UNCHANGED from v0.5/v0.6 — kept for compatibility with any external
    code that imports this function directly. v0.7-A's own training loop
    uses evaluate_segmentation_full (below) instead, which adds val loss
    and WT/TC/ET without altering this function's behavior."""
    model.eval()
    dice_scores, iou_scores = [], []
    first_batch_for_viz = None

    for volume, mask in loader:
        volume = volume.to(device)
        mask = mask.to(device)
        logits = model(volume)   # eval mode -> ResidualDecoder3D returns main_out only
        dice, iou = compute_dice_iou(logits, mask, num_classes=num_classes)
        dice_scores.append(dice)
        iou_scores.append(iou)

        if visualize and first_batch_for_viz is None:
            pred = torch.argmax(logits, dim=1)
            first_batch_for_viz = (volume[0].cpu(), mask[0].cpu(), pred[0].cpu())

    mean_dice = float(np.nanmean(dice_scores)) if dice_scores else float('nan')
    mean_iou = float(np.nanmean(iou_scores)) if iou_scores else float('nan')

    if visualize and first_batch_for_viz is not None:
        vol, gt, pred = first_batch_for_viz
        visualize_segmentation(vol, gt, pred, save_path=save_path)

    return mean_dice, mean_iou


@torch.no_grad()
def evaluate_segmentation_full(model, loader, device, num_classes, criterion=None,
                                visualize=False, save_path=None) -> dict:
    """
    NEW in v0.7-A. Extends the unchanged evaluate_segmentation() with:
      - val loss (using the same DiceCELoss criterion as training, if given)
      - WT / TC / ET region Dice (compute_region_dice)
    without modifying the original function or the 4-class Dice/IoU logic.
    """
    model.eval()
    dice_scores, iou_scores = [], []
    wt_scores, tc_scores, et_scores = [], [], []
    loss_values = []
    first_batch_for_viz = None

    for volume, mask in loader:
        volume = volume.to(device)
        mask = mask.to(device)
        logits = model(volume)

        dice, iou = compute_dice_iou(logits, mask, num_classes=num_classes)
        wt, tc, et = compute_region_dice(logits, mask)
        dice_scores.append(dice)
        iou_scores.append(iou)
        wt_scores.append(wt)
        tc_scores.append(tc)
        et_scores.append(et)

        if criterion is not None:
            loss_values.append(criterion(logits, mask).item())

        if visualize and first_batch_for_viz is None:
            pred = torch.argmax(logits, dim=1)
            first_batch_for_viz = (volume[0].cpu(), mask[0].cpu(), pred[0].cpu())

    results = {
        'dice': float(np.nanmean(dice_scores)) if dice_scores else float('nan'),
        'iou': float(np.nanmean(iou_scores)) if iou_scores else float('nan'),
        'wt': float(np.nanmean(wt_scores)) if wt_scores else float('nan'),
        'tc': float(np.nanmean(tc_scores)) if tc_scores else float('nan'),
        'et': float(np.nanmean(et_scores)) if et_scores else float('nan'),
        'val_loss': float(np.mean(loss_values)) if loss_values else float('nan'),
    }

    if visualize and first_batch_for_viz is not None:
        vol, gt, pred = first_batch_for_viz
        visualize_segmentation(vol, gt, pred, save_path=save_path)

    return results


# ------------------------------------------------------------------------------
# 19b. NEW in v0.8 — Task 1: post-training Channel Attention logging.
#      Evaluation only: runs one inference pass over the validation set and
#      records the mean SE channel-attention weight per channel. Does not
#      modify training, optimization, gradients, or segmentation results.
# ------------------------------------------------------------------------------
@torch.no_grad()
def run_channel_attention_logging(model: 'SegmentationModel3D', loader: DataLoader,
                                   device: torch.device, seed: int, output_dir: str) -> Optional[dict]:
    if model.channel_attention is None:
        logger.info("ℹ️ USE_CHANNEL_SE_ATTENTION=False — skipping channel attention logging.")
        return None

    log_section(f"🎛️ v0.8 — Channel Attention logging (seed {seed})")
    model.eval()
    all_weights = []
    for volume, _mask in loader:
        volume = volume.to(device)
        _ = model(volume)   # channel_attention.last_weights is cached as a side effect
        all_weights.append(model.channel_attention.last_weights.cpu())

    weights_tensor = torch.cat(all_weights, dim=0)          # (N, C)
    mean_per_channel = weights_tensor.mean(dim=0).numpy()

    os.makedirs(output_dir, exist_ok=True)
    csv_path = os.path.join(output_dir, f'channel_attention_weights_seed_{seed}.csv')
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['channel_idx', 'mean_weight'])
        for idx, w in enumerate(mean_per_channel):
            writer.writerow([idx, float(w)])

    overall_mean = float(mean_per_channel.mean())
    overall_var = float(mean_per_channel.var())
    logger.info(f"🎛️ Seed {seed} channel attention — overall mean={overall_mean:.4f} | overall variance={overall_var:.6f}")
    logger.info(f"💾 Channel attention weights saved to {csv_path}")

    return {'csv_path': csv_path, 'overall_mean': overall_mean, 'overall_var': overall_var}


# ------------------------------------------------------------------------------
# 19c. NEW in v0.8 — Task 2: post-hoc Modality Ablation.
#      Evaluation only, executed after the best segmentation model has
#      already been selected. No trainable parameters are introduced and the
#      training loop is never touched.
# ------------------------------------------------------------------------------
@torch.no_grad()
def run_modality_ablation(model: 'SegmentationModel3D', loader: DataLoader, device: torch.device,
                           seed: int, output_dir: str, baseline_dice: Optional[float] = None) -> dict:
    log_section(f"🧪 v0.8 — Post-hoc Modality Ablation (seed {seed})")
    model.eval()

    if baseline_dice is None:
        baseline_results = evaluate_segmentation_full(model, loader, device, num_classes=Config.NUM_SEG_CLASSES)
        baseline_dice = baseline_results['dice']

    modality_names = BraTSSegmentationDataset.MODALITY_ORDER   # ('flair', 't1', 't1ce', 't2')
    rows = []
    drops = []
    for ch_idx, modality in enumerate(modality_names):
        dice_scores = []
        for volume, mask in loader:
            volume = volume.clone().to(device)
            mask = mask.to(device)
            volume[:, ch_idx, ...] = 0.0    # zero masking ONLY — never dataset-mean substitution
            logits = model(volume)
            dice, _ = compute_dice_iou(logits, mask, num_classes=Config.NUM_SEG_CLASSES)
            dice_scores.append(dice)
        ablated_dice = float(np.nanmean(dice_scores)) if dice_scores else float('nan')
        drop = baseline_dice - ablated_dice
        drops.append(drop)
        rows.append({'modality': modality, 'baseline_dice': baseline_dice, 'ablated_dice': ablated_dice,
                      'dice_drop': drop})

    total_drop = sum(d for d in drops if d == d)   # drop NaNs
    for row, drop in zip(rows, drops):
        if total_drop != 0 and total_drop == total_drop and drop == drop:
            row['importance_pct'] = 100.0 * drop / total_drop
        else:
            row['importance_pct'] = 0.0   # avoid division by zero, per spec

    os.makedirs(output_dir, exist_ok=True)
    csv_path = os.path.join(output_dir, f'modality_ablation_seed_{seed}.csv')
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['modality', 'baseline_dice', 'ablated_dice', 'dice_drop', 'importance_pct'])
        writer.writeheader()
        writer.writerows(rows)

    logger.info(f"🧪 Seed {seed} modality ablation (baseline Dice={baseline_dice:.4f}):")
    for row in rows:
        logger.info(f"   {row['modality'].upper():5s} {row['importance_pct']:.0f}%  (drop={row['dice_drop']:.4f})")
    logger.info(f"💾 Modality ablation saved to {csv_path}")

    return {'csv_path': csv_path, 'baseline_dice': baseline_dice, 'rows': rows}


# ------------------------------------------------------------------------------
# 19d. NEW in v0.8 — Task 3: Linear Probe sanity check.
#      Uses the final FINE-TUNED v0.8 encoder (not the original SSL
#      checkpoint), frozen and kept in eval() throughout. Only a fresh
#      Linear(512->2) head is trained. Evaluation only.
# ------------------------------------------------------------------------------
def load_patient_grades() -> dict:
    """NEW in v0.8. Loads Config.NAME_MAPPING_CSV_PATH and returns a mapping
    from patient ID (e.g. 'BraTS20_Training_001') to its HGG/LGG grade.
    Raises a fatal error if the file is missing, consistent with existing
    project style (see run_segmentation_sanity_check above)."""
    path = Config.NAME_MAPPING_CSV_PATH
    if not os.path.isfile(path):
        raise SystemExit(
            f"❌ NAME_MAPPING_CSV_PATH not found: '{path}'. The Linear Probe "
            f"sanity check requires name_mapping.csv (HGG/LGG grade labels) to run."
        )

    grades = {}
    with open(path, 'r', newline='') as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames or []
        grade_col = next((c for c in fieldnames if c.strip().lower() == 'grade'), None)
        id_col = next((c for c in fieldnames if 'brats_2020' in c.strip().lower()), None)
        if id_col is None:
            id_col = next((c for c in fieldnames if 'subject_id' in c.strip().lower()), None)
        if grade_col is None or id_col is None:
            raise SystemExit(
                f"❌ name_mapping.csv at '{path}' is missing a recognizable "
                f"Grade / subject-ID column. Found columns: {fieldnames}"
            )
        for row in reader:
            patient_id = (row.get(id_col) or '').strip()
            grade = (row.get(grade_col) or '').strip()
            if patient_id:
                grades[patient_id] = grade

    logger.info(f"✅ Loaded {len(grades)} patient grade label(s) from {path}")
    return grades


def _extract_patient_id(path: str) -> Optional[str]:
    """NEW in v0.8. Extracts e.g. 'BraTS20_Training_001' from a modality file
    path, for joining against name_mapping.csv grade labels."""
    match = re.search(r'(BraTS20\d*_(?:Training|Validation)_\d+)', path)
    return match.group(1) if match else None


class LinearProbeDataset(Dataset):
    """NEW in v0.8. Wraps the existing (image, mask) pairs together with an
    HGG/LGG grade label looked up from name_mapping.csv. Reuses the exact
    same volume-loading transform already used by BraTSSegmentationDataset —
    no independent image pipeline is introduced."""
    def __init__(self, pairs: List[Tuple[str, str]], grades: dict,
                 img_size: Tuple[int, int, int] = Config.IMG_SIZE):
        self.img_transform = build_base_transform(img_size)
        self.samples: List[Tuple[str, int]] = []
        skipped = 0
        for img_path, _seg_path in pairs:
            patient_id = _extract_patient_id(img_path)
            grade = grades.get(patient_id) if patient_id else None
            if grade not in ('HGG', 'LGG'):
                skipped += 1
                continue
            self.samples.append((img_path, 1 if grade == 'HGG' else 0))
        if skipped:
            logger.warning(f"⚠️ Linear Probe: {skipped} patient(s) had no HGG/LGG grade match and were skipped.")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        img_path, label = self.samples[idx]
        modality_paths = {'flair': img_path}
        for modality in ('t1', 't1ce', 't2'):
            modality_paths[modality] = get_modality_path(img_path, modality)

        channel_volumes = []
        for modality in BraTSSegmentationDataset.MODALITY_ORDER:
            vol = self.img_transform(modality_paths[modality])
            if not torch.is_tensor(vol):
                vol = torch.as_tensor(np.asarray(vol))
            channel_volumes.append(vol.float())
        volume = torch.cat(channel_volumes, dim=0)
        volume = torch.nan_to_num(volume, nan=0.0, posinf=0.0, neginf=0.0)
        return volume, label


def run_linear_probe(encoder: ResNet3D, train_pairs: List[Tuple[str, str]],
                      val_pairs: List[Tuple[str, str]], device: torch.device,
                      seed: int, output_dir: str) -> dict:
    log_section(f"🔬 v0.8 — Linear Probe sanity check (seed {seed})")

    grades = load_patient_grades()

    train_probe_ds = LinearProbeDataset(train_pairs, grades, img_size=Config.IMG_SIZE)
    val_probe_ds = LinearProbeDataset(val_pairs, grades, img_size=Config.IMG_SIZE)

    if len(train_probe_ds) == 0 or len(val_probe_ds) == 0:
        logger.warning("⚠️ Linear Probe: not enough HGG/LGG-labeled patients found — skipping probe.")
        return {'skipped': True}

    train_probe_loader = DataLoader(train_probe_ds, batch_size=Config.LINEAR_PROBE_BATCH_SIZE,
                                     shuffle=True, num_workers=Config.NUM_WORKERS)
    val_probe_loader = DataLoader(val_probe_ds, batch_size=Config.LINEAR_PROBE_BATCH_SIZE,
                                   shuffle=False, num_workers=Config.NUM_WORKERS)

    # Use the final FINE-TUNED v0.8 encoder — frozen, eval() throughout.
    encoder.eval()
    encoder.requires_grad_(False)

    probe = nn.Linear(512, 2).to(device)
    optimizer = optim.Adam(probe.parameters(), lr=Config.LINEAR_PROBE_LR)

    labels_all = [label for _, label in train_probe_ds.samples]
    n_hgg = sum(labels_all)
    n_lgg = len(labels_all) - n_hgg
    if n_hgg > 0 and n_lgg > 0 and max(n_hgg, n_lgg) / min(n_hgg, n_lgg) >= 1.5:
        class_weight = torch.tensor(
            [len(labels_all) / (2.0 * n_lgg), len(labels_all) / (2.0 * n_hgg)], device=device,
        )
        criterion = nn.CrossEntropyLoss(weight=class_weight)
        logger.info(f"⚖️ Class imbalance detected (HGG={n_hgg}, LGG={n_lgg}) — using weighted CrossEntropy.")
    else:
        criterion = nn.CrossEntropyLoss()

    for epoch in range(1, Config.LINEAR_PROBE_EPOCHS + 1):
        probe.train()
        epoch_loss, n_batches = 0.0, 0
        for volume, label in train_probe_loader:
            volume = volume.to(device)
            label = label.to(device)
            with torch.no_grad():
                features = encoder(volume)   # (B, 512) — encoder frozen, no grad ever reaches it
            logits = probe(features)
            loss = criterion(logits, label)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1
        if n_batches:
            logger.info(f"   Linear Probe epoch {epoch}/{Config.LINEAR_PROBE_EPOCHS} — loss={epoch_loss / n_batches:.4f}")

    # ---- Evaluate ----
    probe.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for volume, label in val_probe_loader:
            volume = volume.to(device)
            features = encoder(volume)
            logits = probe(features)
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(label.tolist())

    correct = sum(p == l for p, l in zip(all_preds, all_labels))
    accuracy = correct / len(all_labels) if all_labels else float('nan')

    tp = sum((p == 1 and l == 1) for p, l in zip(all_preds, all_labels))
    fp = sum((p == 1 and l == 0) for p, l in zip(all_preds, all_labels))
    fn = sum((p == 0 and l == 1) for p, l in zip(all_preds, all_labels))
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    n_pos = sum(all_labels)
    majority_class = 1 if n_pos >= (len(all_labels) - n_pos) else 0
    majority_baseline = (sum(l == majority_class for l in all_labels) / len(all_labels)
                          if all_labels else float('nan'))

    if accuracy == accuracy and majority_baseline == majority_baseline and accuracy > majority_baseline:
        signal = "🟢 GREEN — Probe > Baseline"
    else:
        signal = "🔴 RED — Probe <= Baseline"

    logger.info(
        f"🔬 Linear Probe (seed {seed}) — Accuracy={accuracy:.4f} | F1={f1:.4f} | "
        f"Majority baseline={majority_baseline:.4f} | {signal}"
    )

    os.makedirs(output_dir, exist_ok=True)
    csv_path = os.path.join(output_dir, f'linear_probe_seed_{seed}.csv')
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['metric', 'value'])
        writer.writerow(['accuracy', accuracy])
        writer.writerow(['f1', f1])
        writer.writerow(['majority_baseline', majority_baseline])
        writer.writerow(['signal', signal])
    logger.info(f"💾 Linear Probe results saved to {csv_path}")

    encoder.requires_grad_(True)   # restore, in case this encoder object is reused elsewhere

    return {
        'csv_path': csv_path, 'accuracy': accuracy, 'f1': f1,
        'majority_baseline': majority_baseline, 'signal': signal,
    }


# ------------------------------------------------------------------------------
# 20. Mode B — per-epoch training loop.
#     UNCHANGED: optimizer step, scaler branch, gradient-finiteness check,
#     batch loop structure, per-batch bookkeeping, deep-supervision loss
#     combination logic (main + aux_weight * (aux1 + aux2)).
# ------------------------------------------------------------------------------
def train_one_epoch_segmentation(model, loader, optimizer, criterion, device, scaler=None,
                                  aux_weight: float = Config.AUX_LOSS_WEIGHT) -> float:
    model.train()
    total_loss = 0.0
    num_batches = 0

    def compute_loss(output):
        """output is either a single logits tensor (main only) or a
        (main, aux1, aux2) tuple (deep supervision). Returns the combined
        scalar loss in both cases."""
        if isinstance(output, tuple):
            main_logits, aux1_logits, aux2_logits = output
            main_loss = criterion(main_logits, mask)
            aux1_loss = criterion(aux1_logits, mask)
            aux2_loss = criterion(aux2_logits, mask)
            return main_loss + aux_weight * aux1_loss + aux_weight * aux2_loss
        return criterion(output, mask)

    pbar = tqdm(loader, desc='Segmentation training', leave=False)
    for volume, mask in pbar:
        volume = volume.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)
        optimizer.zero_grad()

        if scaler is not None:
            with torch.cuda.amp.autocast():
                output = model(volume)
                loss = compute_loss(output)
            if not torch.isfinite(loss):
                logger.warning("⚠️ Non-finite segmentation loss — skipping this batch's optimizer step.")
                optimizer.zero_grad(set_to_none=True)
                continue
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            output = model(volume)
            loss = compute_loss(output)
            if not torch.isfinite(loss):
                logger.warning("⚠️ Non-finite segmentation loss — skipping this batch's optimizer step.")
                optimizer.zero_grad(set_to_none=True)
                continue
            loss.backward()

            for name, param in model.named_parameters():
                if param.grad is not None and not torch.isfinite(param.grad).all():
                    logger.warning(f"🔴 Gradient is non-finite in layer: {name}")
                    break

            optimizer.step()

        total_loss += loss.item()
        num_batches += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    return total_loss / num_batches if num_batches else float('nan')


# ------------------------------------------------------------------------------
# 21. Mode B — v0.7-A NEW: fresh model builder + per-seed trainer +
#     multi-seed main runner.
# ------------------------------------------------------------------------------
def build_fresh_segmentation_model(device: torch.device) -> SegmentationModel3D:
    """
    Builds a BRAND NEW SegmentationModel3D: a fresh ResNet3D encoder (with
    the SSL-pretrained weights reloaded from disk, if available/enabled) and
    a freshly-initialized ResidualDecoder3D. Called once per seed so that
    each of the Config.SEEDS runs is a genuinely independent training run,
    not a continuation of the previous seed's weights.
    """
    # v0.7-B: 4-channel Early Fusion input (FLAIR, T1, T1CE, T2). This is the
    # ONLY encoder-instantiation call site changed for v0.7-B — the identical
    # call inside run_mode_a_simclr() (Mode A / SSL pretraining) is left at
    # in_channels=1, untouched, since Mode A stays single-modality.
    encoder = resnet18_3d(in_channels=4).to(device)

    if Config.USE_PRETRAINED_ENCODER and os.path.isfile(Config.PRETRAINED_ENCODER_PATH):
        state_dict = None
        try:
            state_dict = torch.load(Config.PRETRAINED_ENCODER_PATH, map_location=device)
        except Exception as e:
            logger.warning(f"⚠️ Could not read encoder checkpoint ({e}). Training encoder from scratch instead.")

        if state_dict is not None:
            try:
                encoder.load_state_dict(state_dict)
                logger.info(f"✅ Loaded pretrained SSL encoder from {Config.PRETRAINED_ENCODER_PATH}")
            except RuntimeError as e:
                logger.warning(f"⚠️ Strict encoder load failed ({e}); retrying with strict=False.")
                # v0.7-B fix: strict=False alone does NOT suppress shape-mismatch
                # errors — only missing/unexpected KEY errors. A param present in
                # both dicts with an incompatible shape (conv1.weight: 1-channel
                # checkpoint vs. 4-channel model) still raises even under
                # strict=False. We must drop the incompatible key(s) first so
                # they become genuinely "missing" instead of "mismatched".
                own_state = encoder.state_dict()
                compatible_state_dict = {
                    k: v for k, v in state_dict.items()
                    if k in own_state and v.shape == own_state[k].shape
                }
                incompatible_keys = set(state_dict.keys()) - set(compatible_state_dict.keys())
                if incompatible_keys:
                    logger.warning(
                        f"⚠️ Skipping incompatible-shape keys (kept at random init): {sorted(incompatible_keys)}"
                    )
                missing, unexpected = encoder.load_state_dict(compatible_state_dict, strict=False)
                logger.warning(f"⚠️ Loaded with mismatches — missing: {missing}, unexpected: {unexpected}")
    else:
        logger.warning(
            "⚠️ No pretrained encoder loaded (file missing or USE_PRETRAINED_ENCODER=False). "
            "Training encoder + segmentation head from scratch instead."
        )

    model = SegmentationModel3D(encoder=encoder, num_classes=Config.NUM_SEG_CLASSES,
                                 target_size=Config.IMG_SIZE).to(device)
    return model


def run_single_seed_segmentation(seed: int, train_loader: DataLoader, val_loader: DataLoader,
                                  device: torch.device) -> dict:
    """
    NEW in v0.7-A, EXTENDED in v0.8 (SE Attention logging + Modality Ablation +
    Linear Probe appended at the end — see Task 1/2/3). Trains ONE independent
    segmentation run for `seed`:
      - re-seeds everything via the existing set_seed()
      - builds a fresh model (fresh SSL encoder reload + fresh decoder)
      - trains for Config.NUM_EPOCHS_SEG epochs with CosineAnnealingLR
      - tracks a 3-epoch moving average of Val Dice and saves a new
        checkpoint (this epoch's weights) whenever that moving average beats
        the previous best — NOT a raw single-epoch Val Dice comparison
      - reloads the best checkpoint at the end and reports its final metrics
    Returns a dict with the seed's final Dice/IoU/WT/TC/ET.
    """
    log_section(f"🌱 v0.8 — Training seed {seed}")
    set_seed(seed)
    # Bug fix (v0.8): re-seed the train_loader's dedicated generator right
    # after set_seed(seed). Without this, the shuffle order is drawn from
    # the same global RNG that set_seed() seeds, so it would depend on how
    # much RNG state model construction consumes (e.g. SEChannelAttention3D's
    # extra nn.Linear layers) — decoupling this makes the shuffle order
    # depend only on `seed`, never on model architecture.
    if train_loader.generator is not None:
        train_loader.generator.manual_seed(seed)

    model = build_fresh_segmentation_model(device)
    optimizer = optim.Adam(model.parameters(), lr=Config.LR_SEG)
    criterion = DiceCELoss(num_classes=Config.NUM_SEG_CLASSES)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=Config.NUM_EPOCHS_SEG, eta_min=Config.SCHEDULER_ETA_MIN,
    )

    scaler = None   # kept disabled, matching v0.6's commented-out AMP branch

    # NEW in v0.8: checkpoints go under the dedicated v0.8 version root
    # (Config.V0_8_CHECKPOINT_DIR), never under Config.SEG_CHECKPOINT_DIR —
    # that directory is where v0.7-A/v0.7-B checkpoints already live, and
    # v0.8 must never overwrite them (Versioned Outputs, Mandatory).
    seed_ckpt_dir = os.path.join(Config.V0_8_CHECKPOINT_DIR, f'seed_{seed}')
    os.makedirs(seed_ckpt_dir, exist_ok=True)
    best_ckpt_path = os.path.join(seed_ckpt_dir, 'best_moving_avg_model.pth')

    val_dice_history: List[float] = []
    epoch_rows: List[dict] = []
    best_moving_avg = -1.0
    best_epoch: Optional[int] = None

    for epoch in range(1, Config.NUM_EPOCHS_SEG + 1):
        logger.info(f"\n{'-' * 40}\nSeed {seed} | Epoch {epoch}/{Config.NUM_EPOCHS_SEG}")
        _t_train_start = time.time()
        train_loss = train_one_epoch_segmentation(
            model, train_loader, optimizer, criterion, device, scaler,
            aux_weight=Config.AUX_LOSS_WEIGHT,
        )
        _t_train_end = time.time()
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        if train_loss != train_loss:   # NaN check
            logger.warning(f"⚠️ Seed {seed} epoch {epoch}: no valid batches — nothing was learned this epoch.")
        else:
            logger.info(f"📉 Train loss: {train_loss:.4f} | LR: {current_lr:.6f}")

        results = evaluate_segmentation_full(
            model, val_loader, device, num_classes=Config.NUM_SEG_CLASSES, criterion=criterion,
        )
        _t_val_end = time.time()
        logger.info(
            f"⏱️ train={_t_train_end - _t_train_start:.1f}s | "
            f"val={_t_val_end - _t_train_end:.1f}s | "
            f"total={_t_val_end - _t_train_start:.1f}s"
        )
        val_dice_history.append(results['dice'])

        logger.info(
            f"📊 Val loss: {results['val_loss']:.4f} | Val Dice: {results['dice']:.4f} | "
            f"Val IoU: {results['iou']:.4f} | WT: {results['wt']:.4f} | "
            f"TC: {results['tc']:.4f} | ET: {results['et']:.4f} | LR: {current_lr:.6f}"
        )

        # ---- v0.7-A checkpoint strategy: 3-epoch moving average of Val Dice ----
        if len(val_dice_history) >= Config.MOVING_AVG_WINDOW:
            window = val_dice_history[-Config.MOVING_AVG_WINDOW:]
            window_clean = [v for v in window if v == v]  # drop NaNs
            moving_avg = float(np.mean(window_clean)) if window_clean else float('nan')

            if moving_avg == moving_avg and moving_avg > best_moving_avg:
                best_moving_avg = moving_avg
                best_epoch = epoch
                _save_t0 = time.time()
                torch.save(model.state_dict(), best_ckpt_path)
                _save_t1 = time.time()
                logger.info(f"💾 Checkpoint save time = {_save_t1 - _save_t0:.2f}s")
                logger.info(
                    f"✅ New best checkpoint (seed {seed}, epoch {epoch}) — "
                    f"{Config.MOVING_AVG_WINDOW}-epoch moving avg Val Dice = {moving_avg:.4f} "
                    f"(window: {[round(v, 4) for v in window]})"
                )

        epoch_rows.append({
            'seed': seed, 'epoch': epoch, 'train_loss': train_loss,
            'val_loss': results['val_loss'], 'val_dice': results['dice'],
            'val_iou': results['iou'], 'val_wt': results['wt'],
            'val_tc': results['tc'], 'val_et': results['et'], 'lr': current_lr,
        })

    # NEW in v0.8: per-seed logs go under Config.V0_8_LOG_DIR, not
    # Config.SEG_LOG_DIR (that's where v0.7-A/v0.7-B logs already live).
    seed_history_path = os.path.join(Config.V0_8_LOG_DIR, f'segmentation_history_seed_{seed}.csv')
    with open(seed_history_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(epoch_rows[0].keys()))
        writer.writeheader()
        writer.writerows(epoch_rows)
    logger.info(f"💾 Seed {seed} epoch history saved to {seed_history_path}")

    # ---- Final metrics for this seed: reload the moving-avg-best checkpoint ----
    if best_epoch is not None:
        logger.info(f"📌 Seed {seed}: reloading best checkpoint from epoch {best_epoch} for final metrics.")
        model.load_state_dict(torch.load(best_ckpt_path, map_location=device))
    else:
        logger.warning(
            f"⚠️ Seed {seed}: moving-average criterion never triggered (fewer than "
            f"{Config.MOVING_AVG_WINDOW} valid epochs?). Using final-epoch weights instead."
        )

    final_results = evaluate_segmentation_full(
        model, val_loader, device, num_classes=Config.NUM_SEG_CLASSES, criterion=criterion,
        visualize=True,
        save_path=os.path.join(Config.V0_8_LOG_DIR, f'seg_visualization_seed_{seed}.png'),   # NEW in v0.8
    )
    logger.info(
        f"🏁 Seed {seed} FINAL (best checkpoint, epoch {best_epoch}) — "
        f"Dice: {final_results['dice']:.4f} | IoU: {final_results['iou']:.4f} | "
        f"WT: {final_results['wt']:.4f} | TC: {final_results['tc']:.4f} | ET: {final_results['et']:.4f}"
    )

    # ==========================================================================
    # NEW in v0.8 — Task 1/2/3: evaluation-only post-training analysis, run
    # AFTER evaluate_segmentation_full() above. None of these touch training,
    # optimization, gradients, or the segmentation results already computed.
    # ==========================================================================
    attention_results = run_channel_attention_logging(
        model, val_loader, device, seed, output_dir=Config.V0_8_ATTENTION_DIR,
    )
    ablation_results = run_modality_ablation(
        model, val_loader, device, seed, output_dir=Config.V0_8_CSV_DIR,
        baseline_dice=final_results['dice'],
    )
    linear_probe_results = run_linear_probe(
        model.get_encoder(),
        train_pairs=train_loader.dataset.pairs, val_pairs=val_loader.dataset.pairs,
        device=device, seed=seed, output_dir=Config.V0_8_LINEAR_PROBE_DIR,
    )

    result = {
        'seed': seed,
        'best_epoch': best_epoch if best_epoch is not None else Config.NUM_EPOCHS_SEG,
        'dice': final_results['dice'],
        'iou': final_results['iou'],
        'wt': final_results['wt'],
        'tc': final_results['tc'],
        'et': final_results['et'],
    }
    # NEW in v0.8: append Task 1/2/3 outputs without replacing existing keys.
    result['channel_attention'] = attention_results
    result['modality_ablation'] = ablation_results
    result['linear_probe'] = linear_probe_results
    return result


def run_mode_b_segmentation() -> None:
    """
    v0.8 main runner (protocol originally established in v0.7-A; extended in
    v0.8 with SE Channel Attention + Channel Attention logging + Modality
    Ablation + Linear Probe). Sanity check + split discovery use Config.SEED
    for reproducible file ordering; the persisted split JSON is what actually
    governs which patients land in train/val/test across all future
    versions. The Test set is loaded/counted (via create_or_load_split's
    logging) and then never referenced again in this function.
    """
    log_section("🧩 MedSSL-3D v0.8 — Mode B: Multi-seed statistical baseline + SE Attention (ResidualDecoder3D)")

    print_environment_report()
    Config.setup_dirs()
    device = torch.device(Config.DEVICE)

    logger.info(f"\n📂 DATA_ROOT : {Config.DATA_ROOT}")
    logger.info(f"📂 MODALITY  : {Config.MODALITY}")
    logger.info(f"📂 Seg checkpoints -> {Config.SEG_CHECKPOINT_DIR}")
    logger.info(f"📂 Seg logs        -> {Config.SEG_LOG_DIR}")
    # NEW in v0.8: dedicated version root — checkpoints/logs/csv for THIS run.
    logger.info(f"📂 v0.8 checkpoints -> {Config.V0_8_CHECKPOINT_DIR}")
    logger.info(f"📂 v0.8 logs        -> {Config.V0_8_LOG_DIR}")
    logger.info(f"📂 v0.8 USE_CHANNEL_SE_ATTENTION = {Config.USE_CHANNEL_SE_ATTENTION}")

    set_seed(Config.SEED)
    pairs = run_segmentation_sanity_check(Config.DATA_ROOT, Config.MODALITY, Config.IMG_SIZE)

    train_pairs, val_pairs, test_pairs = create_or_load_split(
        pairs, Config.SPLIT_FILE_PATH,
        Config.TRAIN_FRACTION, Config.VAL_FRACTION, Config.TEST_FRACTION,
        Config.SEED,
    )

    batch_size = min(Config.BATCH_SIZE_SEG, max(1, len(train_pairs)))
    if batch_size != Config.BATCH_SIZE_SEG:
        logger.warning(f"⚠️ Reducing segmentation BATCH_SIZE to {batch_size} (dataset is small).")

    train_dataset = BraTSSegmentationDataset(train_pairs, img_size=Config.IMG_SIZE,
                                              num_classes=Config.NUM_SEG_CLASSES)
    val_dataset = BraTSSegmentationDataset(val_pairs, img_size=Config.IMG_SIZE,
                                            num_classes=Config.NUM_SEG_CLASSES)
    # test_pairs is intentionally NEVER wrapped in a Dataset/DataLoader here —
    # it is reserved exclusively for v1.0, per the v0.7-A spec.

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=Config.NUM_WORKERS, pin_memory=(device.type == 'cuda'),
        generator=torch.Generator(),
    )
    val_loader = DataLoader(
        val_dataset, batch_size=1, shuffle=False,
        num_workers=Config.NUM_WORKERS, pin_memory=(device.type == 'cuda'),
    )

    logger.info(f"Encoder path: {Config.PRETRAINED_ENCODER_PATH}")
    logger.info(f"Exists? {os.path.isfile(Config.PRETRAINED_ENCODER_PATH)}")

    # ---- Multi-seed statistical training loop (protocol from v0.7-A, run here as v0.8) ----
    seed_results: List[dict] = []
    for seed in Config.SEEDS:
        result = run_single_seed_segmentation(seed, train_loader, val_loader, device)
        seed_results.append(result)

    # ---- Aggregate Mean +/- Std across seeds ----
    log_section("📈 v0.8 MULTI-SEED SUMMARY")
    metric_keys = ['dice', 'iou', 'wt', 'tc', 'et']
    summary_rows = []
    for r in seed_results:
        row = {'seed': r['seed'], 'best_epoch': r['best_epoch']}
        row.update({k: r[k] for k in metric_keys})
        summary_rows.append(row)
        logger.info(
            f"Seed {r['seed']:>5} | best_epoch={r['best_epoch']:>3} | "
            f"Dice={r['dice']:.4f} IoU={r['iou']:.4f} WT={r['wt']:.4f} "
            f"TC={r['tc']:.4f} ET={r['et']:.4f}"
        )

    mean_row = {'seed': 'mean', 'best_epoch': ''}
    std_row = {'seed': 'std', 'best_epoch': ''}
    for k in metric_keys:
        values = [r[k] for r in seed_results if r[k] == r[k]]   # drop NaNs
        mean_row[k] = float(np.mean(values)) if values else float('nan')
        std_row[k] = float(np.std(values)) if values else float('nan')
    summary_rows.append(mean_row)
    summary_rows.append(std_row)

    logger.info(
        f"\n🏆 MEAN ± STD across {len(seed_results)} seeds:\n" +
        "\n".join(f"   {k.upper():4s}: {mean_row[k]:.4f} ± {std_row[k]:.4f}" for k in metric_keys)
    )

    # NEW in v0.8: the multi-seed summary is written to the dedicated v0.8
    # CSV path, NOT Config.SEG_SUMMARY_CSV — that file is v0.7-A/v0.7-B's own
    # summary output and must never be overwritten by a later version.
    with open(Config.SEG_SUMMARY_CSV_V0_8, 'w', newline='') as f:
        fieldnames = ['seed', 'best_epoch'] + metric_keys
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(summary_rows)
    logger.info(f"💾 v0.8 multi-seed summary saved to {Config.SEG_SUMMARY_CSV_V0_8}")

    log_section("🎉 v0.8 MODE B RUN COMPLETE")
    logger.info(f"💾 v0.8 Checkpoints -> {Config.V0_8_CHECKPOINT_DIR} (per-seed subfolders)")
    logger.info(f"💾 v0.8 Logs        -> {Config.V0_8_LOG_DIR}")
    logger.info(f"💾 v0.8 Attention   -> {Config.V0_8_ATTENTION_DIR}")
    logger.info(f"💾 v0.8 Linear Probe-> {Config.V0_8_LINEAR_PROBE_DIR}")
    logger.info(f"🔒 Test set ({len(test_pairs)} patients) reserved untouched for v1.0.")
    logger.info("📌 SimCLR pipeline (Mode A) is untouched — set Config.MODE='simclr' to confirm.")


# ==============================================================================
# 23. NEW in v0.9 — MULTI-TASK LEARNING (Segmentation + HGG/LGG Classification)
#
# Everything below is ADDITIVE. Nothing in Sections 1-22 is rewritten:
#   - Config gained new keys only (no existing key removed).
#   - SegmentationModel3D.forward() gained ONE optional kwarg
#     (return_bottleneck, default False) — the default call path is
#     byte-for-byte identical to v0.8.
#   - Every function below is a SIBLING of its v0.8 counterpart
#     (train_one_epoch_multitask / run_single_seed_multitask /
#      build_fresh_multitask_model / MultiTaskSegmentationDataset), so the
#     validated v0.8 path (Config.MODE = 'segmentation') still runs exactly
#     as it did before.
#
# Architecture:
#   Input (4 modalities) -> Shared Encoder -> SE Attention -> bottleneck l4
#                                                 /             \
#                                           Seg Head        Cls Head (NEW)
#                                           (existing)      AdaptiveAvgPool3d(1)
#                                                           -> Linear(512,128)
#                                                           -> ReLU -> Dropout
#                                                           -> Linear(128,2)
# ==============================================================================

# ------------------------------------------------------------------------------
# 23a. NEW in v0.9 — Task B: Classification head on the shared bottleneck.
# ------------------------------------------------------------------------------
class ClassificationHead3D(nn.Module):                                   # NEW in v0.9
    """Reads the encoder bottleneck l4 (B, 512, d, h, w) and predicts HGG/LGG.

    Deliberately tiny (512 -> 128 -> 2, ~66k params) so the segmentation task
    stays the dominant gradient signal and the extra VRAM cost on a 6GB laptop
    GPU is negligible.
    """
    def __init__(self, in_channels: int = 512,
                 hidden_dim: int = Config.CLS_HEAD_HIDDEN_DIM,
                 num_classes: int = Config.NUM_CLS_CLASSES,
                 dropout: float = Config.CLS_HEAD_DROPOUT):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc1 = nn.Linear(in_channels, hidden_dim)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, l4: torch.Tensor) -> torch.Tensor:
        b, c = l4.shape[0], l4.shape[1]
        x = self.pool(l4).view(b, c)          # (B, 512)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)                    # (B, 2) logits


# ------------------------------------------------------------------------------
# 23b. NEW in v0.9 — Task C: Multi-task wrapper.
#      Owns an UNMODIFIED SegmentationModel3D plus the new head. The wrapper
#      never re-implements any segmentation logic — it only asks the existing
#      model for its bottleneck via return_bottleneck=True.
# ------------------------------------------------------------------------------
class MultiTaskModel3D(nn.Module):                                       # NEW in v0.9
    def __init__(self, seg_model: SegmentationModel3D,
                 cls_num_classes: int = Config.NUM_CLS_CLASSES):
        super().__init__()
        self.seg_model = seg_model
        bottleneck_channels = seg_model.get_encoder().stage_channels[-1]   # 512
        self.cls_head = ClassificationHead3D(
            in_channels=bottleneck_channels,
            hidden_dim=Config.CLS_HEAD_HIDDEN_DIM,
            num_classes=cls_num_classes,
            dropout=Config.CLS_HEAD_DROPOUT,
        )

    def forward(self, x: torch.Tensor):
        # seg_output is (main, aux1, aux2) in train mode and main-only in eval
        # mode — exactly as v0.8's SegmentationModel3D already behaves.
        seg_output, l4 = self.seg_model(x, return_bottleneck=True)
        cls_logits = self.cls_head(l4)
        return seg_output, cls_logits

    def get_encoder(self) -> ResNet3D:
        return self.seg_model.get_encoder()

    def get_seg_model(self) -> SegmentationModel3D:
        return self.seg_model


# ------------------------------------------------------------------------------
# 23c. NEW in v0.9 — Task D: fresh multi-task model builder.
#      Calls the UNCHANGED build_fresh_segmentation_model(device) internally,
#      so the from-scratch-per-seed protocol (fresh SSL encoder reload + fresh
#      decoder init, never a warm start from v0.8's fine-tuned checkpoint) is
#      literally the same code path as every prior version.
# ------------------------------------------------------------------------------
def build_fresh_multitask_model(device: torch.device) -> nn.Module:      # NEW in v0.9
    seg_model = build_fresh_segmentation_model(device)

    if not Config.USE_CLASSIFICATION_HEAD:
        # Ablation branch: no wrapper at all -> identical to v0.8.
        logger.info("ℹ️ USE_CLASSIFICATION_HEAD=False — using plain SegmentationModel3D (v0.8 behavior).")
        return seg_model

    model = MultiTaskModel3D(seg_model, cls_num_classes=Config.NUM_CLS_CLASSES).to(device)
    n_cls_params = sum(p.numel() for p in model.cls_head.parameters())
    logger.info(f"🧠 Multi-task model built — classification head params: {n_cls_params:,}")
    return model


# ------------------------------------------------------------------------------
# 23d. NEW in v0.9 — Task G: synchronized / intensity augmentations.
#
#      All three run on CPU inside __getitem__ (Config.NUM_WORKERS = 0, so
#      they are synchronous with training — see the runtime check in
#      run_single_seed_multitask, which reports the measured overhead and
#      tells you to flip USE_ELASTIC_DEFORM off first if it is too slow).
#
#      Elastic deformation is SPATIAL: the image and the mask are warped with
#      the IDENTICAL random displacement field, in the same call, so labels
#      can never silently desynchronize from the voxels.
#      Gamma + bias field are INTENSITY-ONLY and touch image channels only.
#
#      Randomness is drawn from the global torch RNG, which set_seed(seed)
#      already seeds at the top of each run — so augmentation is reproducible
#      per seed. (This is safe precisely because the v0.8 shuffle-order fix
#      moved batch ordering onto the DataLoader's OWN generator, so it is not
#      perturbed by how much global RNG augmentation consumes.)
# ------------------------------------------------------------------------------
def _smooth_random_field(num_channels: int,
                         spatial_size: Tuple[int, int, int],
                         control_grid: Tuple[int, int, int],
                         amplitude: float) -> torch.Tensor:              # NEW in v0.9
    """Low-resolution uniform noise in [-amplitude, +amplitude], trilinearly
    upsampled to `spatial_size`. Cheap way to get a smooth random field —
    the coarse control grid is what makes the deformation plausible rather
    than per-voxel noise."""
    coarse = (torch.rand(1, num_channels, *control_grid) * 2.0 - 1.0) * amplitude
    return F.interpolate(coarse, size=spatial_size, mode='trilinear', align_corners=True)


def apply_elastic_deformation(volume: torch.Tensor,
                              mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:   # NEW in v0.9
    """Warps image AND mask with ONE shared displacement field.

    volume: (C, X, Y, Z) float | mask: (X, Y, Z) long
    Image is sampled bilinearly (smooth intensities); the mask is sampled with
    nearest-neighbour so class indices stay integral and no new labels are
    invented at boundaries.
    """
    spatial = tuple(volume.shape[1:])
    disp = _smooth_random_field(3, spatial, Config.ELASTIC_CONTROL_GRID,
                                Config.ELASTIC_MAX_DISPLACEMENT)          # (1, 3, X, Y, Z)

    theta = torch.eye(3, 4).unsqueeze(0)                                   # identity affine
    base_grid = F.affine_grid(theta, size=(1, 1, *spatial), align_corners=True)   # (1, X, Y, Z, 3)
    grid = base_grid + disp.permute(0, 2, 3, 4, 1)

    warped_volume = F.grid_sample(volume.unsqueeze(0).float(), grid, mode='bilinear',
                                  padding_mode='border', align_corners=True).squeeze(0)

    mask_f = mask.unsqueeze(0).unsqueeze(0).float()                        # (1, 1, X, Y, Z)
    warped_mask = F.grid_sample(mask_f, grid, mode='nearest',
                                padding_mode='zeros', align_corners=True)
    warped_mask = warped_mask.squeeze(0).squeeze(0).round().long()

    return warped_volume, warped_mask


def apply_gamma_adjust(volume: torch.Tensor) -> torch.Tensor:            # NEW in v0.9
    """Intensity-only, per modality channel. Min-max scales the channel to
    [0, 1], applies x**gamma, then maps back to the original intensity range
    so downstream statistics stay comparable."""
    out = volume.clone()
    lo, hi = Config.GAMMA_RANGE
    for c in range(out.shape[0]):
        channel = out[c]
        vmin = channel.min()
        vmax = channel.max()
        span = (vmax - vmin)
        if float(span) < 1e-6:
            continue
        gamma = float(torch.empty(1).uniform_(lo, hi).item())
        normalized = ((channel - vmin) / span).clamp_(0.0, 1.0)
        out[c] = normalized.pow(gamma) * span + vmin
    return out


def apply_bias_field(volume: torch.Tensor) -> torch.Tensor:              # NEW in v0.9
    """Intensity-only multiplicative low-frequency bias field, simulating MRI
    coil inhomogeneity. ONE field is shared across all four modality channels
    (a real bias field is a scanner/coil property, not a per-sequence one)."""
    spatial = tuple(volume.shape[1:])
    log_field = _smooth_random_field(1, spatial, Config.BIAS_FIELD_CONTROL_GRID,
                                     Config.BIAS_FIELD_MAX_LOG)            # (1, 1, X, Y, Z)
    field = torch.exp(log_field).squeeze(0)                                # (1, X, Y, Z), broadcasts over C
    return volume * field


def apply_multitask_augmentations(volume: torch.Tensor,
                                  mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:   # NEW in v0.9
    """Applies the v0.9 augmentations, each gated by its own Config flag AND
    its own probability. Order matters: the spatial warp runs first (on raw
    intensities), then intensity-only transforms."""
    if Config.USE_ELASTIC_DEFORM and float(torch.rand(1).item()) < Config.ELASTIC_PROB:
        volume, mask = apply_elastic_deformation(volume, mask)

    if Config.USE_GAMMA_ADJUST and float(torch.rand(1).item()) < Config.GAMMA_PROB:
        volume = apply_gamma_adjust(volume)

    if Config.USE_BIAS_FIELD and float(torch.rand(1).item()) < Config.BIAS_FIELD_PROB:
        volume = apply_bias_field(volume)

    return volume, mask


# ------------------------------------------------------------------------------
# 23e. NEW in v0.9 — Task E: multi-task dataset.
#      SUBCLASS of BraTSSegmentationDataset — the parent's __getitem__ (four
#      modality loads, label remap, NaN sanitation, shape check) is reused
#      verbatim via super(); the parent class itself is not modified.
# ------------------------------------------------------------------------------
class MultiTaskSegmentationDataset(BraTSSegmentationDataset):            # NEW in v0.9
    """Returns (volume, mask, grade_label).

    grade_label: 1 = HGG, 0 = LGG, -1 = unknown/unmatched.
    Segmentation proceeds for EVERY patient regardless of grade; -1 samples
    are simply ignored by nn.CrossEntropyLoss(ignore_index=-1), so an
    incomplete name_mapping.csv can never shrink the segmentation training
    set (which is the task that actually has to stay comparable to v0.8).
    """
    UNKNOWN_LABEL = -1

    def __init__(self, pairs: List[Tuple[str, str]],
                 img_size: Tuple[int, int, int] = Config.IMG_SIZE,
                 num_classes: int = 4,
                 grades: Optional[dict] = None,
                 augment: bool = False,
                 split_name: str = ''):
        super().__init__(pairs, img_size=img_size, num_classes=num_classes)

        # Reuses v0.8's load_patient_grades() / _extract_patient_id() exactly —
        # no second grade-parsing implementation is introduced.
        self.grades = grades if grades is not None else load_patient_grades()
        self.augment = augment
        self.split_name = split_name

        self.grade_labels: List[int] = []
        for img_path, _seg_path in pairs:
            patient_id = _extract_patient_id(img_path)
            grade = self.grades.get(patient_id) if patient_id else None
            if grade == 'HGG':
                self.grade_labels.append(1)
            elif grade == 'LGG':
                self.grade_labels.append(0)
            else:
                self.grade_labels.append(self.UNKNOWN_LABEL)

        n_hgg = sum(1 for l in self.grade_labels if l == 1)
        n_lgg = sum(1 for l in self.grade_labels if l == 0)
        n_unk = sum(1 for l in self.grade_labels if l == self.UNKNOWN_LABEL)
        logger.info(
            f"🏷️ MultiTask dataset [{split_name or 'unnamed'}] — {len(pairs)} patients | "
            f"HGG={n_hgg} LGG={n_lgg} unlabeled={n_unk} (unlabeled still train segmentation)"
        )
        if n_unk:
            logger.warning(
                f"⚠️ {n_unk} patient(s) have no HGG/LGG match — their classification loss is "
                f"masked out via ignore_index=-1, their segmentation loss is unaffected."
            )

        # Augmentation-cost accounting (Task G runtime check). Reset per epoch
        # by run_single_seed_multitask.
        self.aug_time_total = 0.0
        self.aug_calls = 0

    def __getitem__(self, idx: int):
        volume, mask = super().__getitem__(idx)          # UNCHANGED parent logic
        label = self.grade_labels[idx]

        if self.augment:
            _t0 = time.time()
            volume, mask = apply_multitask_augmentations(volume, mask)
            mask = torch.clamp(mask, min=0, max=self.num_classes - 1)
            self.aug_time_total += time.time() - _t0
            self.aug_calls += 1

        return volume, mask, label

    # ---- helpers used by the training loop / sampler ----
    def reset_aug_timer(self) -> None:
        self.aug_time_total = 0.0
        self.aug_calls = 0

    def class_counts(self) -> Tuple[int, int]:
        n_hgg = sum(1 for l in self.grade_labels if l == 1)
        n_lgg = sum(1 for l in self.grade_labels if l == 0)
        return n_hgg, n_lgg

    def build_cls_class_weights(self, device: torch.device) -> Optional[torch.Tensor]:
        """Same class-weighting rule already used by the v0.8 Linear Probe:
        inverse-frequency weights, applied only when the imbalance ratio is
        >= 1.5 (below that, weighting adds variance for no real benefit)."""
        n_hgg, n_lgg = self.class_counts()
        n_labeled = n_hgg + n_lgg
        if n_hgg > 0 and n_lgg > 0 and max(n_hgg, n_lgg) / min(n_hgg, n_lgg) >= 1.5:
            weights = torch.tensor(
                [n_labeled / (2.0 * n_lgg), n_labeled / (2.0 * n_hgg)],
                device=device, dtype=torch.float32,
            )
            logger.info(
                f"⚖️ Cls class imbalance (HGG={n_hgg}, LGG={n_lgg}) — weighted CrossEntropy "
                f"weights=[LGG {weights[0]:.3f}, HGG {weights[1]:.3f}]"
            )
            return weights
        logger.info(f"⚖️ Cls classes near-balanced (HGG={n_hgg}, LGG={n_lgg}) — unweighted CrossEntropy.")
        return None

    def build_sampler_weights(self) -> List[float]:
        """Per-sample weights for WeightedRandomSampler (Task G).
        Labeled samples get inverse-class-frequency weight. Unlabeled samples
        get the MEAN of the two class weights, so they are neither
        over-sampled nor pushed out of segmentation training."""
        n_hgg, n_lgg = self.class_counts()
        w_hgg = 1.0 / n_hgg if n_hgg else 0.0
        w_lgg = 1.0 / n_lgg if n_lgg else 0.0
        w_unknown = (w_hgg + w_lgg) / 2.0 if (n_hgg or n_lgg) else 1.0
        return [w_hgg if l == 1 else (w_lgg if l == 0 else w_unknown) for l in self.grade_labels]


# ------------------------------------------------------------------------------
# 23f. NEW in v0.9 — multi-task evaluation.
#      Sibling of evaluate_segmentation_full(); the segmentation metrics are
#      computed with the SAME unchanged compute_dice_iou / compute_region_dice
#      helpers, so Dice numbers stay directly comparable to v0.8.
# ------------------------------------------------------------------------------
@torch.no_grad()
def evaluate_multitask_full(model, loader, device, num_classes, criterion=None,
                            cls_criterion=None, visualize=False, save_path=None) -> dict:   # NEW in v0.9
    model.eval()
    dice_scores, iou_scores = [], []
    wt_scores, tc_scores, et_scores = [], [], []
    loss_values, cls_loss_values = [], []
    all_preds, all_labels = [], []
    all_scores: List[float] = []                                         # NEW in v0.9-metrics (Task F)
    first_batch_for_viz = None

    for volume, mask, grade in loader:
        volume = volume.to(device)
        mask = mask.to(device)
        grade = grade.to(device).long()

        output = model(volume)
        if isinstance(output, tuple) and len(output) == 2 and output[1] is not None \
                and output[1].dim() == 2:
            logits, cls_logits = output           # multi-task model
        else:
            logits, cls_logits = output, None     # USE_CLASSIFICATION_HEAD=False

        dice, iou = compute_dice_iou(logits, mask, num_classes=num_classes)
        wt, tc, et = compute_region_dice(logits, mask)
        dice_scores.append(dice)
        iou_scores.append(iou)
        wt_scores.append(wt)
        tc_scores.append(tc)
        et_scores.append(et)

        if criterion is not None:
            loss_values.append(criterion(logits, mask).item())

        if cls_logits is not None:
            labeled = grade != MultiTaskSegmentationDataset.UNKNOWN_LABEL
            if labeled.any():
                preds = torch.argmax(cls_logits[labeled], dim=1)
                all_preds.extend(preds.cpu().tolist())
                all_labels.extend(grade[labeled].cpu().tolist())
                # NEW in v0.9-metrics (Task F): P(HGG) for ROC AUC. Collected only;
                # `preds` above is unchanged, so every existing metric is identical.
                all_scores.extend(torch.softmax(cls_logits[labeled].float(), dim=1)[:, 1].cpu().tolist())
                if cls_criterion is not None:
                    cls_loss_values.append(cls_criterion(cls_logits, grade).item())

        if visualize and first_batch_for_viz is None:
            pred = torch.argmax(logits, dim=1)
            first_batch_for_viz = (volume[0].cpu(), mask[0].cpu(), pred[0].cpu())

    # ---- classification metrics (same definitions as v0.8's Linear Probe) ----
    if all_labels:
        correct = sum(p == l for p, l in zip(all_preds, all_labels))
        accuracy = correct / len(all_labels)
        tp = sum((p == 1 and l == 1) for p, l in zip(all_preds, all_labels))
        fp = sum((p == 1 and l == 0) for p, l in zip(all_preds, all_labels))
        fn = sum((p == 0 and l == 1) for p, l in zip(all_preds, all_labels))
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        n_pos = sum(all_labels)
        majority_class = 1 if n_pos >= (len(all_labels) - n_pos) else 0
        majority_baseline = sum(l == majority_class for l in all_labels) / len(all_labels)
    else:
        accuracy = f1 = majority_baseline = float('nan')

    results = {
        'dice': float(np.nanmean(dice_scores)) if dice_scores else float('nan'),
        'iou': float(np.nanmean(iou_scores)) if iou_scores else float('nan'),
        'wt': float(np.nanmean(wt_scores)) if wt_scores else float('nan'),
        'tc': float(np.nanmean(tc_scores)) if tc_scores else float('nan'),
        'et': float(np.nanmean(et_scores)) if et_scores else float('nan'),
        'val_loss': float(np.mean(loss_values)) if loss_values else float('nan'),
        'val_cls_loss': float(np.mean(cls_loss_values)) if cls_loss_values else float('nan'),
        'cls_accuracy': accuracy,
        'cls_f1': f1,
        'cls_majority_baseline': majority_baseline,
        'cls_n_eval': len(all_labels),
        # NEW in v0.9-metrics (Task F): balanced accuracy / MCC / AUC / Fisher /
        # permutation p / Wilson CI, APPENDED after every existing key. Existing
        # keys keep their exact names and values, so no downstream code breaks.
        **_metrics_epoch_fields(all_labels, all_preds, all_scores),       # NEW in v0.9-metrics
    }

    if visualize and first_batch_for_viz is not None:
        vol, gt, pred = first_batch_for_viz
        visualize_segmentation(vol, gt, pred, save_path=save_path)

    return results


# ------------------------------------------------------------------------------
# 23g. NEW in v0.9 — Task F: multi-task per-epoch training loop.
#      Sibling of train_one_epoch_segmentation() — that function is untouched.
#      Keeps its exact structure (optimizer step, scaler branch, non-finite
#      guard, deep-supervision aux combination) and adds:
#        L_total = LAMBDA_SEG * L_seg + LAMBDA_CLS * L_cls
#        gradient clipping at Config.GRAD_CLIP_NORM
# ------------------------------------------------------------------------------
def train_one_epoch_multitask(model, loader, optimizer, criterion, cls_criterion, device,
                              scaler=None, aux_weight: float = Config.AUX_LOSS_WEIGHT) -> dict:   # NEW in v0.9
    model.train()
    total_seg, total_cls, total_all = 0.0, 0.0, 0.0
    num_batches = 0
    num_cls_batches = 0

    def compute_seg_loss(output, mask):
        """Identical combination rule to train_one_epoch_segmentation()."""
        if isinstance(output, tuple):
            main_logits, aux1_logits, aux2_logits = output
            return (criterion(main_logits, mask)
                    + aux_weight * criterion(aux1_logits, mask)
                    + aux_weight * criterion(aux2_logits, mask))
        return criterion(output, mask)

    def compute_cls_loss(cls_logits, grade):
        """0.0 (not NaN) when the batch contains no labeled sample — CE with
        reduction='mean' over an all-ignored batch would otherwise be 0/0."""
        if cls_logits is None or cls_criterion is None:
            return torch.zeros((), device=device), False
        labeled = grade != MultiTaskSegmentationDataset.UNKNOWN_LABEL
        if not bool(labeled.any()):
            return torch.zeros((), device=device), False
        return cls_criterion(cls_logits.float(), grade), True

    pbar = tqdm(loader, desc='Multi-task training', leave=False)
    for volume, mask, grade in pbar:
        volume = volume.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)
        grade = grade.to(device, non_blocking=True).long()
        optimizer.zero_grad()

        def _forward():
            output = model(volume)
            if isinstance(output, tuple) and len(output) == 2:
                seg_output, cls_logits = output          # MultiTaskModel3D
            else:
                seg_output, cls_logits = output, None    # plain SegmentationModel3D
            l_seg = compute_seg_loss(seg_output, mask)
            l_cls, had_labels = compute_cls_loss(cls_logits, grade)
            l_total = Config.LAMBDA_SEG * l_seg + Config.LAMBDA_CLS * l_cls
            return l_seg, l_cls, l_total, had_labels

        if scaler is not None:
            with torch.cuda.amp.autocast():
                l_seg, l_cls, loss, had_labels = _forward()
            if not torch.isfinite(loss):
                logger.warning("⚠️ Non-finite multi-task loss — skipping this batch's optimizer step.")
                optimizer.zero_grad(set_to_none=True)
                continue
            scaler.scale(loss).backward()
            if Config.GRAD_CLIP_NORM and Config.GRAD_CLIP_NORM > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            l_seg, l_cls, loss, had_labels = _forward()
            if not torch.isfinite(loss):
                logger.warning("⚠️ Non-finite multi-task loss — skipping this batch's optimizer step.")
                optimizer.zero_grad(set_to_none=True)
                continue
            loss.backward()

            for name, param in model.named_parameters():
                if param.grad is not None and not torch.isfinite(param.grad).all():
                    logger.warning(f"🔴 Gradient is non-finite in layer: {name}")
                    break

            if Config.GRAD_CLIP_NORM and Config.GRAD_CLIP_NORM > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRAD_CLIP_NORM)

            optimizer.step()

        total_seg += float(l_seg.item())
        total_all += float(loss.item())
        if had_labels:
            total_cls += float(l_cls.item())
            num_cls_batches += 1
        num_batches += 1
        pbar.set_postfix({'total': f'{loss.item():.4f}', 'seg': f'{l_seg.item():.4f}',
                          'cls': f'{l_cls.item():.4f}'})

    if not num_batches:
        return {'train_loss': float('nan'), 'l_seg': float('nan'),
                'l_cls': float('nan'), 'l_total': float('nan')}

    return {
        'train_loss': total_all / num_batches,      # == l_total; kept for CSV continuity
        'l_seg': total_seg / num_batches,
        'l_cls': (total_cls / num_cls_batches) if num_cls_batches else 0.0,
        'l_total': total_all / num_batches,
    }


# ------------------------------------------------------------------------------
# 23h. NEW in v0.9 — Task F: per-seed multi-task trainer.
#      Sibling of run_single_seed_segmentation() (untouched). Same structure:
#        set_seed -> re-seed loader/sampler generators -> fresh model ->
#        train -> moving-average Val Dice checkpoint selection (Dice ONLY;
#        classification accuracy is deliberately NOT folded in, so the
#        checkpoint criterion stays identical to v0.7-B/v0.8 and Dice numbers
#        remain comparable) -> reload best -> report.
#      Adds: warmup, gradient clipping, early stopping, augmentation timing.
# ------------------------------------------------------------------------------
def run_single_seed_multitask(seed: int, train_loader: DataLoader, val_loader: DataLoader,
                              device: torch.device) -> dict:             # NEW in v0.9
    log_section(f"🌱 v0.9 — Multi-task training seed {seed}")
    set_seed(seed)

    # v0.8 RNG fix, preserved verbatim, and extended to the NEW sampler's own
    # generator (Task G): batch composition must depend only on `seed`, never
    # on how much global RNG state model construction happens to consume.
    if train_loader.generator is not None:
        train_loader.generator.manual_seed(seed)
    sampler = getattr(train_loader, 'sampler', None)
    sampler_gen = getattr(sampler, 'generator', None)
    # `is not train_loader.generator` matters: with plain shuffle=True the
    # DataLoader hands its OWN generator to an internal RandomSampler, so this
    # would otherwise re-seed the same object twice and log a misleading line.
    if sampler_gen is not None and sampler_gen is not train_loader.generator:
        sampler_gen.manual_seed(seed)
        logger.info(f"🔒 {type(sampler).__name__} generator re-seeded to {seed}")

    model = build_fresh_multitask_model(device)
    optimizer = optim.Adam(model.parameters(), lr=Config.LR_SEG)
    criterion = DiceCELoss(num_classes=Config.NUM_SEG_CLASSES)   # unchanged — NOT Focal

    # Classification criterion: ignore_index=-1 masks unlabeled patients;
    # class weights reuse the v0.8 Linear Probe rule.
    train_ds = train_loader.dataset
    cls_weights = train_ds.build_cls_class_weights(device) if hasattr(train_ds, 'build_cls_class_weights') else None
    cls_criterion = nn.CrossEntropyLoss(
        weight=cls_weights, ignore_index=MultiTaskSegmentationDataset.UNKNOWN_LABEL,
    ) if Config.USE_CLASSIFICATION_HEAD else None

    # Warmup (Task G): the cosine cycle covers the POST-warmup epochs only, so
    # the total stays Config.NUM_EPOCHS_SEG — no extra epochs are added.
    warmup_epochs = max(0, int(Config.WARMUP_EPOCHS))
    cosine_epochs = max(1, Config.NUM_EPOCHS_SEG - warmup_epochs)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cosine_epochs, eta_min=Config.SCHEDULER_ETA_MIN,
    )

    scaler = None   # kept disabled, matching v0.6/v0.7/v0.8

    seed_ckpt_dir = os.path.join(Config.V0_9_CHECKPOINT_DIR, f'seed_{seed}')
    os.makedirs(seed_ckpt_dir, exist_ok=True)
    best_ckpt_path = os.path.join(seed_ckpt_dir, 'best_moving_avg_model.pth')

    val_dice_history: List[float] = []
    epoch_rows: List[dict] = []
    best_moving_avg = -1.0
    best_epoch: Optional[int] = None
    epochs_since_best = 0
    stopped_early_at: Optional[int] = None

    for epoch in range(1, Config.NUM_EPOCHS_SEG + 1):
        logger.info(f"\n{'-' * 40}\nSeed {seed} | Epoch {epoch}/{Config.NUM_EPOCHS_SEG}")

        # ---- Linear LR warmup for the first WARMUP_EPOCHS epochs ----
        if warmup_epochs > 0 and epoch <= warmup_epochs:
            warm_lr = Config.LR_SEG * epoch / warmup_epochs
            for group in optimizer.param_groups:
                group['lr'] = warm_lr
            logger.info(f"🔥 Warmup epoch {epoch}/{warmup_epochs} — LR set to {warm_lr:.6f}")

        if hasattr(train_ds, 'reset_aug_timer'):
            train_ds.reset_aug_timer()

        _t_train_start = time.time()
        losses = train_one_epoch_multitask(
            model, train_loader, optimizer, criterion, cls_criterion, device, scaler,
            aux_weight=Config.AUX_LOSS_WEIGHT,
        )
        _t_train_end = time.time()

        # Cosine only starts once warmup is done (warmup already ends exactly
        # at base LR, so the handover is continuous).
        if epoch > warmup_epochs:
            scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        train_loss = losses['train_loss']
        if train_loss != train_loss:   # NaN check
            logger.warning(f"⚠️ Seed {seed} epoch {epoch}: no valid batches — nothing was learned this epoch.")
        else:
            logger.info(
                f"📉 L_total: {losses['l_total']:.4f} | L_seg: {losses['l_seg']:.4f} | "
                f"L_cls: {losses['l_cls']:.4f} | LR: {current_lr:.6f}"
            )

        results = evaluate_multitask_full(
            model, val_loader, device, num_classes=Config.NUM_SEG_CLASSES,
            criterion=criterion, cls_criterion=cls_criterion,
        )
        _t_val_end = time.time()

        train_secs = _t_train_end - _t_train_start
        logger.info(
            f"⏱️ train={train_secs:.1f}s | val={_t_val_end - _t_train_end:.1f}s | "
            f"total={_t_val_end - _t_train_start:.1f}s"
        )

        # ---- Task G runtime check: how much of the epoch was CPU augmentation? ----
        if epoch <= Config.AUG_TIMING_EPOCHS and getattr(train_ds, 'aug_calls', 0):
            aug_secs = train_ds.aug_time_total
            frac = aug_secs / train_secs if train_secs > 0 else 0.0
            logger.info(
                f"🧪 Augmentation cost (NUM_WORKERS={Config.NUM_WORKERS}, runs synchronously): "
                f"{aug_secs:.1f}s over {train_ds.aug_calls} samples = {frac * 100:.1f}% of the train epoch."
            )
            if frac > Config.AUG_OVERHEAD_WARN_FRACTION:
                logger.warning(
                    f"⚠️ Augmentation is eating {frac * 100:.1f}% of the epoch (> "
                    f"{Config.AUG_OVERHEAD_WARN_FRACTION * 100:.0f}%). Set "
                    f"Config.USE_ELASTIC_DEFORM = False first (it is by far the most "
                    f"expensive), and keep USE_GAMMA_ADJUST / USE_BIAS_FIELD (both cheap)."
                )

        val_dice_history.append(results['dice'])

        logger.info(
            f"📊 Val loss: {results['val_loss']:.4f} | Val Dice: {results['dice']:.4f} | "
            f"Val IoU: {results['iou']:.4f} | WT: {results['wt']:.4f} | "
            f"TC: {results['tc']:.4f} | ET: {results['et']:.4f} | LR: {current_lr:.6f}"
        )
        logger.info(
            f"🧬 Val Cls — Accuracy: {results['cls_accuracy']:.4f} | F1: {results['cls_f1']:.4f} | "
            f"Baseline: {results['cls_majority_baseline']:.4f} | n={results['cls_n_eval']}"
        )

        # ---- Checkpoint strategy: UNCHANGED 3-epoch moving average of Val Dice ----
        if len(val_dice_history) >= Config.MOVING_AVG_WINDOW:
            window = val_dice_history[-Config.MOVING_AVG_WINDOW:]
            window_clean = [v for v in window if v == v]
            moving_avg = float(np.mean(window_clean)) if window_clean else float('nan')

            if moving_avg == moving_avg and moving_avg > best_moving_avg:
                best_moving_avg = moving_avg
                best_epoch = epoch
                epochs_since_best = 0
                _save_t0 = time.time()
                torch.save(model.state_dict(), best_ckpt_path)
                _save_t1 = time.time()
                logger.info(f"💾 Checkpoint save time = {_save_t1 - _save_t0:.2f}s")
                logger.info(
                    f"✅ New best checkpoint (seed {seed}, epoch {epoch}) — "
                    f"{Config.MOVING_AVG_WINDOW}-epoch moving avg Val Dice = {moving_avg:.4f} "
                    f"(window: {[round(v, 4) for v in window]})"
                )
            else:
                epochs_since_best += 1

        epoch_rows.append({
            'seed': seed, 'epoch': epoch, 'train_loss': train_loss,
            'val_loss': results['val_loss'], 'val_dice': results['dice'],
            'val_iou': results['iou'], 'val_wt': results['wt'],
            'val_tc': results['tc'], 'val_et': results['et'], 'lr': current_lr,
            # NEW in v0.9 columns (appended — no existing column removed)
            'cls_accuracy': results['cls_accuracy'],
            'cls_f1': results['cls_f1'],
            'cls_majority_baseline': results['cls_majority_baseline'],
            'val_cls_loss': results['val_cls_loss'],
            'L_seg': losses['l_seg'],
            'L_cls': losses['l_cls'],
            'L_total': losses['l_total'],
            # NEW in v0.9-metrics (Task F): trailing columns only — the DictWriter
            # takes its fieldnames from this dict's key order, so every v0.9 column
            # above keeps its exact position and name.
            **{_k: results[_k] for _k in _V0_9_METRICS_EPOCH_KEYS},        # NEW in v0.9-metrics
        })

        # ---- Early stopping (Task G) — same signal as checkpoint selection ----
        if Config.USE_EARLY_STOPPING and epochs_since_best >= Config.EARLY_STOP_PATIENCE:
            stopped_early_at = epoch
            logger.warning(
                f"🛑 EARLY STOP at epoch {epoch}/{Config.NUM_EPOCHS_SEG} (seed {seed}) — "
                f"moving-average Val Dice has not improved for {epochs_since_best} consecutive "
                f"epochs (patience={Config.EARLY_STOP_PATIENCE}). Best was epoch {best_epoch} "
                f"with moving avg {best_moving_avg:.4f}."
            )
            break

    seed_history_path = os.path.join(Config.V0_9_LOG_DIR, f'multitask_history_seed_{seed}.csv')
    with open(seed_history_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(epoch_rows[0].keys()))
        writer.writeheader()
        writer.writerows(epoch_rows)
    logger.info(f"💾 Seed {seed} epoch history saved to {seed_history_path}")

    # ---- Final metrics for this seed: reload the moving-avg-best checkpoint ----
    if best_epoch is not None:
        logger.info(f"📌 Seed {seed}: reloading best checkpoint from epoch {best_epoch} for final metrics.")
        model.load_state_dict(torch.load(best_ckpt_path, map_location=device))
    else:
        logger.warning(
            f"⚠️ Seed {seed}: moving-average criterion never triggered (fewer than "
            f"{Config.MOVING_AVG_WINDOW} valid epochs?). Using final-epoch weights instead."
        )

    final_results = evaluate_multitask_full(
        model, val_loader, device, num_classes=Config.NUM_SEG_CLASSES,
        criterion=criterion, cls_criterion=cls_criterion, visualize=True,
        save_path=os.path.join(Config.V0_9_LOG_DIR, f'seg_visualization_seed_{seed}.png'),
    )

    acc = final_results['cls_accuracy']
    baseline = final_results['cls_majority_baseline']
    if acc == acc and baseline == baseline and acc > baseline:
        cls_signal = "🟢 GREEN — Cls Head > Majority Baseline"
    else:
        cls_signal = "🔴 RED — Cls Head <= Majority Baseline"

    logger.info(
        f"🏁 Seed {seed} FINAL (best checkpoint, epoch {best_epoch}) — "
        f"Dice: {final_results['dice']:.4f} | IoU: {final_results['iou']:.4f} | "
        f"WT: {final_results['wt']:.4f} | TC: {final_results['tc']:.4f} | ET: {final_results['et']:.4f}"
    )
    logger.info(
        f"🧬 Seed {seed} FINAL Cls — Accuracy: {acc:.4f} | F1: {final_results['cls_f1']:.4f} | "
        f"Majority baseline: {baseline:.4f} | {cls_signal}"
    )
    logger.info(
        f"📎 Reference — v0.8 Linear Probe accuracy was {Config.V0_8_LINEAR_PROBE_REFERENCE_ACC:.2f}; "
        f"v0.9 joint-trained head {'BEATS' if (acc == acc and acc > Config.V0_8_LINEAR_PROBE_REFERENCE_ACC) else 'does NOT beat'} it."
    )

    return {
        'seed': seed,
        'best_epoch': best_epoch if best_epoch is not None else Config.NUM_EPOCHS_SEG,
        'stopped_early_at': stopped_early_at if stopped_early_at is not None else '',
        'dice': final_results['dice'],
        'iou': final_results['iou'],
        'wt': final_results['wt'],
        'tc': final_results['tc'],
        'et': final_results['et'],
        'cls_accuracy': final_results['cls_accuracy'],
        'cls_f1': final_results['cls_f1'],
        'cls_majority_baseline': final_results['cls_majority_baseline'],
        'cls_signal': cls_signal,
    }


# ------------------------------------------------------------------------------
# 23i. NEW in v0.9 — Mode C runner: multi-task multi-seed protocol.
#      Sibling of run_mode_b_segmentation() (untouched). Reuses the SAME
#      persisted patient split (Config.SPLIT_FILE_PATH) — it is loaded, never
#      recomputed — so v0.9 trains/validates on exactly the same patients as
#      v0.7-A / v0.7-B / v0.8, and the Test set stays reserved for v1.0.
# ------------------------------------------------------------------------------
def run_mode_c_multitask() -> None:                                      # NEW in v0.9
    log_section("🧩 MedSSL-3D v0.9 — Mode C: Multi-Task (Segmentation + HGG/LGG Classification)")

    print_environment_report()
    Config.setup_dirs()
    device = torch.device(Config.DEVICE)

    logger.info(f"\n📂 DATA_ROOT : {Config.DATA_ROOT}")
    logger.info(f"📂 MODALITY  : {Config.MODALITY}")
    logger.info(f"📂 v0.9 checkpoints -> {Config.V0_9_CHECKPOINT_DIR}")
    logger.info(f"📂 v0.9 logs        -> {Config.V0_9_LOG_DIR}")
    logger.info(f"📂 v0.9 csv         -> {Config.V0_9_CSV_DIR}")
    logger.info(f"⚙️ USE_CLASSIFICATION_HEAD = {Config.USE_CLASSIFICATION_HEAD} | "
                f"LAMBDA_SEG={Config.LAMBDA_SEG} LAMBDA_CLS={Config.LAMBDA_CLS}")
    logger.info(f"⚙️ USE_CHANNEL_SE_ATTENTION = {Config.USE_CHANNEL_SE_ATTENTION} (unchanged from v0.8)")
    logger.info(f"⚙️ Augmentation — elastic={Config.USE_ELASTIC_DEFORM} gamma={Config.USE_GAMMA_ADJUST} "
                f"bias_field={Config.USE_BIAS_FIELD}")
    logger.info(f"⚙️ Warmup={Config.WARMUP_EPOCHS} | GradClip={Config.GRAD_CLIP_NORM} | "
                f"EarlyStop={Config.USE_EARLY_STOPPING} (patience={Config.EARLY_STOP_PATIENCE}) | "
                f"Epochs={Config.NUM_EPOCHS_SEG}")

    set_seed(Config.SEED)
    pairs = run_segmentation_sanity_check(Config.DATA_ROOT, Config.MODALITY, Config.IMG_SIZE)

    train_pairs, val_pairs, test_pairs = create_or_load_split(
        pairs, Config.SPLIT_FILE_PATH,
        Config.TRAIN_FRACTION, Config.VAL_FRACTION, Config.TEST_FRACTION,
        Config.SEED,
    )

    batch_size = min(Config.BATCH_SIZE_SEG, max(1, len(train_pairs)))
    if batch_size != Config.BATCH_SIZE_SEG:
        logger.warning(f"⚠️ Reducing segmentation BATCH_SIZE to {batch_size} (dataset is small).")

    grades = load_patient_grades()   # loaded once, shared by both datasets

    train_dataset = MultiTaskSegmentationDataset(
        train_pairs, img_size=Config.IMG_SIZE, num_classes=Config.NUM_SEG_CLASSES,
        grades=grades, augment=True, split_name='train',
    )
    val_dataset = MultiTaskSegmentationDataset(
        val_pairs, img_size=Config.IMG_SIZE, num_classes=Config.NUM_SEG_CLASSES,
        grades=grades, augment=False, split_name='val',       # never augment validation
    )
    # test_pairs is intentionally NEVER wrapped in a Dataset/DataLoader — still
    # reserved exclusively for v1.0.

    # ---- Task G: WeightedRandomSampler replaces shuffle=True ----
    # It gets its OWN generator, re-seeded per seed inside
    # run_single_seed_multitask, exactly like the v0.8 DataLoader fix.
    if Config.USE_WEIGHTED_SAMPLER:
        sampler_generator = torch.Generator()
        sampler_generator.manual_seed(Config.SEED)
        train_sampler = torch.utils.data.WeightedRandomSampler(
            weights=train_dataset.build_sampler_weights(),
            num_samples=len(train_dataset),      # same epoch length as v0.8
            replacement=True,
            generator=sampler_generator,
        )
        logger.info(f"⚖️ WeightedRandomSampler active — {len(train_dataset)} draws/epoch, "
                    f"class-balanced over HGG/LGG.")
        train_loader = DataLoader(
            train_dataset, batch_size=batch_size, sampler=train_sampler,
            num_workers=Config.NUM_WORKERS, pin_memory=(device.type == 'cuda'),
            generator=torch.Generator(),          # v0.8 RNG fix — preserved
        )
    else:
        train_loader = DataLoader(
            train_dataset, batch_size=batch_size, shuffle=True,
            num_workers=Config.NUM_WORKERS, pin_memory=(device.type == 'cuda'),
            generator=torch.Generator(),          # v0.8 RNG fix — preserved
        )

    val_loader = DataLoader(
        val_dataset, batch_size=1, shuffle=False,
        num_workers=Config.NUM_WORKERS, pin_memory=(device.type == 'cuda'),
    )

    logger.info(f"Encoder path: {Config.PRETRAINED_ENCODER_PATH}")
    logger.info(f"Exists? {os.path.isfile(Config.PRETRAINED_ENCODER_PATH)}")

    seed_results: List[dict] = []
    for seed in Config.SEEDS:
        result = run_single_seed_multitask(seed, train_loader, val_loader, device)
        seed_results.append(result)

    # ---- Aggregate Mean +/- Std across seeds ----
    log_section("📈 v0.9 MULTI-SEED SUMMARY (Multi-Task)")
    metric_keys = ['dice', 'iou', 'wt', 'tc', 'et', 'cls_accuracy', 'cls_f1']
    summary_rows = []
    for r in seed_results:
        row = {'seed': r['seed'], 'best_epoch': r['best_epoch'],
               'stopped_early_at': r['stopped_early_at']}
        row.update({k: r[k] for k in metric_keys})
        row['cls_majority_baseline'] = r['cls_majority_baseline']
        summary_rows.append(row)
        logger.info(
            f"Seed {r['seed']:>5} | best_epoch={r['best_epoch']:>3} | "
            f"Dice={r['dice']:.4f} IoU={r['iou']:.4f} WT={r['wt']:.4f} "
            f"TC={r['tc']:.4f} ET={r['et']:.4f} | ClsAcc={r['cls_accuracy']:.4f} "
            f"ClsF1={r['cls_f1']:.4f} | {r['cls_signal']}"
        )

    mean_row = {'seed': 'mean', 'best_epoch': '', 'stopped_early_at': '', 'cls_majority_baseline': ''}
    std_row = {'seed': 'std', 'best_epoch': '', 'stopped_early_at': '', 'cls_majority_baseline': ''}
    for k in metric_keys:
        values = [r[k] for r in seed_results if r[k] == r[k]]
        mean_row[k] = float(np.mean(values)) if values else float('nan')
        std_row[k] = float(np.std(values)) if values else float('nan')
    summary_rows.append(mean_row)
    summary_rows.append(std_row)

    logger.info(
        f"\n🏆 MEAN ± STD across {len(seed_results)} seeds:\n" +
        "\n".join(f"   {k.upper():13s}: {mean_row[k]:.4f} ± {std_row[k]:.4f}" for k in metric_keys)
    )
    logger.info(
        f"\n📎 v0.8 references — Linear Probe accuracy {Config.V0_8_LINEAR_PROBE_REFERENCE_ACC:.2f}. "
        f"v0.9 mean Cls accuracy = {mean_row['cls_accuracy']:.4f}."
    )

    with open(Config.SEG_SUMMARY_CSV_V0_9, 'w', newline='') as f:
        fieldnames = ['seed', 'best_epoch', 'stopped_early_at'] + metric_keys + ['cls_majority_baseline']
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(summary_rows)
    logger.info(f"💾 v0.9 multi-seed summary saved to {Config.SEG_SUMMARY_CSV_V0_9}")

    log_section("🎉 v0.9 MODE C RUN COMPLETE")
    logger.info(f"💾 v0.9 Checkpoints -> {Config.V0_9_CHECKPOINT_DIR} (per-seed subfolders)")
    logger.info(f"💾 v0.9 Logs        -> {Config.V0_9_LOG_DIR}")
    logger.info(f"💾 v0.9 CSV         -> {Config.V0_9_CSV_DIR}")
    logger.info(f"🔒 Test set ({len(test_pairs)} patients) reserved untouched for v1.0.")
    logger.info("📌 v0.7-A / v0.7-B / v0.8 outputs are untouched — v0.9 writes only under outputs/v0_9/.")


# ------------------------------------------------------------------------------
# 23j. NEW in v0.9 — self-check: verifies the v0.8 path is byte-for-byte
#      preserved (forward() with the default kwargs) and that the multi-task
#      wrapper is wired correctly. Runs on CPU with tiny random tensors — no
#      dataset, no GPU, ~seconds. Called from main() before training.
# ------------------------------------------------------------------------------
@torch.no_grad()
def run_v0_9_self_check() -> None:                                       # NEW in v0.9
    log_section("🔍 v0.9 SELF-CHECK (no data / no GPU required)")
    cpu = torch.device('cpu')
    small = (32, 32, 16)

    torch.manual_seed(0)
    encoder = resnet18_3d(in_channels=4)
    seg_model = SegmentationModel3D(encoder=encoder, num_classes=Config.NUM_SEG_CLASSES,
                                    target_size=small).to(cpu)
    seg_model.eval()
    x = torch.randn(1, 4, *small)

    # 1) Default forward path unchanged vs. return_bottleneck=True's seg output.
    out_default = seg_model(x)
    out_with_bn, l4 = seg_model(x, return_bottleneck=True)
    same = torch.allclose(out_default, out_with_bn, atol=0.0)
    logger.info(f"   [1] forward() default == forward(return_bottleneck=True)[0]: "
                f"{'✅ identical' if same else '❌ DIFFERENT'} | l4 shape={tuple(l4.shape)}")

    # 2) Multi-task wrapper returns (seg, cls_logits) with the right shapes.
    mt = MultiTaskModel3D(seg_model).to(cpu)
    mt.eval()
    seg_out, cls_logits = mt(x)
    logger.info(f"   [2] MultiTaskModel3D eval -> seg {tuple(seg_out.shape)}, "
                f"cls {tuple(cls_logits.shape)} {'✅' if cls_logits.shape[-1] == 2 else '❌'}")

    # 3) Train-mode deep supervision still returns the (main, aux1, aux2) tuple.
    #    Batch of 2: BatchNorm3d cannot run in train mode on a 1-sample batch
    #    whose bottleneck is 1x1x1 (this tiny 32x32x16 probe input, not the real
    #    128x128x64 volumes).
    mt.train()
    x2 = torch.randn(2, 4, *small)
    with torch.enable_grad():
        seg_out_train, cls_logits_train = mt(x2)
    ok3 = isinstance(seg_out_train, tuple) and len(seg_out_train) == 3
    logger.info(f"   [3] MultiTaskModel3D train -> deep-supervision tuple preserved: "
                f"{'✅' if ok3 else '❌'}")

    # 4) Elastic deformation keeps image and mask synchronized in shape/labels.
    vol = torch.randn(4, *small)
    msk = torch.randint(0, Config.NUM_SEG_CLASSES, small)
    wv, wm = apply_elastic_deformation(vol, msk)
    ok4 = (wv.shape == vol.shape and wm.shape == msk.shape
           and wm.dtype == torch.long and int(wm.max()) < Config.NUM_SEG_CLASSES)
    logger.info(f"   [4] Elastic deform image+mask synchronized & label-safe: {'✅' if ok4 else '❌'} "
                f"(vol {tuple(wv.shape)}, mask {tuple(wm.shape)}, labels<= {int(wm.max())})")

    # 5) All-unlabeled batch -> L_cls is exactly 0.0, never NaN.
    ce = nn.CrossEntropyLoss(ignore_index=MultiTaskSegmentationDataset.UNKNOWN_LABEL)
    fake_logits = torch.randn(2, 2)
    fake_targets = torch.tensor([-1, -1])
    guarded = torch.zeros(()) if not bool((fake_targets != -1).any()) else ce(fake_logits, fake_targets)
    logger.info(f"   [5] All-unlabeled batch guard -> L_cls={guarded.item():.1f} "
                f"{'✅ (no NaN)' if torch.isfinite(guarded) else '❌'}")

    logger.info("🔍 Self-check done.\n")


# ==============================================================================
# NEW in v0.9-diag — SECTION 24-DIAG: READ-ONLY GRADE-INFORMATION DIAGNOSTIC
#
# Nothing below trains, fine-tunes, or overwrites anything. It loads the
# already-trained v0.9 checkpoint, freezes it, extracts BOTH candidate feature
# spaces in ONE forward pass, and trains three tiny probes on the cached
# 512-d vectors. Purpose: separate four competing explanations for
# "v0.9 Cls accuracy == majority baseline" BEFORE spending GPU-hours on a fix.
#
#   H1 optimization  — the joint head's effective LR is LR_SEG * LAMBDA_CLS
#                      = 3e-5 (33x below the v0.8 probe's 1e-3), further
#                      attenuated by a GLOBAL clip_grad_norm_ dominated by the
#                      segmentation branch.
#   H2 representation— segmentation fine-tuning erased HGG/LGG information
#                      from the shared encoder.
#   H3 SE attention  — SEChannelAttention3D suppresses the grade-discriminative
#                      channels. NOTE the never-before-compared asymmetry: the
#                      v0.8 probe read the RAW encoder vector (no SE), the v0.9
#                      head reads the POST-SE bottleneck. 0.80 vs 0.7636 was
#                      never apples-to-apples.
#   H4 architecture  — 512->128->2 with Dropout(0.3) is weaker than a bare
#                      nn.Linear(512, 2).
#
#   Probe A: raw    features + nn.Linear(512,2)     -> isolates H2 (vs v0.8's 0.80)
#   Probe B: postse features + nn.Linear(512,2)     -> isolates H3
#   Probe C: postse features + ClassificationHead3D -> isolates H1 vs H4
#
# The Test split is never wrapped in a Dataset. The only file written is
# Config.DIAG_CSV_PATH. Section numbering note: the pristine file already uses
# "# 24. Dispatcher", so this block is labelled 24a-24h to avoid renumbering an
# existing comment line.
# ==============================================================================

# ------------------------------------------------------------------------------
# 24a. NEW in v0.9-diag — small shared helpers.
# ------------------------------------------------------------------------------
class _DiagCachedFeatureHead(nn.Module):                                 # NEW in v0.9-diag
    """Adapter so a head that expects the (B, 512, d, h, w) l4 map can be
    trained on the CACHED (B, 512) vectors.

    ClassificationHead3D starts with AdaptiveAvgPool3d(1); pooling an already
    (B, 512, 1, 1, 1) tensor is the identity, so the wrapped head sees exactly
    the same tensor it would see in the real multi-task model. This is why the
    architecture comparison in probe C stays honest — no layer is skipped or
    re-implemented.
    """
    def __init__(self, inner: nn.Module):
        super().__init__()
        self.inner = inner

    def forward(self, feats: torch.Tensor) -> torch.Tensor:
        b, c = feats.shape[0], feats.shape[1]
        return self.inner(feats.view(b, c, 1, 1, 1))


def _diag_class_weight(labels: List[int], device: torch.device) -> Optional[torch.Tensor]:  # NEW in v0.9-diag
    """The class-weight rule COPIED VERBATIM from run_linear_probe(): inverse
    frequency, applied only when max/min >= 1.5. Not re-derived, not improved —
    a different rule here would silently invalidate the comparison against the
    v0.8 number 0.80."""
    n_hgg = sum(labels)
    n_lgg = len(labels) - n_hgg
    if n_hgg > 0 and n_lgg > 0 and max(n_hgg, n_lgg) / min(n_hgg, n_lgg) >= 1.5:
        return torch.tensor(
            [len(labels) / (2.0 * n_lgg), len(labels) / (2.0 * n_hgg)], device=device,
        )
    return None


def _diag_resolve_checkpoint_path() -> str:                              # NEW in v0.9-diag
    """None -> outputs/v0_9/checkpoints/seed_{DIAG_SEED}/best_moving_avg_model.pth,
    i.e. exactly where run_single_seed_multitask() saved it."""
    if Config.DIAG_CHECKPOINT_PATH:
        return Config.DIAG_CHECKPOINT_PATH
    return os.path.join(Config.V0_9_CHECKPOINT_DIR, f'seed_{Config.DIAG_SEED}',
                        'best_moving_avg_model.pth')


def _diag_resolve_device() -> torch.device:                              # NEW in v0.9-diag
    """Config.DEVICE is 'cuda' project-wide. This diagnostic is cheap enough to
    fall back to CPU rather than crash, so a missing GPU costs minutes, not a
    failed run."""
    if Config.DEVICE == 'cuda' and not torch.cuda.is_available():
        logger.warning("⚠️ CUDA not available — running the diagnostic on CPU (slower, identical numbers).")
        return torch.device('cpu')
    return torch.device(Config.DEVICE)


# ------------------------------------------------------------------------------
# 24b. NEW in v0.9-diag — Task B: ONE forward pass, BOTH feature spaces.
#
#      ResNet3D.forward(x, return_feature_map=True) already returns
#      (l4_map, global_vec), so no extra compute is needed to get both:
#          feat_raw    = global_vec                     == avgpool(l4), PRE-SE.
#                        Exactly what run_linear_probe() consumed in v0.8.
#          feat_postse = AdaptiveAvgPool3d(1)(SE(l4))   == what
#                        ClassificationHead3D.pool sees in v0.9.
#      Extraction runs ONCE for the whole diagnostic; the three probes then
#      train on cached vectors in seconds.
# ------------------------------------------------------------------------------
@torch.no_grad()
def _diag_features_from_batch(seg_model: SegmentationModel3D,             # NEW in v0.9-diag
                              volume: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    """The single source of truth for both feature spaces. Factored out so
    run_v0_9_diag_self_check() can assert THIS code path — not a re-typed copy
    of it — matches encoder(x) and AdaptiveAvgPool3d(1)(SE(l4))."""
    encoder = seg_model.get_encoder()
    l4, global_vec = encoder(volume, return_feature_map=True)
    feat_raw = global_vec                                   # (B, 512), pre-SE
    se = seg_model.channel_attention
    if se is None:
        feat_postse = feat_raw                              # SE disabled -> spaces coincide
    else:
        feat_postse = torch.flatten(F.adaptive_avg_pool3d(se(l4), 1), 1)
    return feat_raw, feat_postse


@torch.no_grad()
def extract_grade_features(model: nn.Module, pairs: List[Tuple[str, str]],  # NEW in v0.9-diag
                           grades: dict, device: torch.device) -> dict:
    """Returns {'raw': (N,512), 'postse': (N,512), 'labels': (N,)} on CPU.

    Reuses LinearProbeDataset VERBATIM: identical preprocessing, identical
    MODALITY_ORDER, identical HGG/LGG filtering. A new Dataset here would risk a
    silent preprocessing drift that would invalidate the comparison against 0.80.
    """
    seg_model = model.get_seg_model() if isinstance(model, MultiTaskModel3D) else model

    if seg_model.channel_attention is None:
        logger.warning("⚠️ USE_CHANNEL_SE_ATTENTION is False on this model — SE is disabled, so "
                       "feat_postse == feat_raw by construction and probes A and B are identical. "
                       "H3 cannot be tested from this checkpoint.")

    dataset = LinearProbeDataset(pairs, grades, img_size=Config.IMG_SIZE)
    if len(dataset) == 0:
        logger.warning("⚠️ No HGG/LGG-labeled patients in this split — nothing to extract.")
        return {'raw': torch.empty(0, 512), 'postse': torch.empty(0, 512),
                'labels': torch.empty(0, dtype=torch.long)}

    loader = DataLoader(dataset, batch_size=Config.DIAG_FEATURE_BATCH, shuffle=False,
                        num_workers=Config.NUM_WORKERS)

    # eval() matters beyond speed: BatchNorm3d in train mode would recompute
    # statistics from the diagnostic's own batches and change the very features
    # being measured.
    model.eval()

    raw_chunks, postse_chunks, label_chunks = [], [], []
    t0 = time.time()
    for volume, label in tqdm(loader, desc='Extracting grade features', leave=False):
        volume = volume.to(device, non_blocking=True)
        feat_raw, feat_postse = _diag_features_from_batch(seg_model, volume)
        raw_chunks.append(feat_raw.detach().float().cpu())
        postse_chunks.append(feat_postse.detach().float().cpu())
        label_chunks.append(label.detach().long().cpu() if torch.is_tensor(label)
                            else torch.as_tensor(label, dtype=torch.long))
    elapsed = time.time() - t0

    out = {
        'raw': torch.cat(raw_chunks, dim=0),
        'postse': torch.cat(postse_chunks, dim=0),
        'labels': torch.cat(label_chunks, dim=0),
    }
    n = out['labels'].numel()
    n_hgg = int(out['labels'].sum().item())
    logger.info(
        f"   🔬 Extracted {n} sample(s) in {elapsed:.1f}s "
        f"({(n / elapsed) if elapsed > 0 else float('nan'):.2f} samples/s) | "
        f"HGG={n_hgg} LGG={n - n_hgg} | raw {tuple(out['raw'].shape)} postse {tuple(out['postse'].shape)}"
    )
    return out


# ------------------------------------------------------------------------------
# 24c. NEW in v0.9-diag — Task C: probe trainer over CACHED features.
#      head_factory is a zero-argument callable, so the SAME trainer serves a
#      bare nn.Linear(512,2) and a full ClassificationHead3D. Everything that
#      could shift the number relative to v0.8 (LR, epochs, batch size, class
#      weighting, metric definitions) is mirrored from run_linear_probe().
# ------------------------------------------------------------------------------
def train_probe_on_features(train_feats: torch.Tensor, train_labels: torch.Tensor,   # NEW in v0.9-diag
                            val_feats: torch.Tensor, val_labels: torch.Tensor,
                            head_factory, device: torch.device,
                            seed: int, tag: str) -> dict:
    # set_seed BEFORE instantiating the head, so the three probes are not
    # confounded by initialization luck and the whole thing is reproducible.
    set_seed(seed)
    head = head_factory().to(device)
    optimizer = optim.Adam(head.parameters(), lr=Config.DIAG_PROBE_LR)

    # v0.8 RNG rule applies here too: batch order depends only on the seed,
    # never on how much global RNG head construction happened to consume.
    gen = torch.Generator()
    gen.manual_seed(seed)

    train_x = train_feats.to(device)
    train_y = train_labels.to(device).long()
    val_x = val_feats.to(device)
    val_y = val_labels.to(device).long()

    labels_all = train_labels.long().tolist()
    class_weight = _diag_class_weight(labels_all, device)
    if class_weight is not None:
        criterion = nn.CrossEntropyLoss(weight=class_weight)
        n_hgg = sum(labels_all)
        logger.info(f"   ⚖️ [{tag}] Class imbalance (HGG={n_hgg}, LGG={len(labels_all) - n_hgg}) "
                    f"— using weighted CrossEntropy (same rule as run_linear_probe).")
    else:
        criterion = nn.CrossEntropyLoss()

    n_train = train_x.shape[0]
    bs = max(1, Config.DIAG_PROBE_BATCH_SIZE)
    final_train_loss = float('nan')

    for epoch in range(1, Config.DIAG_PROBE_EPOCHS + 1):
        head.train()                    # probe C has Dropout — this is not cosmetic
        perm = torch.randperm(n_train, generator=gen)
        epoch_loss, n_batches = 0.0, 0
        for start in range(0, n_train, bs):
            idx = perm[start:start + bs].to(device)
            logits = head(train_x[idx])
            loss = criterion(logits, train_y[idx])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += float(loss.item())
            n_batches += 1
        if n_batches:
            final_train_loss = epoch_loss / n_batches
            logger.info(f"   [{tag}] probe epoch {epoch}/{Config.DIAG_PROBE_EPOCHS} — loss={final_train_loss:.4f}")

    # ---- Evaluate (metric definitions copied verbatim from run_linear_probe) ----
    head.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for start in range(0, val_x.shape[0], bs):
            logits = head(val_x[start:start + bs])
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(val_y[start:start + bs].cpu().tolist())

    correct = sum(p == l for p, l in zip(all_preds, all_labels))
    accuracy = correct / len(all_labels) if all_labels else float('nan')

    tp = sum((p == 1 and l == 1) for p, l in zip(all_preds, all_labels))
    fp = sum((p == 1 and l == 0) for p, l in zip(all_preds, all_labels))
    fn = sum((p == 0 and l == 1) for p, l in zip(all_preds, all_labels))
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    n_pos = sum(all_labels)
    majority_class = 1 if n_pos >= (len(all_labels) - n_pos) else 0
    majority_baseline = (sum(l == majority_class for l in all_labels) / len(all_labels)
                          if all_labels else float('nan'))

    if accuracy == accuracy and majority_baseline == majority_baseline and accuracy > majority_baseline:
        signal = "🟢 GREEN — Probe > Baseline"
    else:
        signal = "🔴 RED — Probe <= Baseline"

    logger.info(
        f"   🔬 [{tag}] Accuracy={accuracy:.4f} | F1={f1:.4f} | "
        f"Majority baseline={majority_baseline:.4f} | {signal}"
    )

    return {
        'tag': tag,
        'accuracy': accuracy,
        'f1': f1,
        'majority_baseline': majority_baseline,
        'final_train_loss': final_train_loss,
        'n_val': len(all_labels),
        # extra keys, for the CSV only — the six above are the contract
        'precision': precision,
        'recall': recall,
        'signal': signal,
        'n_train': n_train,
    }


# ------------------------------------------------------------------------------
# 24d. NEW in v0.9-diag — Task E: the decision table.
#      Verdicts are printed as an explicit conclusion, not left to the reader.
# ------------------------------------------------------------------------------
def _diag_near(a: float, b: float, tol: float = None) -> bool:            # NEW in v0.9-diag
    tol = Config.DIAG_NEAR_TOL if tol is None else tol
    return (a == a) and (b == b) and abs(a - b) <= tol


def _diag_emit_decision_table(res_a: dict, res_b: dict, res_c: dict) -> List[str]:   # NEW in v0.9-diag
    acc_a, acc_b, acc_c = res_a['accuracy'], res_b['accuracy'], res_c['accuracy']
    ref_probe = Config.V0_8_LINEAR_PROBE_REFERENCE_ACC          # 0.80
    ref_base = Config.DIAG_REF_MAJORITY_BASELINE               # 0.7636
    ref_joint = Config.DIAG_REF_V0_9_JOINT_ACC                 # 0.7636
    gap = Config.DIAG_GAP_TOL

    log_section("🏁 v0.9-diag — DECISION TABLE")
    logger.info("📊 Anchors (logged, NOT re-derived here):")
    logger.info(f"   v0.8 Linear Probe accuracy      = {ref_probe:.4f}")
    logger.info(f"   v0.9 joint cls head accuracy    = {ref_joint:.4f}")
    logger.info(f"   Majority baseline               = {ref_base:.4f}")
    logger.info("")
    logger.info(f"   Probe A (raw    + Linear)            = {acc_a:.4f}")
    logger.info(f"   Probe B (postse + Linear)            = {acc_b:.4f}")
    logger.info(f"   Probe C (postse + ClassificationHead)= {acc_c:.4f}")
    logger.info("")

    verdicts: List[str] = []

    # ---- H2: did segmentation fine-tuning erase the grade signal? ----
    if acc_a > ref_probe + Config.DIAG_NEAR_TOL:
        v = (f"A = {acc_a:.4f} EXCEEDS the v0.8 probe anchor {ref_probe:.4f} -> grade information "
             f"not only survived segmentation fine-tuning, it is more linearly separable than in "
             f"v0.8. H2 REJECTED. The failure is in how the head is OPTIMIZED.")
        logger.info(f"   ✅ {v}")
    elif _diag_near(acc_a, ref_probe):
        v = ("A ≈ 0.80 -> grade information SURVIVED segmentation fine-tuning. "
             "H2 REJECTED. The failure is in how the head is OPTIMIZED.")
        logger.info(f"   ✅ {v}")
    elif acc_a <= ref_base + Config.DIAG_NEAR_TOL:
        v = ("A ≈ 0.76 -> grade information was ERASED by fine-tuning. "
             "H2 CONFIRMED. No sampler or LR change can fix this; LAMBDA_CLS must rise "
             "or the head must branch earlier.")
        logger.info(f"   🔴 {v}")
    else:
        v = (f"A = {acc_a:.4f} sits between the {ref_base:.4f} baseline and the {ref_probe:.4f} "
             f"probe anchor (tolerance ±{Config.DIAG_NEAR_TOL}). H2 is PARTIAL: some grade "
             f"information survived, some was lost. Read this as 'degraded', not 'erased'.")
        logger.info(f"   ⚠️ {v}")
    verdicts.append(v)

    # ---- H3: does SE attention suppress the grade channels? ----
    if acc_b < acc_a - gap:
        v = (f"B ({acc_b:.4f}) << A ({acc_a:.4f}) -> SE attention SUPPRESSES grade channels. "
             f"H3 CONFIRMED; the cls head should read the PRE-SE bottleneck.")
        logger.info(f"   🔴 {v}")
    elif acc_b > acc_a + gap:
        v = (f"B ({acc_b:.4f}) > A ({acc_a:.4f}) -> SE attention HELPS grade separability. "
             f"H3 rejected in the opposite direction; keep the post-SE bottleneck.")
        logger.info(f"   ✅ {v}")
    else:
        v = (f"B ≈ A ({acc_b:.4f} vs {acc_a:.4f}) -> SE attention is NEUTRAL for grade. H3 REJECTED.")
        logger.info(f"   ✅ {v}")
    verdicts.append(v)

    # ---- H1 vs H4: optimization or architecture? ----
    if acc_c < acc_a - gap:
        v = (f"C ({acc_c:.4f}) << A ({acc_a:.4f}) -> the head ARCHITECTURE is the bottleneck. "
             f"H4 CONFIRMED; revisit CLS_HEAD_DROPOUT / CLS_HEAD_HIDDEN_DIM.")
        logger.info(f"   🔴 {v}")
    elif acc_c >= ref_probe - Config.DIAG_NEAR_TOL and acc_a >= ref_probe - Config.DIAG_NEAR_TOL:
        v = (f"C ({acc_c:.4f}) ≈ 0.80 with A ({acc_a:.4f}) ≈ 0.80 -> the head architecture is "
             f"ADEQUATE. H1 CONFIRMED: the joint-training step size (LR_SEG x LAMBDA_CLS = 3e-5, "
             f"plus the global grad-norm clip) is the binding constraint.")
        logger.info(f"   ✅ {v}")
    else:
        v = (f"C ≈ A ({acc_c:.4f} vs {acc_a:.4f}) -> the head architecture is not the bottleneck "
             f"relative to a bare Linear, but neither probe reached the {ref_probe:.4f} anchor. "
             f"H4 rejected; H1 supported only as far as A allows.")
        logger.info(f"   ⚠️ {v}")
    verdicts.append(v)

    # ---- Contradiction guard (success criterion: never resolve one silently) ----
    if _diag_near(acc_a, ref_base) and acc_c > ref_base + gap:
        logger.warning("⚠️ CONTRADICTION: probe A sits at the baseline while probe C clears it on the "
                       "SAME frozen encoder. The 'information was erased' reading cannot hold — "
                       "report this rather than picking a story.")
    if acc_a < ref_base - gap:
        logger.warning(f"⚠️ CONTRADICTION with Section 0's framing: probe A ({acc_a:.4f}) is BELOW the "
                       f"majority baseline ({ref_base:.4f}). A frozen-feature linear probe underperforming "
                       f"'always predict HGG' points at the probe's own optimization (class weighting on a "
                       f"55-sample val set), not at the encoder.")
    measured_base = res_a['majority_baseline']
    if not _diag_near(measured_base, ref_base, 1e-3):
        logger.warning(f"⚠️ Measured majority baseline ({measured_base:.4f}) differs from the recorded "
                       f"anchor ({ref_base:.4f}). The val split or the grade join has changed — the "
                       f"0.80 / 0.7636 comparisons below are no longer strictly apples-to-apples.")
    return verdicts


# ------------------------------------------------------------------------------
# 24e. NEW in v0.9-diag — Task D: the three probes, end to end.
# ------------------------------------------------------------------------------
def run_v0_9_grade_diagnostic(device: torch.device = None) -> dict:      # NEW in v0.9-diag
    log_section("🔬 MedSSL-3D v0.9-diag — GRADE-INFORMATION DIAGNOSTIC (read-only)")

    if not Config.RUN_GRADE_DIAGNOSTIC:
        logger.warning("⚠️ RUN_GRADE_DIAGNOSTIC=False — no checkpoint loaded, nothing written.")
        return {'skipped': True}

    device = _diag_resolve_device() if device is None else device
    Config.setup_dirs()          # only ever creates directories; overwrites nothing

    logger.info("🔒 Read-only run: no optimizer over the backbone, no training loader, "
                "no run_single_seed_* / run_mode_* call, no checkpoint written.")

    # ---- Resolve and validate the checkpoint --------------------------------
    ckpt_path = _diag_resolve_checkpoint_path()
    logger.info(f"📂 Checkpoint : {ckpt_path}")
    if not os.path.isfile(ckpt_path):
        logger.error(f"🔴 Checkpoint not found at the resolved path: {ckpt_path}\n"
                     f"   Set Config.DIAG_CHECKPOINT_PATH explicitly, or check Config.DIAG_SEED "
                     f"(currently {Config.DIAG_SEED}). Refusing to fall back to a different "
                     f"checkpoint — that would silently measure the wrong model.")
        return {'skipped': True}

    if not Config.USE_CLASSIFICATION_HEAD:
        logger.error("🔴 USE_CLASSIFICATION_HEAD=False, so build_fresh_multitask_model() returns a plain "
                     "SegmentationModel3D and a strict load of a MultiTaskModel3D checkpoint will fail. "
                     "Set it back to True to run the diagnostic.")
        return {'skipped': True}

    # ---- Build + load STRICTLY ---------------------------------------------
    model = build_fresh_multitask_model(device)
    try:
        state_dict = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(state_dict)          # strict on purpose
    except Exception as e:
        logger.error(f"🔴 Strict state_dict load failed — stopping.\n{e}\n"
                     f"   A strict=False fallback here would measure a partially-random encoder "
                     f"and produce a meaningless answer.")
        return {'skipped': True}
    logger.info("✅ Checkpoint loaded strictly into MultiTaskModel3D.")

    # Frozen for the entire diagnostic: eval() + requires_grad_(False).
    model.eval()
    model.requires_grad_(False)
    logger.info("🧊 Encoder + SE + decoder frozen (eval + requires_grad_(False)).")

    # ---- Persisted split: LOADED, never recomputed --------------------------
    if not os.path.isfile(Config.SPLIT_FILE_PATH):
        logger.error(f"🔴 Split file not found: {Config.SPLIT_FILE_PATH}. This diagnostic must run on "
                     f"the SAME patients as v0.7-A/v0.7-B/v0.8/v0.9 — it will not create a new split.")
        return {'skipped': True}
    # The existing loader short-circuits to a pure load when the file exists,
    # so the (empty) `pairs` argument is never read. Guarded by the isfile
    # check above precisely so that stays true.
    train_pairs, val_pairs, _test_pairs = create_or_load_split(
        [], Config.SPLIT_FILE_PATH,
        Config.TRAIN_FRACTION, Config.VAL_FRACTION, Config.TEST_FRACTION,
        Config.SEED,
    )
    logger.info("🔒 Test split loaded for counting only — never wrapped in a Dataset here.")

    grades = load_patient_grades()

    # ---- ONE extraction pass for train and val, both feature spaces ---------
    log_section("🔬 v0.9-diag — Feature extraction (one pass, both feature spaces)")
    t_extract = time.time()
    train_feats = extract_grade_features(model, train_pairs, grades, device)
    val_feats = extract_grade_features(model, val_pairs, grades, device)
    logger.info(f"⏱️ Total extraction time: {time.time() - t_extract:.1f}s "
                f"({train_feats['labels'].numel() + val_feats['labels'].numel()} encoder forwards, "
                f"run ONCE and cached for all three probes).")

    if train_feats['labels'].numel() == 0 or val_feats['labels'].numel() == 0:
        logger.error("🔴 No labeled samples extracted — cannot run the probes.")
        return {'skipped': True}

    # Free the 3D model before the probes; the probes only need cached vectors.
    if device.type == 'cuda':
        torch.cuda.empty_cache()

    # ---- The three probes ---------------------------------------------------
    log_section("🔬 v0.9-diag — Probe A: raw (pre-SE) features + nn.Linear(512, 2)")
    res_a = train_probe_on_features(
        train_feats['raw'], train_feats['labels'], val_feats['raw'], val_feats['labels'],
        head_factory=lambda: nn.Linear(512, 2), device=device,
        seed=Config.DIAG_SEED, tag='A_raw_linear',
    )

    log_section("🔬 v0.9-diag — Probe B: post-SE features + nn.Linear(512, 2)")
    res_b = train_probe_on_features(
        train_feats['postse'], train_feats['labels'], val_feats['postse'], val_feats['labels'],
        head_factory=lambda: nn.Linear(512, 2), device=device,
        seed=Config.DIAG_SEED, tag='B_postse_linear',
    )

    log_section("🔬 v0.9-diag — Probe C: post-SE features + ClassificationHead3D")
    res_c = train_probe_on_features(
        train_feats['postse'], train_feats['labels'], val_feats['postse'], val_feats['labels'],
        head_factory=lambda: _DiagCachedFeatureHead(ClassificationHead3D(
            in_channels=512, hidden_dim=Config.CLS_HEAD_HIDDEN_DIM,
            num_classes=Config.NUM_CLS_CLASSES, dropout=Config.CLS_HEAD_DROPOUT,
        )),
        device=device, seed=Config.DIAG_SEED, tag='C_postse_clshead',
    )

    verdicts = _diag_emit_decision_table(res_a, res_b, res_c)

    # ---- CSV: one row per probe, APPENDED (never overwrites) ----------------
    csv_path = Config.DIAG_CSV_PATH
    isolates = {'A_raw_linear': 'H2 (representation collapse)',
                'B_postse_linear': 'H3 (SE suppression)',
                'C_postse_clshead': 'H1 optimization vs H4 architecture'}
    features_of = {'A_raw_linear': 'raw (avgpool(l4), pre-SE)',
                   'B_postse_linear': 'postse (avgpool(SE(l4)))',
                   'C_postse_clshead': 'postse (avgpool(SE(l4)))'}
    head_of = {'A_raw_linear': 'nn.Linear(512,2)',
               'B_postse_linear': 'nn.Linear(512,2)',
               'C_postse_clshead': 'ClassificationHead3D'}
    header = ['timestamp', 'probe', 'features', 'head', 'accuracy', 'precision', 'recall', 'f1',
              'majority_baseline', 'final_train_loss', 'n_train', 'n_val', 'seed',
              'probe_lr', 'probe_epochs', 'probe_batch_size', 'checkpoint', 'isolates', 'verdict']
    write_header = not os.path.isfile(csv_path)
    stamp = time.strftime('%Y-%m-%d %H:%M:%S')
    with open(csv_path, 'a', newline='') as f:
        writer = csv.writer(f)
        if write_header:
            writer.writerow(header)
        for res, verdict in zip((res_a, res_b, res_c), verdicts):
            writer.writerow([
                stamp, res['tag'], features_of[res['tag']], head_of[res['tag']],
                f"{res['accuracy']:.6f}", f"{res['precision']:.6f}", f"{res['recall']:.6f}",
                f"{res['f1']:.6f}", f"{res['majority_baseline']:.6f}",
                f"{res['final_train_loss']:.6f}", res['n_train'], res['n_val'], Config.DIAG_SEED,
                Config.DIAG_PROBE_LR, Config.DIAG_PROBE_EPOCHS, Config.DIAG_PROBE_BATCH_SIZE,
                ckpt_path, isolates[res['tag']], verdict,
            ])
    logger.info(f"💾 Diagnostic results appended to {csv_path}")
    logger.info("🏁 v0.9-diag complete — nothing was retrained, nothing was overwritten.")

    return {'skipped': False, 'probe_a': res_a, 'probe_b': res_b, 'probe_c': res_c,
            'checkpoint': ckpt_path, 'csv_path': csv_path, 'verdicts': verdicts}


# ------------------------------------------------------------------------------
# 24f. NEW in v0.9-diag — Task F: instrument the GLOBAL gradient clip, cheaply.
#
#      The hypothesis is testable with two logged numbers and no retraining: if
#      the TOTAL grad norm routinely exceeds GRAD_CLIP_NORM = 1.0, then every
#      cls-head gradient is multiplied by 1.0 / total_norm — a factor set by the
#      segmentation branch. That is a measurement, not an inference.
#
#      DELIBERATE DEVIATION FROM THE SPEC, FLAGGED RATHER THAN MADE SILENTLY:
#      Task F asks for this option "in train_one_epoch_multitask", but §1 and §3
#      both forbid modifying train_one_epoch_*. Both are honoured here by
#      instrumenting from OUTSIDE: install_grad_norm_instrumentation() rebinds
#      the two module-level names the existing loop resolves at CALL time
#      (torch.nn.utils.clip_grad_norm_, build_fresh_multitask_model). No line of
#      the training loop is edited, and with Config.DIAG_LOG_GRAD_NORMS = False
#      (the default) nothing is installed at all, so the default run path is
#      byte-for-byte unchanged.
#
#      If you would rather have the literal in-place edit, it is these 4 lines
#      inside train_one_epoch_multitask's non-AMP branch:
#          if Config.GRAD_CLIP_NORM and Config.GRAD_CLIP_NORM > 0:
#              cls_before = _diag_param_grad_norm(list(model.cls_head.parameters()))
#              total = torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRAD_CLIP_NORM)
#              cls_after = _diag_param_grad_norm(list(model.cls_head.parameters()))
#              logger.info(f"📐 total={total:.3f} cls_before={cls_before:.5f} cls_after={cls_after:.5f}")
# ------------------------------------------------------------------------------
_DIAG_GRAD_STATE = {                                                     # NEW in v0.9-diag
    'installed': False, 'orig_clip': None, 'orig_builder': None,
    'cls_params': [], 'calls': 0, 'n_exceed': 0, 'sum_total': 0.0, 'max_total': 0.0,
    'sum_cls_before': 0.0, 'sum_cls_after': 0.0,
}


def _diag_param_grad_norm(params) -> float:                              # NEW in v0.9-diag
    """L2 norm over a parameter subset's gradients — the same norm definition
    clip_grad_norm_ uses, restricted to those parameters."""
    grads = [p.grad.detach() for p in params if p.grad is not None]
    if not grads:
        return float('nan')
    return float(torch.norm(torch.stack([torch.norm(g.float(), 2) for g in grads]), 2).item())


def install_grad_norm_instrumentation() -> None:                         # NEW in v0.9-diag
    global build_fresh_multitask_model
    if not Config.DIAG_LOG_GRAD_NORMS:
        logger.warning("⚠️ DIAG_LOG_GRAD_NORMS=False — grad-norm instrumentation NOT installed "
                       "(default path unchanged).")
        return
    if _DIAG_GRAD_STATE['installed']:
        return

    _DIAG_GRAD_STATE['orig_clip'] = torch.nn.utils.clip_grad_norm_
    _DIAG_GRAD_STATE['orig_builder'] = build_fresh_multitask_model
    orig_clip = _DIAG_GRAD_STATE['orig_clip']
    orig_builder = _DIAG_GRAD_STATE['orig_builder']

    def _instrumented_builder(device):
        model = orig_builder(device)
        # Capture the cls-head parameters by identity — clip_grad_norm_ only
        # receives a generator over ALL parameters and cannot tell them apart.
        _DIAG_GRAD_STATE['cls_params'] = (list(model.cls_head.parameters())
                                           if hasattr(model, 'cls_head') else [])
        logger.info(f"📐 Grad-norm instrumentation bound to {len(_DIAG_GRAD_STATE['cls_params'])} "
                    f"cls_head parameter tensor(s).")
        return model

    def _instrumented_clip(parameters, max_norm, *args, **kwargs):
        cls_params = _DIAG_GRAD_STATE['cls_params']
        cls_before = _diag_param_grad_norm(cls_params)
        total = orig_clip(parameters, max_norm, *args, **kwargs)   # returns the PRE-clip total norm
        cls_after = _diag_param_grad_norm(cls_params)
        total_f = float(total.item()) if torch.is_tensor(total) else float(total)

        s = _DIAG_GRAD_STATE
        s['calls'] += 1
        s['sum_total'] += total_f
        s['max_total'] = max(s['max_total'], total_f)
        if total_f > float(max_norm):
            s['n_exceed'] += 1
        if cls_before == cls_before:
            s['sum_cls_before'] += cls_before
            s['sum_cls_after'] += cls_after

        every = max(1, int(Config.DIAG_GRAD_LOG_EVERY))
        if s['calls'] % every == 1 or every == 1:
            scale = (float(max_norm) / total_f) if total_f > float(max_norm) else 1.0
            logger.info(
                f"📐 [clip step {s['calls']}] total_norm_before_clip={total_f:.4f} "
                f"(clip={float(max_norm)}) -> scale={scale:.4f} | "
                f"cls_head_norm before={cls_before:.6f} after={cls_after:.6f}"
            )
        return total

    torch.nn.utils.clip_grad_norm_ = _instrumented_clip
    build_fresh_multitask_model = _instrumented_builder
    _DIAG_GRAD_STATE['installed'] = True
    logger.warning(f"📐 Grad-norm instrumentation INSTALLED (one line per {Config.DIAG_GRAD_LOG_EVERY} "
                   f"optimizer steps ≈ once per epoch at 258 train samples / batch 2). "
                   f"This changes logging only — no gradient, loss, or update is altered.")


def report_grad_norm_instrumentation() -> dict:                          # NEW in v0.9-diag
    s = _DIAG_GRAD_STATE
    if not s['calls']:
        logger.warning("⚠️ No clip calls recorded — instrumentation was never installed or never ran.")
        return {}
    mean_total = s['sum_total'] / s['calls']
    frac_exceed = s['n_exceed'] / s['calls']
    log_section("📐 v0.9-diag — GLOBAL GRADIENT CLIP MEASUREMENT")
    logger.info(f"   optimizer steps observed        : {s['calls']}")
    logger.info(f"   mean total grad norm (pre-clip) : {mean_total:.4f}")
    logger.info(f"   max  total grad norm (pre-clip) : {s['max_total']:.4f}")
    logger.info(f"   steps exceeding GRAD_CLIP_NORM  : {s['n_exceed']} ({frac_exceed * 100:.1f}%)")
    logger.info(f"   mean cls_head grad norm before  : {s['sum_cls_before'] / s['calls']:.6f}")
    logger.info(f"   mean cls_head grad norm after   : {s['sum_cls_after'] / s['calls']:.6f}")
    if frac_exceed > 0.5:
        logger.info(f"   🔴 The clip is active on {frac_exceed * 100:.1f}% of steps, so the cls-head "
                    f"gradient is being scaled by a factor set by the segmentation branch.")
    else:
        logger.info("   ✅ The clip is rarely active — the global-clip term is NOT the binding "
                    "constraint on the cls head. Look at LR_SEG x LAMBDA_CLS instead.")
    return {'calls': s['calls'], 'mean_total_norm': mean_total, 'max_total_norm': s['max_total'],
            'frac_steps_clipped': frac_exceed}


def uninstall_grad_norm_instrumentation() -> None:                       # NEW in v0.9-diag
    global build_fresh_multitask_model
    if not _DIAG_GRAD_STATE['installed']:
        return
    torch.nn.utils.clip_grad_norm_ = _DIAG_GRAD_STATE['orig_clip']
    build_fresh_multitask_model = _DIAG_GRAD_STATE['orig_builder']
    _DIAG_GRAD_STATE['installed'] = False
    logger.info("📐 Grad-norm instrumentation removed — original functions restored.")


# ------------------------------------------------------------------------------
# 24g. NEW in v0.9-diag — Task G: self-check BEFORE the diagnostic runs.
#      CPU-only, seconds, no data, no checkpoint. Separate from the existing
#      run_v0_9_self_check(), which is left untouched.
# ------------------------------------------------------------------------------
def run_v0_9_diag_self_check() -> None:                                  # NEW in v0.9-diag
    log_section("🔍 v0.9-diag SELF-CHECK (no data / no checkpoint / no GPU required)")
    cpu = torch.device('cpu')
    small = (32, 32, 16)

    torch.manual_seed(0)
    encoder = resnet18_3d(in_channels=4)
    seg_model = SegmentationModel3D(encoder=encoder, num_classes=Config.NUM_SEG_CLASSES,
                                    target_size=small).to(cpu)
    seg_model.eval()
    # Batch >= 2 throughout: with this tiny probe input the bottleneck collapses
    # to 1x1x1 and BatchNorm3d cannot run in train mode on a batch of 1.
    x = torch.randn(2, 4, *small)

    with torch.no_grad():
        feat_raw, feat_postse = _diag_features_from_batch(seg_model, x)

        # [1] raw == encoder(x), i.e. exactly what run_linear_probe() measured in v0.8.
        direct_raw = seg_model.get_encoder()(x)
        ok1 = torch.allclose(feat_raw, direct_raw, atol=0.0)
        logger.info(f"   [1] extract raw == encoder(x) (the v0.8 probe's feature): "
                    f"{'✅ identical' if ok1 else '❌ DRIFTED'} | shape={tuple(feat_raw.shape)}")

        # [2] postse == AdaptiveAvgPool3d(1)(channel_attention(l4)), i.e. exactly
        #     what ClassificationHead3D.pool sees in v0.9.
        l4_direct, _ = seg_model.get_encoder()(x, return_feature_map=True)
        if seg_model.channel_attention is None:
            direct_postse = direct_raw
            note = " (SE disabled -> equals raw by construction)"
        else:
            direct_postse = torch.flatten(nn.AdaptiveAvgPool3d(1)(seg_model.channel_attention(l4_direct)), 1)
            note = ""
        ok2 = torch.allclose(feat_postse, direct_postse, atol=1e-6)
        logger.info(f"   [2] extract postse == pool(SE(l4)) (the v0.9 head's feature): "
                    f"{'✅ identical' if ok2 else '❌ DRIFTED'}{note}")

    # [3] train_probe_on_features is deterministic: same seed, same features, twice.
    torch.manual_seed(0)
    n_tr, n_va = 24, 12
    f_tr = torch.randn(n_tr, 512)
    y_tr = torch.tensor([1] * 18 + [0] * 6)                 # imbalanced on purpose (3.0 >= 1.5)
    f_va = torch.randn(n_va, 512)
    y_va = torch.tensor([1] * 9 + [0] * 3)
    # Signal injected so the probe has something to learn and the accuracies are
    # not both trivially identical for the wrong reason.
    f_tr[y_tr == 1, 0] += 3.0
    f_va[y_va == 1, 0] += 3.0
    prev_level = logger.level
    logger.setLevel(logging.WARNING)                        # keep the self-check log short
    r1 = train_probe_on_features(f_tr, y_tr, f_va, y_va, lambda: nn.Linear(512, 2), cpu, 7, 'selfcheck1')
    r2 = train_probe_on_features(f_tr, y_tr, f_va, y_va, lambda: nn.Linear(512, 2), cpu, 7, 'selfcheck2')
    r3 = train_probe_on_features(
        f_tr, y_tr, f_va, y_va,
        lambda: _DiagCachedFeatureHead(ClassificationHead3D(
            in_channels=512, hidden_dim=Config.CLS_HEAD_HIDDEN_DIM,
            num_classes=Config.NUM_CLS_CLASSES, dropout=Config.CLS_HEAD_DROPOUT)),
        cpu, 7, 'selfcheck3')
    logger.setLevel(prev_level)
    ok3 = (r1['accuracy'] == r2['accuracy']) and (r1['f1'] == r2['f1'])
    logger.info(f"   [3] train_probe_on_features reproducible (same seed twice): "
                f"{'✅' if ok3 else '❌'} | acc {r1['accuracy']:.4f} vs {r2['accuracy']:.4f}")
    logger.info(f"   [4] ClassificationHead3D trains through the cached-feature adapter: "
                f"{'✅' if r3['accuracy'] == r3['accuracy'] else '❌'} | acc {r3['accuracy']:.4f}")

    # [5] The class-weight tensor equals the one run_linear_probe() would build.
    labels_all = y_tr.tolist()
    n_hgg = sum(labels_all)
    n_lgg = len(labels_all) - n_hgg
    expected = torch.tensor([len(labels_all) / (2.0 * n_lgg), len(labels_all) / (2.0 * n_hgg)])
    produced = _diag_class_weight(labels_all, cpu)
    ok5 = produced is not None and torch.allclose(produced, expected, atol=0.0)
    logger.info(f"   [5] class-weight rule matches run_linear_probe() verbatim: "
                f"{'✅' if ok5 else '❌'} | {produced.tolist() if produced is not None else None} "
                f"vs {expected.tolist()}")

    # [6] The balanced case must produce NO weighting (ratio 1.0 < 1.5).
    ok6 = _diag_class_weight([1] * 10 + [0] * 10, cpu) is None
    logger.info(f"   [6] balanced labels -> unweighted CrossEntropy (ratio < 1.5): {'✅' if ok6 else '❌'}")

    if all((ok1, ok2, ok3, ok5, ok6)):
        logger.info("🔍 v0.9-diag self-check: ✅ all green — the measured feature spaces match the "
                    "v0.8 probe path and the v0.9 head path exactly.")
    else:
        logger.warning("🔴 v0.9-diag self-check FAILED — do not trust the probe numbers until this is fixed.")
    logger.info("")


# ==============================================================================
# NEW in v0.9-metrics — SECTION 25: HONEST CLASSIFICATION METRICS + RE-SCORING
#
# Why this section exists. The v0.9-diag run reported the joint classification
# head as 🔴 RED because its accuracy (0.7636) equalled the majority baseline
# (0.7636). That verdict was an ARTIFACT of the metric, not a property of the
# model. On a 76/24 split, accuracy is nearly blind: the head traded 7 HGG
# errors for 7 LGG successes, which leaves accuracy identical while balanced
# accuracy moves from 0.500 to 0.686. A true majority predictor would also have
# scored F1 = 2*42/(2*42+13) = 0.8660; the head scored 0.8434 — a DIFFERENT
# classifier, and Fisher's exact test on its 2x2 gives p = 0.012.
#
# Equally important in the other direction: with n = 55 (HGG 42 / LGG 13),
# P(X >= 44 | p = 0.7636) = 0.325, so v0.8's headline "Linear Probe 0.80 🟢
# GREEN" was never statistically significant either. A 6-vs-7 difference in LGG
# recall is ONE PATIENT and carries no information — the 95% Wilson interval on
# 7/13 is [0.29, 0.77].
#
# This section therefore:
#   Task A  classification_report_full() — confusion matrix first, then
#           balanced accuracy / MCC / AUC / Fisher / permutation p / Wilson CI.
#   Task B  classification_verdict() — GREEN / AMBER / RED, where AMBER means
#           "real-looking effect, underpowered". Collapsing AMBER into RED is
#           precisely the error being corrected here.
#   Task C  rescore_v0_9_checkpoint() — re-scores the EXISTING checkpoint's own
#           trained cls_head on the cached post-SE features. No retraining, no
#           second encoder pass, and it must reproduce acc=0.7636 / F1=0.8434 /
#           confusion (35, 6, 7, 7) or the whole re-scoring path is suspect.
#   Task D  the three Section 24 probes re-scored through the same report.
#   Task E  one five-row comparison table, sorted by balanced accuracy, with
#           underpowered comparisons labelled as such instead of ranked.
#   Task F  the same metrics wired into evaluate_multitask_full for future runs
#           (added keys only — every existing key keeps its name and value).
#   Task G  a CPU-only self-check that pins this implementation against the
#           reconstruction in the task prompt.
#
# NOTHING here trains a backbone. The only fitting is the three cached-feature
# probe re-fits, which are already read-only with respect to the checkpoint.
# The Test split is never wrapped in a Dataset. The only file written is
# Config.METRICS_CSV_PATH (plus an optional feature cache under
# outputs/v0_9/checkpoints/diag_features/).
# ==============================================================================
import math                                                              # NEW in v0.9-metrics

# ------------------------------------------------------------------------------
# 25a. NEW in v0.9-metrics — Task A primitives.
#      Each is implemented directly rather than pulled from sklearn, so the
#      notebook keeps its existing dependency list. scipy is used for Fisher
#      when present, with an exact math.comb fallback that was verified to
#      agree with scipy to ~1e-15.
# ------------------------------------------------------------------------------
def _metrics_wilson_ci(k: int, n: int, z: float = 1.959963984540054) -> Tuple[float, float]:  # NEW in v0.9-metrics
    """95% Wilson score interval for a binomial proportion. Chosen over the
    normal approximation because k = 0 or k = n (entirely possible with 13 LGG
    patients) would give a zero-width Wald interval, which is nonsense."""
    if n <= 0:
        return (float('nan'), float('nan'))
    p = k / n
    denom = 1.0 + z * z / n
    centre = (p + z * z / (2.0 * n)) / denom
    half = (z / denom) * math.sqrt(p * (1.0 - p) / n + z * z / (4.0 * n * n))
    return (max(0.0, centre - half), min(1.0, centre + half))


def _metrics_auc_mannwhitney(y_true: List[int], y_score: List[float]) -> float:  # NEW in v0.9-metrics
    """ROC AUC via the Mann-Whitney U identity:

        AUC = (#concordant pairs + 0.5 * #tied pairs) / (n_pos * n_neg)

    computed with AVERAGE RANKS so ties contribute exactly 0.5. This matters
    here: a majority predictor emits one constant score, every pair ties, and
    the correct answer is exactly 0.500 — a naive strict '>' count would return
    0.0 and look like a catastrophic model.
    """
    n = len(y_true)
    if n == 0 or len(y_score) != n:
        return float('nan')
    n_pos = sum(1 for t in y_true if t == 1)
    n_neg = n - n_pos
    if n_pos == 0 or n_neg == 0:
        return float('nan')

    order = sorted(range(n), key=lambda i: y_score[i])
    ranks = [0.0] * n
    i = 0
    while i < n:
        j = i
        while j + 1 < n and y_score[order[j + 1]] == y_score[order[i]]:
            j += 1
        avg_rank = (i + j) / 2.0 + 1.0                # 1-based average rank
        for k in range(i, j + 1):
            ranks[order[k]] = avg_rank
        i = j + 1

    sum_ranks_pos = sum(r for r, t in zip(ranks, y_true) if t == 1)
    return (sum_ranks_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)


def _metrics_fisher_greater(a: int, b: int, c: int, d: int) -> float:    # NEW in v0.9-metrics
    """One-sided Fisher exact p-value for the 2x2

            | true HGG | true LGG
        ----+----------+---------
        pred HGG |   a  |    b
        pred LGG |   c  |    d

    testing whether the model discriminates the grades at all (alternative =
    'greater'). scipy when available; otherwise the hypergeometric upper tail
    P(X >= a) with fixed margins, computed exactly with math.comb.
    """
    try:
        from scipy.stats import fisher_exact                             # NEW in v0.9-metrics
        return float(fisher_exact([[a, b], [c, d]], alternative='greater')[1])
    except Exception:
        n = a + b + c + d
        if n == 0:
            return float('nan')
        row1, col1 = a + b, a + c
        hi = min(row1, col1)
        total = math.comb(n, col1)
        if total == 0:
            return float('nan')
        tail = sum(math.comb(row1, x) * math.comb(n - row1, col1 - x)
                   for x in range(a, hi + 1))
        return tail / total


def _metrics_binom_tail_ge(k: int, n: int, p: float) -> float:           # NEW in v0.9-metrics
    """P(X >= k) for X ~ Binomial(n, p). Used for the power note: it answers
    'how often would a coin that predicts the majority class at rate p beat
    this accuracy by luck alone?'. Computed live so the sentence is true for
    whatever n the run actually saw, instead of quoting a hard-coded figure."""
    if n <= 0 or k > n:
        return float('nan')
    k = max(0, k)
    return float(sum(math.comb(n, i) * (p ** i) * ((1.0 - p) ** (n - i)) for i in range(k, n + 1)))


def _metrics_balanced_accuracy_np(y_true_arr, y_pred_arr) -> float:      # NEW in v0.9-metrics
    pos = (y_true_arr == 1)
    neg = ~pos
    n_pos, n_neg = int(pos.sum()), int(neg.sum())
    if n_pos == 0 or n_neg == 0:
        return float('nan')
    correct = (y_pred_arr == y_true_arr)
    return float((int((correct & pos).sum()) / n_pos + int((correct & neg).sum()) / n_neg) / 2.0)


def _metrics_permutation_p(y_true: List[int], y_pred: List[int],         # NEW in v0.9-metrics
                           iters: int = None, seed: int = None) -> float:
    """Permutation p-value on BALANCED ACCURACY: shuffle the labels against the
    fixed predictions `iters` times and report the fraction of null draws >=
    the observed value.

    Uses its OWN np.random.RandomState rather than the global numpy RNG — this
    function is called from evaluate_multitask_full during training, and
    perturbing the global stream would silently change augmentation draws and
    break the project's reproducibility guarantee.

    A reported 0.0000 means "no null draw reached the observed value in `iters`
    permutations", i.e. p < 1/iters — not p = 0.
    """
    iters = Config.METRICS_PERM_ITERS if iters is None else iters
    seed = Config.DIAG_SEED if seed is None else seed
    n = len(y_true)
    if n == 0 or len(set(y_true)) < 2:
        return float('nan')

    yt = np.asarray(y_true, dtype=np.int64)
    yp = np.asarray(y_pred, dtype=np.int64)
    observed = _metrics_balanced_accuracy_np(yt, yp)
    if observed != observed:
        return float('nan')

    rng = np.random.RandomState(seed)
    # argsort of uniforms == a uniformly random permutation per row, vectorised.
    perms = np.argsort(rng.random_sample((iters, n)), axis=1)
    yt_perm = yt[perms]                                   # (iters, n)
    correct = (yp[None, :] == yt_perm)
    pos = (yt_perm == 1)
    n_pos = pos.sum(axis=1)
    n_neg = n - n_pos
    with np.errstate(divide='ignore', invalid='ignore'):
        rec_pos = (correct & pos).sum(axis=1) / n_pos
        rec_neg = (correct & ~pos).sum(axis=1) / n_neg
    null_bal = (rec_pos + rec_neg) / 2.0
    return float(np.mean(null_bal >= observed - 1e-12))


# ------------------------------------------------------------------------------
# 25b. NEW in v0.9-metrics — Task A: the metric block itself.
# ------------------------------------------------------------------------------
def classification_report_full(y_true, y_pred, y_score=None) -> dict:    # NEW in v0.9-metrics
    """Full binary report for the HGG(1) / LGG(0) task.

    `y_score` is P(class = 1); pass None when only hard predictions exist (AUC
    is then NaN rather than silently faked).

    The CONFUSION MATRIX is the primary artifact — every scalar below it is a
    lossy summary of those four counts, and on a 42/13 split accuracy is the
    lossiest of them all.
    """
    y_true = [int(t) for t in y_true]
    y_pred = [int(p) for p in y_pred]
    n = len(y_true)
    if n == 0:
        return {'n': 0, 'confusion': {'a': 0, 'b': 0, 'c': 0, 'd': 0},
                'accuracy': float('nan'), 'f1': float('nan'),
                'majority_baseline': float('nan'), 'recall_hgg': float('nan'),
                'recall_lgg': float('nan'), 'balanced_accuracy': float('nan'),
                'mcc': float('nan'), 'auc': float('nan'), 'fisher_p': float('nan'),
                'perm_p': float('nan'), 'wilson_ci_lgg': (float('nan'), float('nan')),
                'n_hgg': 0, 'n_lgg': 0, 'precision': float('nan')}

    # a = true HGG predicted HGG | b = true LGG predicted HGG
    # c = true HGG predicted LGG | d = true LGG predicted LGG
    a = sum((t == 1 and p == 1) for t, p in zip(y_true, y_pred))
    b = sum((t == 0 and p == 1) for t, p in zip(y_true, y_pred))
    c = sum((t == 1 and p == 0) for t, p in zip(y_true, y_pred))
    d = sum((t == 0 and p == 0) for t, p in zip(y_true, y_pred))
    n_hgg, n_lgg = a + c, b + d

    # ---- kept identical to run_linear_probe / evaluate_multitask_full, for
    #      continuity with the v0.7 / v0.8 / v0.9 CSVs already on disk ----
    accuracy = (a + d) / n
    precision = a / (a + b) if (a + b) > 0 else 0.0
    recall = a / (a + c) if (a + c) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    majority_class = 1 if n_hgg >= n_lgg else 0
    majority_baseline = (n_hgg if majority_class == 1 else n_lgg) / n

    # ---- the imbalance-aware block ----
    recall_hgg = a / n_hgg if n_hgg > 0 else float('nan')
    recall_lgg = d / n_lgg if n_lgg > 0 else float('nan')
    balanced_accuracy = ((recall_hgg + recall_lgg) / 2.0
                         if (n_hgg > 0 and n_lgg > 0) else float('nan'))

    mcc_den = math.sqrt(float(a + b) * float(a + c) * float(d + b) * float(d + c))
    mcc = ((a * d - b * c) / mcc_den) if mcc_den > 0 else 0.0

    auc = _metrics_auc_mannwhitney(y_true, list(y_score)) if y_score is not None else float('nan')
    fisher_p = _metrics_fisher_greater(a, b, c, d)
    perm_p = _metrics_permutation_p(y_true, y_pred)
    wilson_lo, wilson_hi = _metrics_wilson_ci(d, n_lgg)

    return {
        'n': n, 'n_hgg': n_hgg, 'n_lgg': n_lgg,
        'confusion': {'a': a, 'b': b, 'c': c, 'd': d},
        'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1,
        'majority_baseline': majority_baseline,
        'recall_hgg': recall_hgg, 'recall_lgg': recall_lgg,
        'balanced_accuracy': balanced_accuracy,
        'mcc': mcc, 'auc': auc, 'fisher_p': fisher_p, 'perm_p': perm_p,
        'wilson_ci_lgg': (wilson_lo, wilson_hi),
    }


def log_classification_report(rep: dict, title: str) -> None:            # NEW in v0.9-metrics
    """Prints the confusion matrix FIRST and the scalars after it, so the
    reader sees the four counts before any summary that compresses them."""
    cm = rep['confusion']
    verdict = classification_verdict(rep)
    logger.info(f"\n📊 {title}  (n={rep['n']}: HGG={rep['n_hgg']}, LGG={rep['n_lgg']})")
    logger.info("   ┌───────────────┬────────────┬────────────┐")
    logger.info("   │               │  pred HGG  │  pred LGG  │")
    logger.info("   ├───────────────┼────────────┼────────────┤")
    logger.info(f"   │ true HGG      │ {cm['a']:10d} │ {cm['c']:10d} │  recall {rep['recall_hgg']:.4f}")
    logger.info(f"   │ true LGG      │ {cm['b']:10d} │ {cm['d']:10d} │  recall {rep['recall_lgg']:.4f}")
    logger.info("   └───────────────┴────────────┴────────────┘")
    lo, hi = rep['wilson_ci_lgg']
    logger.info(f"   accuracy          = {rep['accuracy']:.4f}   (majority baseline {rep['majority_baseline']:.4f})")
    logger.info(f"   F1 (HGG-positive) = {rep['f1']:.4f}")
    logger.info(f"   balanced accuracy = {rep['balanced_accuracy']:.4f}   <- the honest headline on a 76/24 split")
    logger.info(f"   MCC               = {rep['mcc']:.4f}")
    logger.info(f"   ROC AUC           = {rep['auc']:.4f}" if rep['auc'] == rep['auc']
                else "   ROC AUC           = n/a (no scores provided)")
    logger.info(f"   Fisher exact p    = {rep['fisher_p']:.4f} (one-sided)")
    logger.info(f"   permutation p     = {rep['perm_p']:.4f} ({Config.METRICS_PERM_ITERS} label shuffles"
                f"{'; 0.0000 means p < 1/%d' % Config.METRICS_PERM_ITERS if rep['perm_p'] == 0 else ''})")
    logger.info(f"   LGG recall 95% CI = [{lo:.2f}, {hi:.2f}]  (Wilson, on {cm['d']}/{rep['n_lgg']})")
    logger.info(f"   VERDICT           = {verdict}")


# ------------------------------------------------------------------------------
# 25b-bis. NEW in v0.9-metrics — Task F support: a FLAT dict of the new fields,
#      so wiring them into evaluate_multitask_full and the per-epoch CSV costs
#      one inserted line each and cannot disturb any existing key.
#      Every key is prefixed `cls_` and appended AFTER the existing ones, so
#      csv.DictWriter(fieldnames=list(row.keys())) emits them as trailing
#      columns and older CSVs stay readable.
# ------------------------------------------------------------------------------
_V0_9_METRICS_EPOCH_KEYS = (                                             # NEW in v0.9-metrics
    'cls_balanced_accuracy', 'cls_recall_hgg', 'cls_recall_lgg', 'cls_mcc', 'cls_auc',
    'cls_fisher_p', 'cls_perm_p', 'cls_wilson_lgg_lo', 'cls_wilson_lgg_hi',
    'cls_cm_a', 'cls_cm_b', 'cls_cm_c', 'cls_cm_d', 'cls_verdict',
)


def _metrics_epoch_fields(y_true, y_pred, y_score=None) -> dict:         # NEW in v0.9-metrics
    """The Task-F payload. NaN/empty when there is nothing to score, so a run
    with USE_CLASSIFICATION_HEAD=False still writes well-formed rows."""
    if not y_true:
        return {k: ('' if k == 'cls_verdict' else float('nan')) for k in _V0_9_METRICS_EPOCH_KEYS}
    rep = classification_report_full(y_true, y_pred, y_score)
    cm = rep['confusion']
    lo, hi = rep['wilson_ci_lgg']
    return {
        'cls_balanced_accuracy': rep['balanced_accuracy'],
        'cls_recall_hgg': rep['recall_hgg'], 'cls_recall_lgg': rep['recall_lgg'],
        'cls_mcc': rep['mcc'], 'cls_auc': rep['auc'],
        'cls_fisher_p': rep['fisher_p'], 'cls_perm_p': rep['perm_p'],
        'cls_wilson_lgg_lo': lo, 'cls_wilson_lgg_hi': hi,
        'cls_cm_a': cm['a'], 'cls_cm_b': cm['b'], 'cls_cm_c': cm['c'], 'cls_cm_d': cm['d'],
        'cls_verdict': classification_verdict(rep).split(' —')[0],
    }


# ------------------------------------------------------------------------------
# 25c. NEW in v0.9-metrics — Task B: the replacement GREEN/AMBER/RED rule.
#      The v0.8/v0.9 rule (`accuracy > majority_baseline`) is left untouched
#      wherever it already runs; this is an additional, separate function.
# ------------------------------------------------------------------------------
def classification_verdict(rep: dict) -> str:                            # NEW in v0.9-metrics
    """
        🟢 GREEN : balanced_accuracy > 0.5 AND perm_p <  0.05
        🟡 AMBER : balanced_accuracy > 0.5 AND perm_p >= 0.05
        🔴 RED   : balanced_accuracy <= 0.5

    AMBER is not a courtesy grade. With 13 LGG patients, most genuine effects
    cannot clear p < 0.05 — collapsing AMBER into RED is exactly the error this
    section exists to correct. Whenever AMBER is emitted the caller is given the
    n and the Wilson interval alongside it, so "underpowered" is visible rather
    than implied.
    """
    bal = rep.get('balanced_accuracy', float('nan'))
    perm = rep.get('perm_p', float('nan'))
    if bal != bal:
        return "⚪ N/A — one class absent from the evaluation set"
    if bal <= 0.5:
        return "🔴 RED — no discrimination beyond chance (balanced accuracy <= 0.5)"
    if perm == perm and perm < 0.05:
        return f"🟢 GREEN — discriminates (balanced acc {bal:.4f}, permutation p={perm:.4f} < 0.05)"
    lo, hi = rep.get('wilson_ci_lgg', (float('nan'), float('nan')))
    return (f"🟡 AMBER — real-looking effect but UNDERPOWERED (balanced acc {bal:.4f}, "
            f"permutation p={perm:.4f} >= 0.05; n={rep.get('n')}, LGG n={rep.get('n_lgg')}, "
            f"LGG recall 95% CI [{lo:.2f}, {hi:.2f}])")


# ------------------------------------------------------------------------------
# 25d. NEW in v0.9-metrics — shared checkpoint load + feature cache.
#      The load mirrors run_v0_9_grade_diagnostic() exactly: the SAME resolver
#      (_diag_resolve_checkpoint_path), the SAME builder
#      (build_fresh_multitask_model) and the SAME strict load_state_dict. It is
#      factored into one helper here only so Tasks C and D share a single load
#      instead of each writing their own.
# ------------------------------------------------------------------------------
_METRICS_FEATURE_CACHE = {}                                              # NEW in v0.9-metrics


def _metrics_load_checkpoint_model(device: torch.device):                # NEW in v0.9-metrics
    ckpt_path = _diag_resolve_checkpoint_path()
    logger.info(f"📂 Checkpoint : {ckpt_path}")
    if not os.path.isfile(ckpt_path):
        logger.error(f"🔴 Checkpoint not found at the resolved path: {ckpt_path}\n"
                     f"   Set Config.DIAG_CHECKPOINT_PATH explicitly, or check Config.DIAG_SEED "
                     f"(currently {Config.DIAG_SEED}). No fallback checkpoint will be used.")
        return None, ckpt_path
    if not Config.USE_CLASSIFICATION_HEAD:
        logger.error("🔴 USE_CLASSIFICATION_HEAD=False — build_fresh_multitask_model() would return a "
                     "plain SegmentationModel3D and the strict load would fail. Set it back to True.")
        return None, ckpt_path
    model = build_fresh_multitask_model(device)
    try:
        model.load_state_dict(torch.load(ckpt_path, map_location=device))   # strict, on purpose
    except Exception as e:
        logger.error(f"🔴 Strict state_dict load failed — stopping.\n{e}")
        return None, ckpt_path
    model.eval()
    model.requires_grad_(False)
    logger.info("✅ Checkpoint loaded strictly; encoder + SE + decoder + cls_head frozen (eval, no grad).")
    return model, ckpt_path


def _metrics_plain_tensor(t: torch.Tensor) -> torch.Tensor:              # NEW in v0.9-metrics
    """Strip MONAI's MetaTensor subclass off a tensor.

    LinearProbeDataset builds its volumes with MONAI transforms, so everything
    downstream — including the extracted 512-d features — is a
    monai.data.meta_tensor.MetaTensor. Pickling that subclass makes the cache
    unreadable under torch >= 2.6's weights_only=True default. as_subclass()
    reinterprets the same storage as a plain Tensor with no copy and no
    numerical change."""
    try:
        return t.detach().cpu().as_subclass(torch.Tensor)
    except Exception:
        return torch.as_tensor(np.asarray(t.detach().cpu()))


def _metrics_torch_load_cache(path: str) -> dict:                        # NEW in v0.9-metrics
    """torch.load across versions: >= 2.6 defaults weights_only=True; older
    versions do not accept the kwarg at all. Try the default first, then fall
    back explicitly — the cache is a file this notebook wrote itself."""
    try:
        return torch.load(path, map_location='cpu')
    except Exception:
        try:
            return torch.load(path, map_location='cpu', weights_only=False)
        except TypeError:
            raise


def _metrics_feature_cache_path() -> str:                                # NEW in v0.9-metrics
    return os.path.join(Config.METRICS_FEATURE_CACHE_DIR,
                        f'diag_features_seed_{Config.DIAG_SEED}.pt')


def _metrics_get_features(model, train_pairs, val_pairs, grades,          # NEW in v0.9-metrics
                          device, ckpt_path: str) -> Tuple[dict, dict]:
    """Task D's 'reuse, do not repeat' rule, in three tiers:
         1. in-memory cache from an earlier call in this same session
         2. a .pt cache on disk under outputs/v0_9/checkpoints/diag_features/
         3. a fresh extraction via Section 24's extract_grade_features (unchanged)
    The tier actually used is logged, because silently re-extracting would burn
    ~313 encoder forwards and quietly break the '<2 minutes' budget."""
    key = (ckpt_path, Config.DIAG_SEED)
    if key in _METRICS_FEATURE_CACHE:
        logger.info("♻️ Features: reused from the IN-MEMORY cache (no extraction, no disk read).")
        return _METRICS_FEATURE_CACHE[key]

    cache_path = _metrics_feature_cache_path()
    if os.path.isfile(cache_path):
        try:
            blob = _metrics_torch_load_cache(cache_path)
            if blob.get('checkpoint') == ckpt_path:
                logger.info(f"♻️ Features: loaded from the DISK cache {cache_path} (no extraction).")
                pair = ({'raw': blob['train_raw'], 'postse': blob['train_postse'],
                         'labels': blob['train_labels']},
                        {'raw': blob['val_raw'], 'postse': blob['val_postse'],
                         'labels': blob['val_labels']})
                _METRICS_FEATURE_CACHE[key] = pair
                return pair
            logger.warning(f"⚠️ Disk feature cache was built from a DIFFERENT checkpoint "
                           f"({blob.get('checkpoint')}) — ignoring it and re-extracting.")
        except Exception as e:
            logger.warning(f"⚠️ Could not read the feature cache ({e}) — re-extracting.")

    logger.info("🔬 Features: no usable cache — extracting once via Section 24's "
                "extract_grade_features() (this is the only expensive step).")
    t0 = time.time()
    train_feats = extract_grade_features(model, train_pairs, grades, device)
    val_feats = extract_grade_features(model, val_pairs, grades, device)
    logger.info(f"⏱️ Extraction took {time.time() - t0:.1f}s.")

    # Plain tensors from here on — see _metrics_plain_tensor(). Same values.
    train_feats = {k: _metrics_plain_tensor(v) for k, v in train_feats.items()}
    val_feats = {k: _metrics_plain_tensor(v) for k, v in val_feats.items()}

    os.makedirs(Config.METRICS_FEATURE_CACHE_DIR, exist_ok=True)
    try:
        # FLAT dict of {str, Tensor} on purpose: torch >= 2.6 defaults
        # torch.load(weights_only=True), which rejects nested containers. A flat
        # str/Tensor mapping round-trips on every torch version the project
        # might run on, old or new.
        torch.save({'checkpoint': ckpt_path,
                    'train_raw': train_feats['raw'], 'train_postse': train_feats['postse'],
                    'train_labels': train_feats['labels'],
                    'val_raw': val_feats['raw'], 'val_postse': val_feats['postse'],
                    'val_labels': val_feats['labels']}, cache_path)
        logger.info(f"💾 Feature cache written to {cache_path} — later runs skip the encoder pass.")
    except Exception as e:
        logger.warning(f"⚠️ Could not write the feature cache ({e}) — continuing without it.")

    pair = (train_feats, val_feats)
    _METRICS_FEATURE_CACHE[key] = pair
    return pair


# ------------------------------------------------------------------------------
# 25e. NEW in v0.9-metrics — Task C: re-score the EXISTING v0.9 cls head.
#
#      No second encoder pass is needed. ClassificationHead3D.forward() is
#      pool(l4).view(b, c) -> fc1 -> relu -> dropout -> fc2, and the cached
#      `postse` vectors ARE AdaptiveAvgPool3d(1)(SE(l4)) flattened — i.e.
#      exactly the tensor the head consumes after its own pool. Feeding them
#      back through _DiagCachedFeatureHead (a (B,512) -> (B,512,1,1,1) reshape,
#      on which the head's AdaptiveAvgPool3d(1) is the identity) runs the real
#      head, with the checkpoint's own trained weights, in eval() so dropout is
#      off. A fresh ClassificationHead3D would measure probe C, not this.
# ------------------------------------------------------------------------------
@torch.no_grad()
def rescore_v0_9_checkpoint(device: torch.device, model=None, val_feats: dict = None,  # NEW in v0.9-metrics
                            ckpt_path: str = None) -> Optional[dict]:
    log_section("🔎 v0.9-metrics — Task C: re-scoring the TRAINED v0.9 classification head")

    if model is None:
        model, ckpt_path = _metrics_load_checkpoint_model(device)
        if model is None:
            return None
    if val_feats is None:
        logger.error("🔴 rescore_v0_9_checkpoint needs the cached val features — call it from "
                     "run_v0_9_metrics_rescore(), which supplies them.")
        return None
    if not hasattr(model, 'cls_head'):
        logger.error("🔴 The loaded model has no cls_head — nothing to re-score.")
        return None

    head = _DiagCachedFeatureHead(model.cls_head).to(device)
    head.eval()                                   # dropout OFF — this is inference
    feats = val_feats['postse'].to(device)
    labels = val_feats['labels'].long().tolist()

    logits = head(feats).float()
    probs = torch.softmax(logits, dim=1)[:, 1]
    preds = torch.argmax(logits, dim=1).cpu().tolist()
    scores = probs.cpu().tolist()

    rep = classification_report_full(labels, preds, scores)
    log_classification_report(rep, "v0.9 joint classification head (re-scored from the checkpoint)")

    # ---- Built-in validation: this MUST reproduce the original v0.9 run ----
    cm = rep['confusion']
    exp_a, exp_b, exp_c, exp_d = Config.METRICS_EXPECTED_CONFUSION
    acc_ok = abs(rep['accuracy'] - Config.METRICS_EXPECTED_V0_9_ACC) <= 0.0001
    f1_ok = abs(rep['f1'] - Config.METRICS_EXPECTED_V0_9_F1) <= 0.0001
    cm_ok = (cm['a'], cm['b'], cm['c'], cm['d']) == (exp_a, exp_b, exp_c, exp_d)

    logger.info("\n🧾 Reproduction check against the original v0.9 run:")
    logger.info(f"   accuracy  expected {Config.METRICS_EXPECTED_V0_9_ACC:.4f} | measured "
                f"{rep['accuracy']:.4f} | {'✅ match' if acc_ok else '❌ MISMATCH'}")
    logger.info(f"   F1        expected {Config.METRICS_EXPECTED_V0_9_F1:.4f} | measured "
                f"{rep['f1']:.4f} | {'✅ match' if f1_ok else '❌ MISMATCH'}")
    logger.info(f"   confusion expected (a={exp_a}, b={exp_b}, c={exp_c}, d={exp_d}) | measured "
                f"(a={cm['a']}, b={cm['b']}, c={cm['c']}, d={cm['d']}) | "
                f"{'✅ match' if cm_ok else '❌ MISMATCH'}")

    rep['reproduces_original'] = bool(acc_ok and f1_ok and cm_ok)
    if not rep['reproduces_original']:
        logger.error(
            "🔴 STOP — the re-scoring path does NOT reproduce the original v0.9 numbers.\n"
            "   This means the cached post-SE features + trained cls_head are not measuring what the\n"
            "   training loop measured (candidate causes: a different checkpoint, a changed split, a\n"
            "   changed val preprocessing, or a cls_head that does not read the post-SE bottleneck).\n"
            "   Every conclusion in the task prompt's Section 0 is derived from those numbers and must\n"
            "   be revisited before this table is trusted."
        )
    else:
        logger.info("   ✅ The re-scoring path reproduces the original run exactly — the table below is "
                    "a re-measurement of the SAME predictions, not a new model.")
    return rep


# ------------------------------------------------------------------------------
# 25f. NEW in v0.9-metrics — Task D: re-score the three Section 24 probes.
#
#      The probes are NOT re-implemented. train_probe_on_features() is called
#      unchanged, and the trained head is recovered through the head_factory
#      closure: nn.Module.to() is in-place and returns self, so the object the
#      factory hands out IS the object that gets trained. Re-running it on the
#      val features therefore reproduces that probe's own predictions exactly —
#      and the accuracy is asserted against the value the function itself
#      returned, so any drift would be caught rather than assumed away.
# ------------------------------------------------------------------------------
def _metrics_rescore_probe(train_feats: torch.Tensor, train_labels: torch.Tensor,   # NEW in v0.9-metrics
                           val_feats: torch.Tensor, val_labels: torch.Tensor,
                           head_factory, device: torch.device, seed: int, tag: str) -> dict:
    captured = {}

    def capturing_factory():
        head = head_factory()
        captured['head'] = head          # same object train_probe_on_features trains in place
        return head

    base = train_probe_on_features(train_feats, train_labels, val_feats, val_labels,
                                   capturing_factory, device, seed, tag)
    head = captured.get('head')
    if head is None:
        logger.error(f"🔴 [{tag}] could not recover the trained head — skipping its report.")
        return {'tag': tag, 'report': None, 'base': base}

    head.eval()
    with torch.no_grad():
        logits = head(val_feats.to(device)).float()
        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = torch.argmax(logits, dim=1).cpu().tolist()
        scores = probs.cpu().tolist()

    rep = classification_report_full(val_labels.long().tolist(), preds, scores)

    drift = abs(rep['accuracy'] - base['accuracy'])
    if drift > 1e-9:
        logger.error(f"🔴 [{tag}] re-scored accuracy {rep['accuracy']:.6f} != the accuracy "
                     f"train_probe_on_features reported ({base['accuracy']:.6f}). The recovered head is "
                     f"not the trained head — do not trust this row.")
    else:
        logger.info(f"   ✅ [{tag}] re-scored predictions match train_probe_on_features exactly "
                    f"(accuracy {rep['accuracy']:.4f}).")
    rep['matches_section_24'] = bool(drift <= 1e-9)
    return {'tag': tag, 'report': rep, 'base': base}


# ------------------------------------------------------------------------------
# 25g. NEW in v0.9-metrics — Task E: the five-row comparison table.
# ------------------------------------------------------------------------------
def _metrics_emit_comparison(rows: List[Tuple[str, dict]]) -> List[Tuple[str, dict]]:  # NEW in v0.9-metrics
    """rows = [(label, report), ...]. Returns them sorted by balanced accuracy."""
    ordered = sorted(rows, key=lambda r: (r[1]['balanced_accuracy']
                                          if r[1]['balanced_accuracy'] == r[1]['balanced_accuracy']
                                          else -1.0), reverse=True)

    log_section("📊 v0.9-metrics — COMPARISON TABLE (sorted by balanced accuracy)")
    header = (f"{'model':<30}{'n':>4}{'acc':>9}{'balAcc':>9}{'AUC':>8}{'MCC':>8}"
              f"{'LGG recall (95% CI)':>26}{'perm_p':>9}  verdict")
    logger.info(header)
    logger.info("-" * len(header))
    for label, rep in ordered:
        cm = rep['confusion']
        lo, hi = rep['wilson_ci_lgg']
        lgg = f"{cm['d']}/{rep['n_lgg']} [{lo:.2f}, {hi:.2f}]"
        auc = f"{rep['auc']:.3f}" if rep['auc'] == rep['auc'] else "  n/a"
        short = rep['_verdict_short']
        logger.info(f"{label:<30}{rep['n']:>4}{rep['accuracy']:>9.4f}{rep['balanced_accuracy']:>9.3f}"
                    f"{auc:>8}{rep['mcc']:>8.3f}{lgg:>26}{rep['perm_p']:>9.4f}  {short}")

    # ---- Which differences actually survive? ----
    logger.info("")
    log_section("🧠 v0.9-metrics — WHAT THIS TABLE DOES AND DOES NOT SUPPORT")

    sig = [l for l, r in ordered if r['perm_p'] == r['perm_p'] and r['perm_p'] < 0.05
           and r['balanced_accuracy'] > 0.5]
    nonsig = [l for l, r in ordered if l not in sig]
    logger.info(f"   Clears the permutation test at p < 0.05 : {', '.join(sig) if sig else '(none)'}")
    logger.info(f"   Does NOT clear it                       : {', '.join(nonsig) if nonsig else '(none)'}")

    logger.info("")
    logger.info(f"   Pairwise comparisons (a difference of fewer than "
                f"{Config.METRICS_UNDERPOWERED_PATIENTS} correctly-classified patients out of n is noise):")
    for i in range(len(ordered)):
        for j in range(i + 1, len(ordered)):
            li, ri = ordered[i]
            lj, rj = ordered[j]
            ci = ri['confusion']['a'] + ri['confusion']['d']
            cj = rj['confusion']['a'] + rj['confusion']['d']
            delta = abs(ci - cj)
            if delta < Config.METRICS_UNDERPOWERED_PATIENTS:
                logger.info(f"     • {li} vs {lj}: differ by {delta} patient(s) out of {ri['n']} "
                            f"— UNDERPOWERED, do not rank these.")
            else:
                logger.info(f"     • {li} vs {lj}: differ by {delta} patients out of {ri['n']} "
                            f"— reportable as a difference, still not a significance test.")
    return ordered


# ------------------------------------------------------------------------------
# 25h. NEW in v0.9-metrics — the MODE='metrics' runner.
# ------------------------------------------------------------------------------
def run_v0_9_metrics_rescore(device: torch.device = None) -> dict:       # NEW in v0.9-metrics
    log_section("📏 MedSSL-3D v0.9-metrics — CLASSIFICATION METRIC RE-SCORE (read-only)")

    if not Config.RUN_METRICS_RESCORE:
        logger.warning("⚠️ RUN_METRICS_RESCORE=False — no checkpoint loaded, nothing written.")
        return {'skipped': True}

    device = _diag_resolve_device() if device is None else device
    Config.setup_dirs()
    os.makedirs(Config.METRICS_FEATURE_CACHE_DIR, exist_ok=True)
    logger.info("🔒 Read-only run: no optimizer over the backbone, no training loader, no epoch of "
                "backbone training. The only fitting is the three cached-feature probe re-fits.")

    model, ckpt_path = _metrics_load_checkpoint_model(device)
    if model is None:
        return {'skipped': True}

    if not os.path.isfile(Config.SPLIT_FILE_PATH):
        logger.error(f"🔴 Split file not found: {Config.SPLIT_FILE_PATH}. Refusing to create a new split.")
        return {'skipped': True}
    train_pairs, val_pairs, _test_pairs = create_or_load_split(
        [], Config.SPLIT_FILE_PATH,
        Config.TRAIN_FRACTION, Config.VAL_FRACTION, Config.TEST_FRACTION, Config.SEED,
    )
    logger.info("🔒 Test split loaded for counting only — never wrapped in a Dataset here.")

    grades = load_patient_grades()
    train_feats, val_feats = _metrics_get_features(model, train_pairs, val_pairs, grades,
                                                    device, ckpt_path)
    if val_feats['labels'].numel() == 0:
        logger.error("🔴 No labeled val samples — cannot score anything.")
        return {'skipped': True}

    val_labels = val_feats['labels'].long().tolist()

    # ---- Row 1: the majority baseline, measured rather than asserted --------
    log_section("🔎 v0.9-metrics — reference row: the majority-class predictor")
    majority_class = 1 if sum(val_labels) >= (len(val_labels) - sum(val_labels)) else 0
    maj_preds = [majority_class] * len(val_labels)
    maj_scores = [0.5] * len(val_labels)      # one constant score -> every pair ties -> AUC exactly 0.5
    rep_majority = classification_report_full(val_labels, maj_preds, maj_scores)
    log_classification_report(rep_majority, "majority baseline (always predict HGG)")

    # ---- Row 2: Task C ------------------------------------------------------
    rep_joint = rescore_v0_9_checkpoint(device, model=model, val_feats=val_feats, ckpt_path=ckpt_path)
    if rep_joint is None:
        return {'skipped': True}

    # ---- Rows 3-5: Task D ---------------------------------------------------
    log_section("🔎 v0.9-metrics — Task D: re-scoring the three Section 24 probes")
    probe_specs = [
        ('probe A (raw + Linear)', 'raw', lambda: nn.Linear(512, 2), 'A_raw_linear'),
        ('probe B (post-SE + Linear)', 'postse', lambda: nn.Linear(512, 2), 'B_postse_linear'),
        ('probe C (post-SE + ClsHead)', 'postse',
         lambda: _DiagCachedFeatureHead(ClassificationHead3D(
             in_channels=512, hidden_dim=Config.CLS_HEAD_HIDDEN_DIM,
             num_classes=Config.NUM_CLS_CLASSES, dropout=Config.CLS_HEAD_DROPOUT)),
         'C_postse_clshead'),
    ]
    probe_reports = []
    for label, space, factory, tag in probe_specs:
        out = _metrics_rescore_probe(
            train_feats[space], train_feats['labels'], val_feats[space], val_feats['labels'],
            factory, device, Config.DIAG_SEED, tag,
        )
        if out['report'] is not None:
            log_classification_report(out['report'], label)
            probe_reports.append((label, out['report']))

    # ---- Task E -------------------------------------------------------------
    rows = [('majority baseline', rep_majority), ('v0.9 joint cls head', rep_joint)] + probe_reports
    for _label, rep in rows:
        v = classification_verdict(rep)
        rep['_verdict_short'] = v.split(' ')[0] + ' ' + v.split(' ')[1]   # e.g. "🟢 GREEN"
        rep['_verdict'] = v
    ordered = _metrics_emit_comparison(rows)

    # ---- The conclusion, recorded in the run log, not only in a chat --------
    log_section("🏁 v0.9-metrics — CONCLUSION")
    bal = rep_joint['balanced_accuracy']
    perm = rep_joint['perm_p']
    if bal > 0.5 and perm == perm and perm < 0.05:
        logger.info(
            f"   ✅ v0.9's classification head MET ITS OBJECTIVE. Balanced accuracy = {bal:.4f} "
            f"(chance = 0.500), permutation p = {perm:.4f} < 0.05, Fisher p = {rep_joint['fisher_p']:.4f}.\n"
            f"   The earlier 🔴 RED verdict was a METRIC ARTIFACT: accuracy on a "
            f"{rep_joint['n_hgg']}/{rep_joint['n_lgg']} split is nearly blind to the HGG-error /\n"
            f"   LGG-success trade the head actually made. No sampler, learning-rate, LAMBDA_CLS or\n"
            f"   gradient-clip change is justified by this evidence."
        )
    elif bal > 0.5:
        logger.warning(
            f"   🟡 The head discriminates (balanced accuracy {bal:.4f} > 0.500) but does not clear the "
            f"permutation test (p = {perm:.4f}). With {rep_joint['n_lgg']} LGG patients this is expected "
            f"even for a real effect — AMBER, not RED. Do not treat this as a failure."
        )
    else:
        logger.warning(f"   🔴 Balanced accuracy {bal:.4f} <= 0.500 — this genuinely is no better than "
                       f"chance, and CONTRADICTS the framing in the task prompt's Section 0. Report the "
                       f"contradiction; do not resolve it silently.")

    if not rep_joint.get('reproduces_original', False):
        logger.warning("   ⚠️ ...but note the reproduction check above FAILED, so treat every number in "
                       "this table as provisional until that is explained.")

    # ---- Power note, computed live so the sentence is true for THIS run ----
    logger.info("")
    best_label, best_rep = ordered[0]
    n_eval = best_rep['n']
    k_best = best_rep['confusion']['a'] + best_rep['confusion']['d']
    p_maj = best_rep['majority_baseline']
    tail = _metrics_binom_tail_ge(k_best, n_eval, p_maj)
    logger.info("   📎 POWER NOTE — applies to EVERY row above, including v0.8's 0.80 headline:")
    logger.info(f"      The best row here is '{best_label}' at {k_best}/{n_eval} correct. Against a coin "
                f"that just predicts the majority class at rate {p_maj:.4f},")
    logger.info(f"      P(X >= {k_best} | n={n_eval}) = {tail:.3f}. An accuracy figure alone therefore "
                f"cannot establish that any of these models works.")
    logger.info(f"      On the recorded 55-patient val split this is P(X >= 44 | n=55, p=0.7636) = "
                f"{_metrics_binom_tail_ge(44, 55, 0.7636):.3f} — which is why v0.8's '0.80 🟢 GREEN' was "
                f"never significant either.")
    logger.info("      Balanced accuracy + the permutation test, not accuracy, are what separate these "
                "rows. A one-patient difference in LGG recall carries no information.")

    # ---- CSV ---------------------------------------------------------------
    csv_path = Config.METRICS_CSV_PATH
    header = ['timestamp', 'model', 'n', 'n_hgg', 'n_lgg', 'a_true_hgg_pred_hgg', 'b_true_lgg_pred_hgg',
              'c_true_hgg_pred_lgg', 'd_true_lgg_pred_lgg', 'accuracy', 'majority_baseline', 'f1',
              'recall_hgg', 'recall_lgg', 'balanced_accuracy', 'mcc', 'auc', 'fisher_p', 'perm_p',
              'wilson_lgg_lo', 'wilson_lgg_hi', 'verdict', 'checkpoint', 'seed']
    write_header = not os.path.isfile(csv_path)
    stamp = time.strftime('%Y-%m-%d %H:%M:%S')
    with open(csv_path, 'a', newline='') as f:
        writer = csv.writer(f)
        if write_header:
            writer.writerow(header)
        for label, rep in ordered:
            cm = rep['confusion']
            lo, hi = rep['wilson_ci_lgg']
            writer.writerow([
                stamp, label, rep['n'], rep['n_hgg'], rep['n_lgg'],
                cm['a'], cm['b'], cm['c'], cm['d'],
                f"{rep['accuracy']:.6f}", f"{rep['majority_baseline']:.6f}", f"{rep['f1']:.6f}",
                f"{rep['recall_hgg']:.6f}", f"{rep['recall_lgg']:.6f}",
                f"{rep['balanced_accuracy']:.6f}", f"{rep['mcc']:.6f}", f"{rep['auc']:.6f}",
                f"{rep['fisher_p']:.6f}", f"{rep['perm_p']:.6f}", f"{lo:.6f}", f"{hi:.6f}",
                rep['_verdict'], ckpt_path, Config.DIAG_SEED,
            ])
    logger.info(f"💾 Re-score table appended to {csv_path}")
    logger.info("🏁 v0.9-metrics complete — nothing was retrained, no checkpoint was written.")

    return {'skipped': False, 'majority': rep_majority, 'joint': rep_joint,
            'probes': probe_reports, 'ordered': ordered, 'csv_path': csv_path}


# ------------------------------------------------------------------------------
# 25i. NEW in v0.9-metrics — Task G: self-check. CPU-only, no data, seconds.
#      Pins this implementation against the reconstruction the task prompt is
#      built on, so a silent metric bug cannot masquerade as a finding.
# ------------------------------------------------------------------------------
def run_v0_9_metrics_self_check() -> None:                               # NEW in v0.9-metrics
    log_section("🔍 v0.9-metrics SELF-CHECK (no data / no checkpoint / no GPU required)")

    def _labels(n_hgg: int, n_lgg: int) -> List[int]:
        return [1] * n_hgg + [0] * n_lgg

    # [1] A majority predictor on 42/13 must be exactly chance.
    y_true = _labels(42, 13)
    y_pred = [1] * 55
    rep = classification_report_full(y_true, y_pred, [0.5] * 55)
    v = classification_verdict(rep)
    ok1 = (abs(rep['balanced_accuracy'] - 0.5) < 1e-12 and abs(rep['mcc']) < 1e-12
           and abs(rep['auc'] - 0.5) < 1e-12 and v.startswith('🔴'))
    logger.info(f"   [1] majority predictor (42/13) -> balAcc={rep['balanced_accuracy']:.4f} "
                f"mcc={rep['mcc']:.4f} auc={rep['auc']:.4f} verdict={v.split(' —')[0]} "
                f"{'✅' if ok1 else '❌'}")
    logger.info(f"       (its accuracy {rep['accuracy']:.4f} and F1 {rep['f1']:.4f} are exactly the "
                f"numbers the v0.9 head was wrongly compared against)")

    # [2] A perfect predictor.
    rep2 = classification_report_full(y_true, y_true, [1.0 if t else 0.0 for t in y_true])
    ok2 = (abs(rep2['balanced_accuracy'] - 1.0) < 1e-12 and abs(rep2['mcc'] - 1.0) < 1e-12
           and abs(rep2['auc'] - 1.0) < 1e-12)
    logger.info(f"   [2] perfect predictor -> balAcc={rep2['balanced_accuracy']:.4f} "
                f"mcc={rep2['mcc']:.4f} auc={rep2['auc']:.4f} {'✅' if ok2 else '❌'}")

    # [3] The (35, 6, 7, 7) reconstruction must reproduce the prompt's numbers.
    a, b, c, d = Config.METRICS_EXPECTED_CONFUSION
    y_true3 = [1] * a + [0] * b + [1] * c + [0] * d
    y_pred3 = [1] * (a + b) + [0] * (c + d)
    rep3 = classification_report_full(y_true3, y_pred3)
    ok3 = (abs(rep3['balanced_accuracy'] - 0.686) <= 0.001
           and abs(rep3['accuracy'] - Config.METRICS_EXPECTED_V0_9_ACC) <= 0.0001
           and abs(rep3['f1'] - Config.METRICS_EXPECTED_V0_9_F1) <= 0.0001)
    logger.info(f"   [3] confusion ({a},{b},{c},{d}) -> acc={rep3['accuracy']:.4f} "
                f"F1={rep3['f1']:.4f} balAcc={rep3['balanced_accuracy']:.4f} "
                f"fisher_p={rep3['fisher_p']:.4f} {'✅' if ok3 else '❌'}")
    logger.info(f"       (this pins the task prompt's algebraic reconstruction against this code)")

    # [4] Hand-rolled AUC vs a brute-force pairwise count on a random case.
    rng = random.Random(Config.DIAG_SEED)
    yt4 = [rng.randint(0, 1) for _ in range(40)]
    if 0 not in yt4:
        yt4[0] = 0
    if 1 not in yt4:
        yt4[1] = 1
    ys4 = [round(rng.random(), 2) for _ in range(40)]        # rounding forces real ties
    pos = [s for s, t in zip(ys4, yt4) if t == 1]
    neg = [s for s, t in zip(ys4, yt4) if t == 0]
    brute = sum((1.0 if p > q else (0.5 if p == q else 0.0)) for p in pos for q in neg) / (len(pos) * len(neg))
    mine = _metrics_auc_mannwhitney(yt4, ys4)
    ok4 = abs(brute - mine) < 1e-12
    logger.info(f"   [4] AUC vs brute-force pairwise count (with ties): {mine:.6f} vs {brute:.6f} "
                f"{'✅' if ok4 else '❌'}")

    # [5] The permutation test must be reproducible across two calls.
    p1 = _metrics_permutation_p(y_true3, y_pred3, iters=2000, seed=Config.DIAG_SEED)
    p2 = _metrics_permutation_p(y_true3, y_pred3, iters=2000, seed=Config.DIAG_SEED)
    ok5 = (p1 == p2)
    logger.info(f"   [5] permutation p reproducible across two calls: {p1:.4f} == {p2:.4f} "
                f"{'✅' if ok5 else '❌'}")

    # [6] AMBER must exist as a distinct outcome, not collapse into RED.
    #     6/13 LGG recall with a small n is a real-looking but underpowered effect.
    rep6 = classification_report_full([1] * 8 + [0] * 4, [1] * 7 + [0] + [1] * 2 + [0] * 2)
    v6 = classification_verdict(rep6)
    ok6 = (rep6['balanced_accuracy'] > 0.5 and v6.startswith('🟡')) or rep6['perm_p'] < 0.05
    logger.info(f"   [6] small-n positive effect -> balAcc={rep6['balanced_accuracy']:.4f} "
                f"perm_p={rep6['perm_p']:.4f} verdict={v6.split(' —')[0]} {'✅' if ok6 else '❌'}")

    # [7] Fisher fallback must agree with scipy when scipy is present.
    try:
        from scipy.stats import fisher_exact
        ref = float(fisher_exact([[a, b], [c, d]], alternative='greater')[1])
        nn_ = a + b + c + d
        row1, col1 = a + b, a + c
        hi = min(row1, col1)
        hand = sum(math.comb(row1, x) * math.comb(nn_ - row1, col1 - x)
                   for x in range(a, hi + 1)) / math.comb(nn_, col1)
        ok7 = abs(ref - hand) < 1e-12
        logger.info(f"   [7] math.comb Fisher fallback == scipy: {hand:.6f} vs {ref:.6f} "
                    f"{'✅' if ok7 else '❌'}")
    except Exception:
        ok7 = True
        logger.info("   [7] scipy not installed — the math.comb Fisher fallback is the only path (ok).")

    if all((ok1, ok2, ok3, ok4, ok5, ok6, ok7)):
        logger.info("🔍 v0.9-metrics self-check: ✅ all green.")
    else:
        logger.warning("🔴 v0.9-metrics self-check FAILED — do not trust the re-scored table.")
    logger.info("")


# ------------------------------------------------------------------------------
# 24. Dispatcher — v0.8 branches unchanged; 'multitask' branch added in v0.9.
# ------------------------------------------------------------------------------
def main() -> None:
    if Config.MODE == 'simclr':
        run_mode_a_simclr()
    elif Config.MODE == 'segmentation':
        run_mode_b_segmentation()
    elif Config.MODE == 'multitask':                                     # NEW in v0.9
        run_v0_9_self_check()
        run_mode_c_multitask()
    elif Config.MODE == 'diagnostic':                                    # NEW in v0.9-diag
        # READ-ONLY. Builds NO training loader, NO optimizer, and calls NO
        # run_single_seed_* / run_mode_* function — running the cell must never
        # be able to overwrite best_moving_avg_model.pth, which is the very
        # checkpoint this diagnostic reads.
        if not Config.RUN_GRADE_DIAGNOSTIC:
            logger.warning("⚠️ MODE='diagnostic' but RUN_GRADE_DIAGNOSTIC=False — "
                           "nothing loaded, nothing written, nothing trained.")
        else:
            run_v0_9_diag_self_check()
            run_v0_9_grade_diagnostic()
    elif Config.MODE == 'metrics':                                       # NEW in v0.9-metrics
        # READ-ONLY, wired exactly like 'diagnostic': no training loader, no
        # optimizer over the backbone, no run_single_seed_* / run_mode_* call.
        # The only fitting is the three cached-feature probe re-fits.
        if not Config.RUN_METRICS_RESCORE:
            logger.warning("⚠️ MODE='metrics' but RUN_METRICS_RESCORE=False — "
                           "nothing loaded, nothing written, nothing trained.")
        else:
            run_v0_9_metrics_self_check()
            run_v0_9_metrics_rescore()
    else:
        raise ValueError(
            f"Unknown Config.MODE: '{Config.MODE}' "
            f"(expected 'simclr', 'segmentation', 'multitask', 'diagnostic', or 'metrics')"   # NEW in v0.9-metrics
        )


if __name__ == "__main__":
    main()


🔍 v0.9-metrics SELF-CHECK (no data / no checkpoint / no GPU required)
   [1] majority predictor (42/13) -> balAcc=0.5000 mcc=0.0000 auc=0.5000 verdict=🔴 RED ✅
       (its accuracy 0.7636 and F1 0.8660 are exactly the numbers the v0.9 head was wrongly compared against)
   [2] perfect predictor -> balAcc=1.0000 mcc=1.0000 auc=1.0000 ✅
   [3] confusion (35,6,7,7) -> acc=0.7636 F1=0.8434 balAcc=0.6859 fisher_p=0.0123 ✅
       (this pins the task prompt's algebraic reconstruction against this code)
   [4] AUC vs brute-force pairwise count (with ties): 0.509333 vs 0.509333 ✅
   [5] permutation p reproducible across two calls: 0.0145 == 0.0145 ✅
   [6] small-n positive effect -> balAcc=0.6875 perm_p=0.2396 verdict=🟡 AMBER ✅
   [7] math.comb Fisher fallback == scipy: 0.012332 vs 0.012332 ✅
🔍 v0.9-metrics self-check: ✅ all green.


📏 MedSSL-3D v0.9-metrics — CLASSIFICATION METRIC RE-SCORE (read-only)
🔒 Read-only run: no optimizer over the backbone, no training loader, no epoch of backbone trai